In [3]:
import openvino as ov

print("OpenVINO version:", ov.__version__)

from openvino import Core

core = Core()

print("OpenVINO Core loaded successfully!")
print("Available devices:", core.available_devices)

OpenVINO version: 2026.3.1-22476-759c5a6ab8c-releases/2026/3
OpenVINO Core loaded successfully!
Available devices: ['CPU', 'GPU']


In [7]:
# ==========================================================================================
# CISLR LANDMARK MODEL
# ONE-TIME MODEL SETUP
#
# TFLite -> OpenVINO IR
# ==========================================================================================

from pathlib import Path
import openvino as ov
from openvino import Core

print("=" * 110)
print("CISLR — PRETRAINED MODEL CONVERSION TO OPENVINO")
print("=" * 110)

# ------------------------------------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------------------------------------

ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL"
)

PALM_TFLITE = (
    ROOT
    / "models"
    / "pretrained"
    / "palm"
    / "palm_detection_full.tflite"
)

HAND_TFLITE = (
    ROOT
    / "models"
    / "pretrained"
    / "hand_landmark"
    / "hand_landmark_full.tflite"
)

PALM_OV_DIR = (
    ROOT
    / "models"
    / "openvino"
    / "palm"
)

HAND_OV_DIR = (
    ROOT
    / "models"
    / "openvino"
    / "hand_landmark"
)

PALM_OV_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HAND_OV_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PALM_XML = PALM_OV_DIR / "palm_detection_full.xml"
HAND_XML = HAND_OV_DIR / "hand_landmark_full.xml"

# ------------------------------------------------------------------------------------------
# 2. CHECK SOURCE FILES
# ------------------------------------------------------------------------------------------

print("\n[1/5] Checking source TFLite models...\n")

for name, path in [
    ("Palm detector", PALM_TFLITE),
    ("Hand landmark", HAND_TFLITE),
]:
    if not path.exists():
        raise FileNotFoundError(
            f"{name} model not found:\n{path}"
        )

    size_mb = path.stat().st_size / (1024 * 1024)

    print(f"[FOUND] {name}")
    print(f"        {path}")
    print(f"        {size_mb:.2f} MB\n")

# ------------------------------------------------------------------------------------------
# 3. CONVERSION HELPER
# ------------------------------------------------------------------------------------------

def convert_and_save_tflite(source_path, xml_path, model_name):

    print("-" * 110)
    print(f"Processing: {model_name}")
    print("-" * 110)

    # If a valid IR already exists, keep it.
    bin_path = xml_path.with_suffix(".bin")

    if xml_path.exists() and bin_path.exists():

        print("[INFO] Existing OpenVINO IR found.")
        print("       XML:", xml_path)
        print("       BIN:", bin_path)

        try:
            core = Core()
            test_model = core.read_model(str(xml_path))

            print("[PASS] Existing model is readable.")
            return

        except Exception as e:
            print("[WARNING] Existing IR could not be read.")
            print("          Re-converting...")
            print("          Reason:", e)

    print("[INFO] Converting TFLite -> OpenVINO...")

    try:
        model = ov.convert_model(
            str(source_path)
        )

    except Exception as e:

        print("\n[ERROR] OpenVINO conversion failed.")
        print("Model :", model_name)
        print("Source:", source_path)
        print("Error :", repr(e))

        raise

    print("[PASS] Conversion completed.")

    print("[INFO] Saving OpenVINO IR...")

    ov.save_model(
        model,
        str(xml_path),
        compress_to_fp16=False
    )

    if not xml_path.exists():
        raise RuntimeError(
            f"XML was not created:\n{xml_path}"
        )

    if not bin_path.exists():
        raise RuntimeError(
            f"BIN was not created:\n{bin_path}"
        )

    print("[PASS] Saved:")
    print("       XML:", xml_path)
    print("       BIN:", bin_path)

# ------------------------------------------------------------------------------------------
# 4. CONVERT PALM
# ------------------------------------------------------------------------------------------

print("\n[2/5] Converting palm detector...\n")

convert_and_save_tflite(
    PALM_TFLITE,
    PALM_XML,
    "Palm Detection Full"
)

# ------------------------------------------------------------------------------------------
# 5. CONVERT HAND LANDMARK
# ------------------------------------------------------------------------------------------

print("\n[3/5] Converting hand-landmark model...\n")

convert_and_save_tflite(
    HAND_TFLITE,
    HAND_XML,
    "Hand Landmark Full"
)

# ------------------------------------------------------------------------------------------
# 6. LOAD THE SAVED IR MODELS
# ------------------------------------------------------------------------------------------

print("\n[4/5] Loading saved OpenVINO models...\n")

core = Core()

palm_model = core.read_model(
    str(PALM_XML)
)

hand_model = core.read_model(
    str(HAND_XML)
)

palm_net = core.compile_model(
    palm_model,
    "CPU"
)

hand_net = core.compile_model(
    hand_model,
    "CPU"
)

print("[PASS] Both models compile on CPU.")

# ------------------------------------------------------------------------------------------
# 7. MODEL INSPECTION
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("PALM MODEL")
print("=" * 110)

print("\nInputs:")

for i, inp in enumerate(palm_model.inputs):

    print(
        f"  Input {i}:",
        "name =", inp.get_any_name(),
        "| shape =", inp.get_partial_shape(),
        "| type =", inp.get_element_type()
    )

print("\nOutputs:")

for i, out in enumerate(palm_model.outputs):

    try:
        name = out.get_any_name()
    except:
        name = "<unnamed>"

    print(
        f"  Output {i}:",
        "name =", name,
        "| shape =", out.get_partial_shape(),
        "| type =", out.get_element_type()
    )

print("\n" + "=" * 110)
print("HAND LANDMARK MODEL")
print("=" * 110)

print("\nInputs:")

for i, inp in enumerate(hand_model.inputs):

    print(
        f"  Input {i}:",
        "name =", inp.get_any_name(),
        "| shape =", inp.get_partial_shape(),
        "| type =", inp.get_element_type()
    )

print("\nOutputs:")

for i, out in enumerate(hand_model.outputs):

    try:
        name = out.get_any_name()
    except:
        name = "<unnamed>"

    print(
        f"  Output {i}:",
        "name =", name,
        "| shape =", out.get_partial_shape(),
        "| type =", out.get_element_type()
    )

# ------------------------------------------------------------------------------------------
# 8. FINAL FILE CHECK
# ------------------------------------------------------------------------------------------

print("\n[5/5] Final file verification...\n")

expected_files = [
    PALM_XML,
    PALM_XML.with_suffix(".bin"),
    HAND_XML,
    HAND_XML.with_suffix(".bin"),
]

all_ok = True

for path in expected_files:

    exists = path.exists()

    print(
        "[OK]     " if exists else "[MISSING]",
        path
    )

    if not exists:
        all_ok = False

print("\n" + "=" * 110)

if all_ok:

    print("MODEL SETUP COMPLETE")
    print("=" * 110)

    print("\nThe permanent model structure is now:\n")

    print(
        ROOT
        / "models"
        / "openvino"
        / "palm"
    )

    print("    ├── palm_detection_full.xml")
    print("    └── palm_detection_full.bin")

    print()

    print(
        ROOT
        / "models"
        / "openvino"
        / "hand_landmark"
    )

    print("    ├── hand_landmark_full.xml")
    print("    └── hand_landmark_full.bin")

    print("\n[PASS] L6 model dependency is now satisfied.")

else:

    raise RuntimeError(
        "Model setup did not create all four required files."
    )

print("=" * 110)

CISLR — PRETRAINED MODEL CONVERSION TO OPENVINO

[1/5] Checking source TFLite models...

[FOUND] Palm detector
        /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/pretrained/palm/palm_detection_full.tflite
        2.23 MB

[FOUND] Hand landmark
        /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/pretrained/hand_landmark/hand_landmark_full.tflite
        5.23 MB


[2/5] Converting palm detector...

--------------------------------------------------------------------------------------------------------------
Processing: Palm Detection Full
--------------------------------------------------------------------------------------------------------------
[INFO] Existing OpenVINO IR found.
       XML: /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/openvino/palm/palm_detection_full.xml
       BIN: /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/openvino/palm/palm_detection_full.bin
[PASS] Existing model 

In [8]:
# ==========================================================================================
# CISLR LANDMARK MODEL — L6
# FULL CISLR LANDMARK EXTRACTION
# CORRECTED SELF-CONTAINED VERSION
# ==========================================================================================

from pathlib import Path
import os
import cv2
import json
import time
import math
import warnings
import traceback

import numpy as np
import pandas as pd

from openvino import Core

warnings.filterwarnings("ignore")


# ==========================================================================================
# 1. PATHS
# ==========================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

DATASET_CSV = Path(
    "/home/dipshikha/Music/cao/dataset.csv"
)

VIDEO_DIR = Path(
    "/home/dipshikha/Music/cao/CISLR_v1.5-a_videos"
)

LANDMARK_ROOT = (
    PROJECT_ROOT
    / "CISLR_LANDMARK_MODEL"
)

PALM_XML = (
    LANDMARK_ROOT
    / "models"
    / "openvino"
    / "palm"
    / "palm_detection_full.xml"
)

HAND_XML = (
    LANDMARK_ROOT
    / "models"
    / "openvino"
    / "hand_landmark"
    / "hand_landmark_full.xml"
)


# ==========================================================================================
# 2. L6 OUTPUT DIRECTORIES
# ==========================================================================================

L6_ROOT = (
    LANDMARK_ROOT
    / "audit"
    / "l6_full_extraction"
)

RAW_DIR = (
    L6_ROOT
    / "raw_sequences"
)

FEATURE_DIR = (
    L6_ROOT
    / "features_32x254"
)

REPORT_DIR = (
    L6_ROOT
    / "reports"
)

FAILURE_DIR = (
    L6_ROOT
    / "failures"
)

for directory in [
    RAW_DIR,
    FEATURE_DIR,
    REPORT_DIR,
    FAILURE_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


MANIFEST_PATH = (
    REPORT_DIR
    / "l6_manifest.csv"
)

FAILURE_PATH = (
    REPORT_DIR
    / "l6_failures.csv"
)

PROGRESS_PATH = (
    REPORT_DIR
    / "l6_progress.json"
)

AUDIT_PATH = (
    REPORT_DIR
    / "l6_final_dataset_audit.csv"
)

SUMMARY_PATH = (
    REPORT_DIR
    / "l6_summary.json"
)

SCALE_REPAIR_PATH = (
    REPORT_DIR
    / "l6_scale_repairs.csv"
)


# ==========================================================================================
# 3. FROZEN CONFIGURATION
# ==========================================================================================

PALM_SIZE = 192
LANDMARK_SIZE = 224

SEQUENCE_LENGTH = 32

NUM_HANDS = 2
NUM_LANDMARKS = 21
NUM_COORDS = 3

FEATURE_DIM = 254

MAX_INTERPOLATION_GAP = 3

MIN_VALID_FRAMES = 8
MIN_DETECTION_RATE_FOR_TRAINING = 0.20

EPS = 1e-6
EPS_SCALE = 1e-6


# ------------------------------------------------------------------------------------------
# L5.1 robust scale
# ------------------------------------------------------------------------------------------

SCALE_RATIO_THRESHOLD = 2.5
NEIGHBOR_AGREEMENT_RATIO = 1.8


# ------------------------------------------------------------------------------------------
# Palm detector
# ------------------------------------------------------------------------------------------

PALM_SCORE_THRESHOLD = 0.50
PALM_NMS_IOU = 0.30

PALM_X_SCALE = 192.0
PALM_Y_SCALE = 192.0
PALM_W_SCALE = 192.0
PALM_H_SCALE = 192.0


# ------------------------------------------------------------------------------------------
# Frozen MediaPipe-style ROI
# ------------------------------------------------------------------------------------------

ROI_SCALE_X = 2.6
ROI_SCALE_Y = 2.6
ROI_SHIFT_Y = -0.5

TARGET_ANGLE = math.pi / 2.0


# ------------------------------------------------------------------------------------------
# Runtime
# ------------------------------------------------------------------------------------------

DEVICE = "CPU"

SAVE_RAW = True

# IMPORTANT:
# If the notebook stops, rerun the same cell.
# Existing valid files will be skipped.
RESUME = True

CHECKPOINT_EVERY = 25

VERBOSE_PROGRESS = True


# ==========================================================================================
# 4. START
# ==========================================================================================

print("=" * 120)
print("CISLR LANDMARK MODEL — L6")
print("FULL CISLR LANDMARK EXTRACTION")
print("=" * 120)

print("Dataset       :", DATASET_CSV)
print("Videos        :", VIDEO_DIR)
print("Palm model    :", PALM_XML)
print("Landmark model:", HAND_XML)
print("Output        :", L6_ROOT)
print("Device        :", DEVICE)
print("Representation:", (SEQUENCE_LENGTH, FEATURE_DIM))
print("Resume        :", RESUME)

print("=" * 120)


# ==========================================================================================
# 5. VERIFY REQUIRED FILES
# ==========================================================================================

required_files = [
    DATASET_CSV,
    PALM_XML,
    PALM_XML.with_suffix(".bin"),
    HAND_XML,
    HAND_XML.with_suffix(".bin"),
]

for path in required_files:

    if not path.exists():
        raise FileNotFoundError(
            f"Required file missing:\n{path}"
        )

print("\n[PASS] Required files found.")


# ==========================================================================================
# 6. LOAD OPENVINO MODELS
# ==========================================================================================

core = Core()

palm_model = core.read_model(
    str(PALM_XML)
)

hand_model = core.read_model(
    str(HAND_XML)
)

palm_net = core.compile_model(
    palm_model,
    DEVICE
)

hand_net = core.compile_model(
    hand_model,
    DEVICE
)

print("\nPalm input :", palm_net.input(0).shape)

print("Palm outputs:")

for i, output in enumerate(palm_net.outputs):
    print(
        f"  {i}:",
        output.shape
    )

print("\nHand input :", hand_net.input(0).shape)

print("Hand outputs:")

for i, output in enumerate(hand_net.outputs):
    print(
        f"  {i}:",
        output.shape
    )


# ==========================================================================================
# 7. VERIFY NHWC INPUTS
# ==========================================================================================

palm_input_shape = tuple(
    int(x)
    for x in palm_net.input(0).shape
)

hand_input_shape = tuple(
    int(x)
    for x in hand_net.input(0).shape
)

if palm_input_shape != (1, 192, 192, 3):

    raise RuntimeError(
        f"Unexpected palm input shape: {palm_input_shape}\n"
        f"Expected: (1,192,192,3)"
    )

if hand_input_shape != (1, 224, 224, 3):

    raise RuntimeError(
        f"Unexpected hand input shape: {hand_input_shape}\n"
        f"Expected: (1,224,224,3)"
    )

print("\n[PASS] Models use expected NHWC layout.")


# ==========================================================================================
# 8. PALM ANCHORS
# ==========================================================================================

def generate_palm_anchors():

    input_size = 192

    min_scale = 0.1484375
    max_scale = 0.75

    strides = [
        8,
        16,
        16,
        16
    ]

    num_layers = len(strides)

    anchors = []

    layer_id = 0

    while layer_id < num_layers:

        scales = []

        last_same_stride_layer = layer_id

        while (
            last_same_stride_layer < num_layers
            and
            strides[last_same_stride_layer] == strides[layer_id]
        ):

            scale = (
                min_scale
                +
                (max_scale - min_scale)
                *
                last_same_stride_layer
                /
                max(num_layers - 1, 1)
            )

            scales.append(scale)

            if last_same_stride_layer == num_layers - 1:
                scale_next = 1.0

            else:
                scale_next = (
                    min_scale
                    +
                    (max_scale - min_scale)
                    *
                    (last_same_stride_layer + 1)
                    /
                    max(num_layers - 1, 1)
                )

            scales.append(
                math.sqrt(
                    scale * scale_next
                )
            )

            last_same_stride_layer += 1

        stride = strides[layer_id]

        feature_map_height = int(
            math.ceil(
                input_size / stride
            )
        )

        feature_map_width = int(
            math.ceil(
                input_size / stride
            )
        )

        for y in range(feature_map_height):

            for x in range(feature_map_width):

                for _ in range(len(scales)):

                    anchors.append(
                        [
                            (x + 0.5) / feature_map_width,
                            (y + 0.5) / feature_map_height,
                            1.0,
                            1.0
                        ]
                    )

        layer_id = last_same_stride_layer

    return np.asarray(
        anchors,
        dtype=np.float32
    )


ANCHORS = generate_palm_anchors()

print("\nPalm anchors:", ANCHORS.shape)

if ANCHORS.shape != (2016, 4):

    raise RuntimeError(
        f"Expected palm anchors (2016,4), "
        f"got {ANCHORS.shape}"
    )

print("[PASS] Palm anchors generated.")


# ==========================================================================================
# 9. GENERAL HELPERS
# ==========================================================================================

def sigmoid(x):

    x = np.clip(
        x,
        -100,
        100
    )

    return (
        1.0
        /
        (
            1.0
            +
            np.exp(-x)
        )
    )


# ==========================================================================================
# 10. PALM PREPROCESSING — CORRECT NHWC
# ==========================================================================================

def palm_preprocess(frame):

    h, w = frame.shape[:2]

    scale = min(
        PALM_SIZE / w,
        PALM_SIZE / h
    )

    new_w = int(
        round(
            w * scale
        )
    )

    new_h = int(
        round(
            h * scale
        )
    )

    resized = cv2.resize(
        frame,
        (new_w, new_h)
    )

    canvas = np.zeros(
        (
            PALM_SIZE,
            PALM_SIZE,
            3
        ),
        dtype=np.uint8
    )

    pad_x = (
        PALM_SIZE
        -
        new_w
    ) // 2

    pad_y = (
        PALM_SIZE
        -
        new_h
    ) // 2

    canvas[
        pad_y:pad_y + new_h,
        pad_x:pad_x + new_w
    ] = resized

    rgb = cv2.cvtColor(
        canvas,
        cv2.COLOR_BGR2RGB
    )

    # IMPORTANT:
    # OpenVINO model expects NHWC:
    #
    # (1, 192, 192, 3)
    #
    tensor = (
        rgb.astype(np.float32)
        /
        255.0
    )[None, ...]

    info = {
        "scale": scale,
        "pad_x": pad_x,
        "pad_y": pad_y,
        "original_width": w,
        "original_height": h
    }

    return tensor, info


# ==========================================================================================
# 11. PALM INFERENCE
# ==========================================================================================

def palm_inference(frame):

    tensor, info = palm_preprocess(
        frame
    )

    if tensor.shape != (1, 192, 192, 3):

        raise RuntimeError(
            f"Bad palm tensor shape: {tensor.shape}"
        )

    result = palm_net(
        [tensor]
    )

    outputs = [
        np.asarray(
            result[output]
        )
        for output in palm_net.outputs
    ]

    regression = None
    logits = None

    for output in outputs:

        squeezed = np.squeeze(
            output
        )

        if (
            squeezed.ndim == 2
            and
            squeezed.shape == (2016, 18)
        ):

            regression = squeezed

        elif squeezed.size == 2016:

            logits = squeezed.reshape(-1)

    if regression is None:

        raise RuntimeError(
            "Palm regression output not found."
        )

    if logits is None:

        raise RuntimeError(
            "Palm classification output not found."
        )

    return (
        regression,
        logits,
        info
    )


# ==========================================================================================
# 12. PALM DECODING
# ==========================================================================================

def decode_palms(
    regression,
    logits
):

    scores = sigmoid(
        logits
    )

    candidate_indices = np.where(
        scores >= PALM_SCORE_THRESHOLD
    )[0]

    detections = []

    for i in candidate_indices:

        anchor = ANCHORS[i]
        raw = regression[i]

        x_center = (
            raw[0]
            /
            PALM_X_SCALE
            *
            anchor[2]
            +
            anchor[0]
        )

        y_center = (
            raw[1]
            /
            PALM_Y_SCALE
            *
            anchor[3]
            +
            anchor[1]
        )

        width = (
            raw[2]
            /
            PALM_W_SCALE
            *
            anchor[2]
        )

        height = (
            raw[3]
            /
            PALM_H_SCALE
            *
            anchor[3]
        )

        keypoints = []

        for k in range(7):

            kp_x = (
                raw[4 + 2 * k]
                /
                PALM_X_SCALE
                *
                anchor[2]
                +
                anchor[0]
            )

            kp_y = (
                raw[5 + 2 * k]
                /
                PALM_Y_SCALE
                *
                anchor[3]
                +
                anchor[1]
            )

            keypoints.append(
                [
                    kp_x,
                    kp_y
                ]
            )

        detections.append(
            {
                "score": float(scores[i]),

                "xmin": float(
                    x_center
                    -
                    width / 2
                ),

                "ymin": float(
                    y_center
                    -
                    height / 2
                ),

                "xmax": float(
                    x_center
                    +
                    width / 2
                ),

                "ymax": float(
                    y_center
                    +
                    height / 2
                ),

                "keypoints": np.asarray(
                    keypoints,
                    dtype=np.float32
                )
            }
        )

    return detections


# ==========================================================================================
# 13. NMS
# ==========================================================================================

def detection_iou(a, b):

    x1 = max(
        a["xmin"],
        b["xmin"]
    )

    y1 = max(
        a["ymin"],
        b["ymin"]
    )

    x2 = min(
        a["xmax"],
        b["xmax"]
    )

    y2 = min(
        a["ymax"],
        b["ymax"]
    )

    intersection = (
        max(
            0.0,
            x2 - x1
        )
        *
        max(
            0.0,
            y2 - y1
        )
    )

    area_a = (
        max(
            0.0,
            a["xmax"] - a["xmin"]
        )
        *
        max(
            0.0,
            a["ymax"] - a["ymin"]
        )
    )

    area_b = (
        max(
            0.0,
            b["xmax"] - b["xmin"]
        )
        *
        max(
            0.0,
            b["ymax"] - b["ymin"]
        )
    )

    union = (
        area_a
        +
        area_b
        -
        intersection
    )

    if union <= 0:
        return 0.0

    return intersection / union


def weighted_nms(detections):

    if len(detections) == 0:
        return []

    remaining = sorted(
        detections,
        key=lambda x: x["score"],
        reverse=True
    )

    output = []

    while remaining:

        first = remaining[0]

        overlapping = []
        rest = []

        for detection in remaining:

            if (
                detection_iou(
                    first,
                    detection
                )
                >
                PALM_NMS_IOU
            ):
                overlapping.append(
                    detection
                )

            else:
                rest.append(
                    detection
                )

        weights = np.asarray(
            [
                d["score"]
                for d in overlapping
            ],
            dtype=np.float32
        )

        weights = (
            weights
            /
            (
                weights.sum()
                +
                EPS
            )
        )

        merged = {
            "score": max(
                d["score"]
                for d in overlapping
            ),

            "xmin": float(
                sum(
                    w * d["xmin"]
                    for w, d
                    in zip(
                        weights,
                        overlapping
                    )
                )
            ),

            "ymin": float(
                sum(
                    w * d["ymin"]
                    for w, d
                    in zip(
                        weights,
                        overlapping
                    )
                )
            ),

            "xmax": float(
                sum(
                    w * d["xmax"]
                    for w, d
                    in zip(
                        weights,
                        overlapping
                    )
                )
            ),

            "ymax": float(
                sum(
                    w * d["ymax"]
                    for w, d
                    in zip(
                        weights,
                        overlapping
                    )
                )
            )
        }

        kp = np.zeros(
            (7, 2),
            dtype=np.float32
        )

        for w, detection in zip(
            weights,
            overlapping
        ):

            kp += (
                w
                *
                detection["keypoints"]
            )

        merged["keypoints"] = kp

        output.append(
            merged
        )

        remaining = rest

    return output


# ==========================================================================================
# 14. CONVERT PALM COORDINATES BACK TO ORIGINAL FRAME
# ==========================================================================================

def palm_detection_to_original(
    detection,
    info
):

    scale = info["scale"]
    pad_x = info["pad_x"]
    pad_y = info["pad_y"]

    def convert_point(x, y):

        px = (
            x * PALM_SIZE
            -
            pad_x
        ) / scale

        py = (
            y * PALM_SIZE
            -
            pad_y
        ) / scale

        return [
            float(px),
            float(py)
        ]

    keypoints = np.asarray(
        [
            convert_point(
                x,
                y
            )
            for x, y
            in detection["keypoints"]
        ],
        dtype=np.float32
    )

    p1 = convert_point(
        detection["xmin"],
        detection["ymin"]
    )

    p2 = convert_point(
        detection["xmax"],
        detection["ymax"]
    )

    return {
        "score": detection["score"],

        "xmin": p1[0],
        "ymin": p1[1],

        "xmax": p2[0],
        "ymax": p2[1],

        "keypoints": keypoints
    }


# ==========================================================================================
# 15. PALM → ORIENTED HAND RECT
# ==========================================================================================

def normalize_angle(angle):

    return (
        angle
        -
        2.0
        *
        math.pi
        *
        math.floor(
            (
                angle
                +
                math.pi
            )
            /
            (
                2.0
                *
                math.pi
            )
        )
    )


def palm_to_hand_rect(detection):

    keypoints = detection[
        "keypoints"
    ]

    x0, y0 = keypoints[0]
    x1, y1 = keypoints[2]

    rotation = (
        TARGET_ANGLE
        -
        math.atan2(
            -(y1 - y0),
            x1 - x0
        )
    )

    rotation = normalize_angle(
        rotation
    )

    width = (
        detection["xmax"]
        -
        detection["xmin"]
    )

    height = (
        detection["ymax"]
        -
        detection["ymin"]
    )

    center_x = (
        detection["xmin"]
        +
        detection["xmax"]
    ) / 2.0

    center_y = (
        detection["ymin"]
        +
        detection["ymax"]
    ) / 2.0

    long_side = max(
        width,
        height
    )

    rect_width = (
        long_side
        *
        ROI_SCALE_X
    )

    rect_height = (
        long_side
        *
        ROI_SCALE_Y
    )

    shift_y = (
        ROI_SHIFT_Y
        *
        rect_height
    )

    cos_r = math.cos(
        rotation
    )

    sin_r = math.sin(
        rotation
    )

    center_x += (
        -shift_y
        *
        sin_r
    )

    center_y += (
        shift_y
        *
        cos_r
    )

    return {
        "cx": float(center_x),
        "cy": float(center_y),
        "width": float(rect_width),
        "height": float(rect_height),
        "rotation": float(rotation)
    }


# ==========================================================================================
# 16. ROI CORNERS
# ==========================================================================================

def rect_corners(rect):

    width = rect["width"]
    height = rect["height"]

    local = np.asarray(
        [
            [
                -width / 2,
                -height / 2
            ],

            [
                width / 2,
                -height / 2
            ],

            [
                width / 2,
                height / 2
            ],

            [
                -width / 2,
                height / 2
            ]
        ],
        dtype=np.float32
    )

    c = math.cos(
        rect["rotation"]
    )

    s = math.sin(
        rect["rotation"]
    )

    rotation_matrix = np.asarray(
        [
            [
                c,
                -s
            ],

            [
                s,
                c
            ]
        ],
        dtype=np.float32
    )

    corners = (
        local
        @
        rotation_matrix.T
    )

    corners[:, 0] += rect["cx"]
    corners[:, 1] += rect["cy"]

    return corners.astype(
        np.float32
    )


# ==========================================================================================
# 17. CROP HAND ROI
# ==========================================================================================

def crop_hand_roi(
    frame,
    rect
):

    corners = rect_corners(
        rect
    )

    destination = np.asarray(
        [
            [
                0,
                0
            ],

            [
                LANDMARK_SIZE - 1,
                0
            ],

            [
                LANDMARK_SIZE - 1,
                LANDMARK_SIZE - 1
            ],

            [
                0,
                LANDMARK_SIZE - 1
            ]
        ],
        dtype=np.float32
    )

    M = cv2.getPerspectiveTransform(
        corners,
        destination
    )

    M_inv = cv2.getPerspectiveTransform(
        destination,
        corners
    )

    crop = cv2.warpPerspective(
        frame,
        M,
        (
            LANDMARK_SIZE,
            LANDMARK_SIZE
        ),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT
    )

    return (
        crop,
        M_inv
    )


# ==========================================================================================
# 18. HAND PREPROCESSING — CORRECT NHWC
# ==========================================================================================

def hand_preprocess(crop):

    rgb = cv2.cvtColor(
        crop,
        cv2.COLOR_BGR2RGB
    )

    # IMPORTANT:
    # OpenVINO model expects:
    #
    # (1,224,224,3)
    #
    tensor = (
        rgb.astype(np.float32)
        /
        255.0
    )[None, ...]

    return tensor


# ==========================================================================================
# 19. HAND LANDMARK INFERENCE
# ==========================================================================================

def run_hand_landmark(crop):

    tensor = hand_preprocess(
        crop
    )

    if tensor.shape != (1, 224, 224, 3):

        raise RuntimeError(
            f"Bad hand tensor shape: {tensor.shape}"
        )

    result = hand_net(
        [tensor]
    )

    outputs = [
        np.asarray(
            result[output]
        )
        for output in hand_net.outputs
    ]

    landmark_outputs = []
    scalar_outputs = []

    for output in outputs:

        squeezed = np.squeeze(
            output
        )

        if squeezed.size == 63:

            landmark_outputs.append(
                squeezed.reshape(
                    21,
                    3
                ).astype(
                    np.float32
                )
            )

        elif squeezed.size == 1:

            scalar_outputs.append(
                float(
                    squeezed.reshape(-1)[0]
                )
            )

    if len(landmark_outputs) == 0:

        raise RuntimeError(
            "No 63-value landmark output found."
        )

    # ------------------------------------------------------------------
    # The model contains:
    #
    # Identity   : screen landmarks
    # Identity_3 : world landmarks
    #
    # But instead of depending on output ordering, we preserve the
    # already-tested L1 logic:
    #
    # screen landmarks have much larger coordinate magnitude.
    # ------------------------------------------------------------------

    magnitudes = [
        float(
            np.mean(
                np.abs(output)
            )
        )
        for output in landmark_outputs
    ]

    screen_index = int(
        np.argmax(
            magnitudes
        )
    )

    screen_landmarks = landmark_outputs[
        screen_index
    ]

    world_landmarks = np.zeros(
        (21, 3),
        dtype=np.float32
    )

    if len(landmark_outputs) > 1:

        for i, output in enumerate(
            landmark_outputs
        ):

            if i != screen_index:

                world_landmarks = output

                break

    return (
        screen_landmarks,
        world_landmarks,
        scalar_outputs
    )


# ==========================================================================================
# 20. PROJECT LANDMARKS BACK INTO ORIGINAL VIDEO
# ==========================================================================================

def project_landmarks_back(
    landmarks,
    M_inv
):

    xy = landmarks[
        :,
        :2
    ].reshape(
        -1,
        1,
        2
    )

    projected_xy = cv2.perspectiveTransform(
        xy.astype(
            np.float32
        ),
        M_inv
    ).reshape(
        -1,
        2
    )

    projected = np.concatenate(
        [
            projected_xy,

            landmarks[
                :,
                2:3
            ]
        ],
        axis=1
    )

    return projected.astype(
        np.float32
    )


# ==========================================================================================
# 21. EXTRACT UP TO TWO HANDS FROM ONE FRAME
# ==========================================================================================

def extract_frame_hands(frame):

    start = time.perf_counter()

    regression, logits, info = palm_inference(
        frame
    )

    detections = decode_palms(
        regression,
        logits
    )

    detections = weighted_nms(
        detections
    )

    detections = sorted(
        detections,
        key=lambda x: x["score"],
        reverse=True
    )[:2]

    hands = []

    for detection in detections:

        try:

            original_detection = palm_detection_to_original(
                detection,
                info
            )

            rect = palm_to_hand_rect(
                original_detection
            )

            crop, M_inv = crop_hand_roi(
                frame,
                rect
            )

            (
                screen_landmarks,
                world_landmarks,
                scalar_outputs
            ) = run_hand_landmark(
                crop
            )

            projected = project_landmarks_back(
                screen_landmarks,
                M_inv
            )

            hands.append(
                {
                    "landmarks": projected,

                    "world_landmarks":
                        world_landmarks,

                    "palm_score":
                        float(
                            original_detection[
                                "score"
                            ]
                        ),

                    "scalars":
                        scalar_outputs
                }
            )

        except Exception:
            continue

    elapsed_ms = (
        time.perf_counter()
        -
        start
    ) * 1000.0

    return (
        hands,
        elapsed_ms
    )


# ==========================================================================================
# 22. TEMPORAL SLOT STABILIZATION
# ==========================================================================================

def stabilize_hand_slots(
    landmarks,
    hand_mask,
    palm_scores
):

    T = landmarks.shape[0]

    tracked_landmarks = np.zeros_like(
        landmarks,
        dtype=np.float32
    )

    tracked_mask = np.zeros_like(
        hand_mask,
        dtype=np.uint8
    )

    tracked_scores = np.zeros_like(
        palm_scores,
        dtype=np.float32
    )

    previous_wrists = [
        None,
        None
    ]

    for t in range(T):

        detected = np.where(
            hand_mask[t] > 0
        )[0]

        if len(detected) == 0:
            continue

        # ------------------------------------------------------------------
        # ONE HAND
        # ------------------------------------------------------------------

        if len(detected) == 1:

            source = int(
                detected[0]
            )

            wrist = landmarks[
                t,
                source,
                0,
                :2
            ]

            previous_available = [
                i
                for i in range(2)
                if previous_wrists[i] is not None
            ]

            if len(previous_available) > 0:

                distances = [
                    np.linalg.norm(
                        wrist
                        -
                        previous_wrists[i]
                    )
                    for i in previous_available
                ]

                destination = previous_available[
                    int(
                        np.argmin(
                            distances
                        )
                    )
                ]

            else:

                destination = 0

            tracked_landmarks[
                t,
                destination
            ] = landmarks[
                t,
                source
            ]

            tracked_mask[
                t,
                destination
            ] = 1

            tracked_scores[
                t,
                destination
            ] = palm_scores[
                t,
                source
            ]

            previous_wrists[
                destination
            ] = wrist.copy()

        # ------------------------------------------------------------------
        # TWO HANDS
        # ------------------------------------------------------------------

        else:

            source0 = int(
                detected[0]
            )

            source1 = int(
                detected[1]
            )

            wrist0 = landmarks[
                t,
                source0,
                0,
                :2
            ]

            wrist1 = landmarks[
                t,
                source1,
                0,
                :2
            ]

            if (
                previous_wrists[0] is not None
                and
                previous_wrists[1] is not None
            ):

                keep_cost = (
                    np.linalg.norm(
                        wrist0
                        -
                        previous_wrists[0]
                    )
                    +
                    np.linalg.norm(
                        wrist1
                        -
                        previous_wrists[1]
                    )
                )

                swap_cost = (
                    np.linalg.norm(
                        wrist0
                        -
                        previous_wrists[1]
                    )
                    +
                    np.linalg.norm(
                        wrist1
                        -
                        previous_wrists[0]
                    )
                )

                if keep_cost <= swap_cost:

                    assignment = [
                        (
                            source0,
                            0
                        ),
                        (
                            source1,
                            1
                        )
                    ]

                else:

                    assignment = [
                        (
                            source0,
                            1
                        ),
                        (
                            source1,
                            0
                        )
                    ]

            else:

                # Initial ordering by image x-coordinate.

                if wrist0[0] <= wrist1[0]:

                    assignment = [
                        (
                            source0,
                            0
                        ),
                        (
                            source1,
                            1
                        )
                    ]

                else:

                    assignment = [
                        (
                            source1,
                            0
                        ),
                        (
                            source0,
                            1
                        )
                    ]

            for source, destination in assignment:

                tracked_landmarks[
                    t,
                    destination
                ] = landmarks[
                    t,
                    source
                ]

                tracked_mask[
                    t,
                    destination
                ] = 1

                tracked_scores[
                    t,
                    destination
                ] = palm_scores[
                    t,
                    source
                ]

                previous_wrists[
                    destination
                ] = landmarks[
                    t,
                    source,
                    0,
                    :2
                ].copy()

    return (
        tracked_landmarks,
        tracked_mask,
        tracked_scores
    )


# ==========================================================================================
# 23. SHORT-GAP INTERPOLATION
# ==========================================================================================

def interpolate_short_gaps(
    hand_landmarks,
    hand_mask,
    max_gap=3
):

    landmarks = np.asarray(
        hand_landmarks,
        dtype=np.float32
    ).copy()

    mask = np.asarray(
        hand_mask,
        dtype=np.uint8
    )

    T = len(mask)

    # 0 = missing
    # 1 = original
    # 2 = interpolated

    state = np.zeros(
        T,
        dtype=np.uint8
    )

    state[
        mask > 0
    ] = 1

    valid_indices = np.where(
        mask > 0
    )[0]

    if len(valid_indices) < 2:

        return (
            landmarks,
            state
        )

    for i in range(
        len(valid_indices) - 1
    ):

        left = int(
            valid_indices[i]
        )

        right = int(
            valid_indices[i + 1]
        )

        gap = (
            right
            -
            left
            -
            1
        )

        if (
            gap <= 0
            or
            gap > max_gap
        ):
            continue

        for t in range(
            left + 1,
            right
        ):

            alpha = (
                t - left
            ) / (
                right - left
            )

            landmarks[t] = (
                (1.0 - alpha)
                *
                landmarks[left]
                +
                alpha
                *
                landmarks[right]
            )

            state[t] = 2

    return (
        landmarks,
        state
    )


# ==========================================================================================
# 24. L5.1 HAND SCALE
# ==========================================================================================

def calculate_hand_scale(
    hand_landmarks
):

    hand = np.asarray(
        hand_landmarks,
        dtype=np.float32
    )

    if hand.shape != (21, 3):
        return 0.0

    if not np.isfinite(hand).all():
        return 0.0

    wrist = hand[
        0,
        :2
    ]

    distances = []

    for index in [
        5,
        9,
        13,
        17
    ]:

        distance = float(
            np.linalg.norm(
                hand[
                    index,
                    :2
                ]
                -
                wrist
            )
        )

        if (
            np.isfinite(distance)
            and
            distance > EPS_SCALE
        ):

            distances.append(
                distance
            )

    if len(distances) == 0:
        return 0.0

    scale = float(
        np.mean(
            distances
        )
    )

    if (
        not np.isfinite(scale)
        or
        scale <= EPS_SCALE
    ):
        return 0.0

    return scale


def calculate_raw_scales(
    landmarks,
    state
):

    T = landmarks.shape[0]

    scales = np.zeros(
        (
            T,
            2
        ),
        dtype=np.float32
    )

    for t in range(T):

        for hand in range(2):

            if state[
                t,
                hand
            ] == 0:

                continue

            scales[
                t,
                hand
            ] = calculate_hand_scale(
                landmarks[
                    t,
                    hand
                ]
            )

    return scales


# ==========================================================================================
# 25. L5.1 TEMPORAL SCALE REPAIR
# ==========================================================================================

def repair_temporal_scales(
    raw_scales,
    state
):

    robust = raw_scales.copy()

    repairs = []

    T = raw_scales.shape[0]

    for hand in range(2):

        # IMPORTANT:
        # Decisions use original scale sequence.
        # Repairs do not influence subsequent repair decisions.

        original = raw_scales[
            :,
            hand
        ].copy()

        for t in range(
            1,
            T - 1
        ):

            if not (
                state[
                    t - 1,
                    hand
                ] > 0
                and
                state[
                    t,
                    hand
                ] > 0
                and
                state[
                    t + 1,
                    hand
                ] > 0
            ):
                continue

            previous_scale = float(
                original[
                    t - 1
                ]
            )

            current_scale = float(
                original[t]
            )

            next_scale = float(
                original[
                    t + 1
                ]
            )

            if (
                previous_scale <= EPS_SCALE
                or
                current_scale <= EPS_SCALE
                or
                next_scale <= EPS_SCALE
            ):
                continue

            neighbor_ratio = (
                max(
                    previous_scale,
                    next_scale
                )
                /
                max(
                    min(
                        previous_scale,
                        next_scale
                    ),
                    EPS_SCALE
                )
            )

            if (
                neighbor_ratio
                >
                NEIGHBOR_AGREEMENT_RATIO
            ):
                continue

            reference = float(
                np.median(
                    [
                        previous_scale,
                        next_scale
                    ]
                )
            )

            outlier_ratio = (
                max(
                    current_scale,
                    reference
                )
                /
                max(
                    min(
                        current_scale,
                        reference
                    ),
                    EPS_SCALE
                )
            )

            if (
                outlier_ratio
                <
                SCALE_RATIO_THRESHOLD
            ):
                continue

            robust[
                t,
                hand
            ] = reference

            repairs.append(
                {
                    "frame": int(t),
                    "track": int(hand),

                    "raw_scale":
                        current_scale,

                    "repaired_scale":
                        reference,

                    "outlier_ratio":
                        float(
                            outlier_ratio
                        )
                }
            )

    return (
        robust,
        repairs
    )


# ==========================================================================================
# 26. ROBUST NORMALIZATION
# ==========================================================================================

def normalize_sequence_robust(
    landmarks,
    state,
    scales
):

    T = landmarks.shape[0]

    normalized = np.zeros(
        (
            T,
            2,
            21,
            3
        ),
        dtype=np.float32
    )

    for t in range(T):

        for hand in range(2):

            if state[
                t,
                hand
            ] == 0:

                continue

            scale = float(
                scales[
                    t,
                    hand
                ]
            )

            if (
                not np.isfinite(scale)
                or
                scale <= EPS_SCALE
            ):

                continue

            current = landmarks[
                t,
                hand
            ]

            wrist = current[0]

            normalized[
                t,
                hand
            ] = (
                current
                -
                wrist
            ) / scale

    normalized = np.nan_to_num(
        normalized,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return normalized.astype(
        np.float32
    )


# ==========================================================================================
# 27. MOTION
# ==========================================================================================

def calculate_motion(
    normalized,
    state
):

    motion = np.zeros_like(
        normalized,
        dtype=np.float32
    )

    for t in range(
        1,
        normalized.shape[0]
    ):

        for hand in range(2):

            if (
                state[
                    t,
                    hand
                ] > 0
                and
                state[
                    t - 1,
                    hand
                ] > 0
            ):

                motion[
                    t,
                    hand
                ] = (
                    normalized[
                        t,
                        hand
                    ]
                    -
                    normalized[
                        t - 1,
                        hand
                    ]
                )

    return motion


# ==========================================================================================
# 28. BUILD 254-D FEATURE VECTOR
# ==========================================================================================

def build_features(
    normalized,
    motion,
    state
):

    T = normalized.shape[0]

    normalized_flat = normalized.reshape(
        T,
        126
    )

    motion_flat = motion.reshape(
        T,
        126
    )

    presence = (
        state > 0
    ).astype(
        np.float32
    )

    features = np.concatenate(
        [
            normalized_flat,
            motion_flat,
            presence
        ],
        axis=1
    ).astype(
        np.float32
    )

    if features.shape[1] != FEATURE_DIM:

        raise RuntimeError(
            f"Expected {FEATURE_DIM} features, "
            f"got {features.shape}"
        )

    return features


# ==========================================================================================
# 29. TEMPORAL RESAMPLING
# ==========================================================================================

def resample_indices(
    length,
    target_length=32
):

    if length <= 0:

        raise ValueError(
            "Cannot resample zero-frame sequence."
        )

    indices = np.linspace(
        0,
        length - 1,
        target_length
    )

    indices = np.rint(
        indices
    ).astype(
        np.int32
    )

    return np.clip(
        indices,
        0,
        length - 1
    )


# ==========================================================================================
# 30. PROCESS ONE VIDEO
# ==========================================================================================

def process_video(
    video_path,
    uid,
    gloss
):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"Could not open video: {video_path}"
        )

    fps = float(
        cap.get(
            cv2.CAP_PROP_FPS
        )
    )

    width = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )

    height = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )

    reported_frames = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    landmarks_list = []
    world_list = []
    mask_list = []
    score_list = []

    scalar0_list = []
    scalar1_list = []

    frame_numbers = []
    latencies = []

    frame_index = 0

    while True:

        ok, frame = cap.read()

        if not ok:
            break

        hands, latency = extract_frame_hands(
            frame
        )

        # --------------------------------------------------------------
        # Temporary per-frame source ordering.
        #
        # Final temporal Track0 / Track1 stabilization happens later.
        # --------------------------------------------------------------

        hands = sorted(
            hands,
            key=lambda x:
                float(
                    x["landmarks"][
                        0,
                        0
                    ]
                )
        )[:2]

        frame_landmarks = np.zeros(
            (
                2,
                21,
                3
            ),
            dtype=np.float32
        )

        frame_world = np.zeros(
            (
                2,
                21,
                3
            ),
            dtype=np.float32
        )

        frame_mask = np.zeros(
            2,
            dtype=np.uint8
        )

        frame_scores = np.zeros(
            2,
            dtype=np.float32
        )

        # We retain the two scalar outputs but DO NOT assign
        # semantic labels until independently verified.

        frame_scalar0 = np.full(
            2,
            np.nan,
            dtype=np.float32
        )

        frame_scalar1 = np.full(
            2,
            np.nan,
            dtype=np.float32
        )

        for slot, hand in enumerate(
            hands
        ):

            frame_landmarks[
                slot
            ] = hand[
                "landmarks"
            ]

            frame_world[
                slot
            ] = hand[
                "world_landmarks"
            ]

            frame_mask[
                slot
            ] = 1

            frame_scores[
                slot
            ] = hand[
                "palm_score"
            ]

            scalars = hand[
                "scalars"
            ]

            if len(scalars) >= 1:

                frame_scalar0[
                    slot
                ] = scalars[0]

            if len(scalars) >= 2:

                frame_scalar1[
                    slot
                ] = scalars[1]

        landmarks_list.append(
            frame_landmarks
        )

        world_list.append(
            frame_world
        )

        mask_list.append(
            frame_mask
        )

        score_list.append(
            frame_scores
        )

        scalar0_list.append(
            frame_scalar0
        )

        scalar1_list.append(
            frame_scalar1
        )

        frame_numbers.append(
            frame_index
        )

        latencies.append(
            latency
        )

        frame_index += 1

    cap.release()

    if frame_index == 0:

        raise RuntimeError(
            "Video contains zero readable frames."
        )

    landmarks = np.stack(
        landmarks_list
    ).astype(
        np.float32
    )

    world_landmarks = np.stack(
        world_list
    ).astype(
        np.float32
    )

    hand_mask = np.stack(
        mask_list
    ).astype(
        np.uint8
    )

    palm_scores = np.stack(
        score_list
    ).astype(
        np.float32
    )

    scalar0 = np.stack(
        scalar0_list
    ).astype(
        np.float32
    )

    scalar1 = np.stack(
        scalar1_list
    ).astype(
        np.float32
    )

    frame_numbers = np.asarray(
        frame_numbers,
        dtype=np.int32
    )

    # ======================================================================================
    # Original hand coverage
    # ======================================================================================

    original_any_hand = (
        np.sum(
            hand_mask,
            axis=1
        )
        >
        0
    )

    original_valid_frames = int(
        np.sum(
            original_any_hand
        )
    )

    original_detection_rate = float(
        original_valid_frames
        /
        frame_index
    )

    # ======================================================================================
    # Temporal tracking
    # ======================================================================================

    (
        tracked_landmarks,
        tracked_mask,
        tracked_scores
    ) = stabilize_hand_slots(
        landmarks,
        hand_mask,
        palm_scores
    )

    # ======================================================================================
    # Interpolation
    # ======================================================================================

    repaired_landmarks = (
        tracked_landmarks.copy()
    )

    state = np.zeros(
        (
            frame_index,
            2
        ),
        dtype=np.uint8
    )

    for hand in range(2):

        (
            repaired_hand,
            hand_state
        ) = interpolate_short_gaps(
            tracked_landmarks[
                :,
                hand
            ],

            tracked_mask[
                :,
                hand
            ],

            MAX_INTERPOLATION_GAP
        )

        repaired_landmarks[
            :,
            hand
        ] = repaired_hand

        state[
            :,
            hand
        ] = hand_state

    repaired_any_hand = (
        np.sum(
            state > 0,
            axis=1
        )
        >
        0
    )

    repaired_detection_rate = float(
        np.mean(
            repaired_any_hand
        )
    )

    # ======================================================================================
    # L5.1 robust scale
    # ======================================================================================

    raw_scales = calculate_raw_scales(
        repaired_landmarks,
        state
    )

    (
        robust_scales,
        scale_repairs
    ) = repair_temporal_scales(
        raw_scales,
        state
    )

    # ======================================================================================
    # Normalize
    # ======================================================================================

    normalized = normalize_sequence_robust(
        repaired_landmarks,
        state,
        robust_scales
    )

    # ======================================================================================
    # Motion
    # ======================================================================================

    motion = calculate_motion(
        normalized,
        state
    )

    # ======================================================================================
    # Features
    # ======================================================================================

    features_full = build_features(
        normalized,
        motion,
        state
    )

    # ======================================================================================
    # Resample to exactly 32 frames
    # ======================================================================================

    sample_indices = resample_indices(
        frame_index,
        SEQUENCE_LENGTH
    )

    features_32 = features_full[
        sample_indices
    ].astype(
        np.float32
    )

    normalized_32 = normalized[
        sample_indices
    ].astype(
        np.float32
    )

    motion_32 = motion[
        sample_indices
    ].astype(
        np.float32
    )

    state_32 = state[
        sample_indices
    ].astype(
        np.uint8
    )

    raw_scales_32 = raw_scales[
        sample_indices
    ].astype(
        np.float32
    )

    robust_scales_32 = robust_scales[
        sample_indices
    ].astype(
        np.float32
    )

    sampled_detection_rate = float(
        np.mean(
            np.sum(
                state_32 > 0,
                axis=1
            )
            >
            0
        )
    )

    # ======================================================================================
    # Numerical checks
    # ======================================================================================

    finite = bool(
        np.isfinite(
            features_32
        ).all()
    )

    correct_shape = (
        features_32.shape
        ==
        (
            SEQUENCE_LENGTH,
            FEATURE_DIM
        )
    )

    trainable = bool(
        original_valid_frames
        >=
        MIN_VALID_FRAMES
        and
        original_detection_rate
        >=
        MIN_DETECTION_RATE_FOR_TRAINING
        and
        finite
        and
        correct_shape
    )

    max_abs_normalized = float(
        np.max(
            np.abs(
                normalized_32
            )
        )
    )

    max_abs_motion = float(
        np.max(
            np.abs(
                motion_32
            )
        )
    )

    max_abs_feature = float(
        np.max(
            np.abs(
                features_32
            )
        )
    )

    return {
        "uid":
            uid,

        "gloss":
            gloss,

        "video_path":
            str(video_path),

        "fps":
            fps,

        "width":
            width,

        "height":
            height,

        "reported_frames":
            reported_frames,

        "processed_frames":
            frame_index,

        "frame_numbers":
            frame_numbers,

        # --------------------------------------------------------------
        # Raw extraction
        # --------------------------------------------------------------

        "landmarks_px":
            landmarks,

        "world_landmarks":
            world_landmarks,

        "hand_mask":
            hand_mask,

        "palm_scores":
            palm_scores,

        "scalar0":
            scalar0,

        "scalar1":
            scalar1,

        # --------------------------------------------------------------
        # Derived full temporal state
        # --------------------------------------------------------------

        "tracked_landmarks":
            tracked_landmarks,

        "tracked_mask":
            tracked_mask,

        "tracked_scores":
            tracked_scores,

        "repaired_landmarks":
            repaired_landmarks,

        "state":
            state,

        "raw_scales":
            raw_scales,

        "robust_scales":
            robust_scales,

        # --------------------------------------------------------------
        # 32-frame representation
        # --------------------------------------------------------------

        "sample_indices":
            sample_indices,

        "normalized_32":
            normalized_32,

        "motion_32":
            motion_32,

        "state_32":
            state_32,

        "raw_scales_32":
            raw_scales_32,

        "robust_scales_32":
            robust_scales_32,

        "features_32":
            features_32,

        # --------------------------------------------------------------
        # Audit
        # --------------------------------------------------------------

        "original_valid_frames":
            original_valid_frames,

        "original_detection_rate":
            original_detection_rate,

        "repaired_detection_rate":
            repaired_detection_rate,

        "sampled_detection_rate":
            sampled_detection_rate,

        "scale_repairs":
            scale_repairs,

        "scale_repair_count":
            len(scale_repairs),

        "mean_latency_ms":
            float(
                np.mean(
                    latencies
                )
            ),

        "median_latency_ms":
            float(
                np.median(
                    latencies
                )
            ),

        "max_abs_normalized":
            max_abs_normalized,

        "max_abs_motion":
            max_abs_motion,

        "max_abs_feature":
            max_abs_feature,

        "finite":
            finite,

        "correct_shape":
            correct_shape,

        "trainable":
            trainable
    }


# ==========================================================================================
# 31. SAVE ONE VIDEO
# ==========================================================================================

def save_video_result(result):

    uid = result["uid"]

    raw_path = (
        RAW_DIR
        /
        f"{uid}.npz"
    )

    feature_path = (
        FEATURE_DIR
        /
        f"{uid}.npz"
    )

    # ======================================================================================
    # RAW DATA
    # ======================================================================================

    if SAVE_RAW:

        np.savez_compressed(
            raw_path,

            uid=np.array(
                result["uid"]
            ),

            gloss=np.array(
                result["gloss"]
            ),

            video_path=np.array(
                result["video_path"]
            ),

            fps=np.array(
                result["fps"],
                dtype=np.float32
            ),

            width=np.array(
                result["width"],
                dtype=np.int32
            ),

            height=np.array(
                result["height"],
                dtype=np.int32
            ),

            frame_numbers=
                result["frame_numbers"],

            landmarks_px=
                result["landmarks_px"],

            world_landmarks=
                result["world_landmarks"],

            hand_mask=
                result["hand_mask"],

            palm_scores=
                result["palm_scores"],

            # Scalar meanings intentionally NOT guessed.

            hand_scalar_0=
                result["scalar0"],

            hand_scalar_1=
                result["scalar1"]
        )

    # ======================================================================================
    # FINAL 32 × 254 REPRESENTATION
    # ======================================================================================

    np.savez_compressed(
        feature_path,

        uid=np.array(
            result["uid"]
        ),

        gloss=np.array(
            result["gloss"]
        ),

        features=
            result["features_32"],

        normalized_landmarks=
            result["normalized_32"],

        motion=
            result["motion_32"],

        hand_state=
            result["state_32"],

        raw_scales=
            result["raw_scales_32"],

        robust_scales=
            result["robust_scales_32"],

        sample_indices=
            result["sample_indices"],

        sequence_length=np.array(
            SEQUENCE_LENGTH,
            dtype=np.int32
        ),

        feature_dim=np.array(
            FEATURE_DIM,
            dtype=np.int32
        ),

        original_detection_rate=np.array(
            result[
                "original_detection_rate"
            ],
            dtype=np.float32
        ),

        repaired_detection_rate=np.array(
            result[
                "repaired_detection_rate"
            ],
            dtype=np.float32
        ),

        sampled_detection_rate=np.array(
            result[
                "sampled_detection_rate"
            ],
            dtype=np.float32
        ),

        scale_repairs=np.array(
            result[
                "scale_repair_count"
            ],
            dtype=np.int32
        ),

        trainable=np.array(
            result["trainable"],
            dtype=np.bool_
        )
    )

    return (
        raw_path,
        feature_path
    )


# ==========================================================================================
# 32. LOAD CISLR METADATA
# ==========================================================================================

print("\n" + "=" * 120)
print("DATASET INDEX")
print("=" * 120)

dataset = pd.read_csv(
    DATASET_CSV
)

required_columns = {
    "uid",
    "gloss"
}

missing_columns = (
    required_columns
    -
    set(dataset.columns)
)

if missing_columns:

    raise RuntimeError(
        f"dataset.csv missing columns: "
        f"{missing_columns}"
    )

dataset["uid"] = (
    dataset["uid"]
    .astype(str)
)

dataset["gloss"] = (
    dataset["gloss"]
    .astype(str)
)


# ==========================================================================================
# 33. INDEX MP4 FILES
# ==========================================================================================

print("\nIndexing MP4 files...")

video_files = list(
    VIDEO_DIR.rglob(
        "*.mp4"
    )
)

video_lookup = {
    path.stem: path
    for path in video_files
}

dataset["video_path"] = (
    dataset["uid"]
    .map(video_lookup)
)

matched = int(
    dataset[
        "video_path"
    ].notna().sum()
)

missing = int(
    dataset[
        "video_path"
    ].isna().sum()
)

print("\nDataset rows   :", len(dataset))
print("Unique glosses :", dataset["gloss"].nunique())
print("MP4 files      :", len(video_files))
print("Matched        :", matched)
print("Missing        :", missing)

available_df = dataset[
    dataset[
        "video_path"
    ].notna()
].copy().reset_index(
    drop=True
)

if len(available_df) == 0:

    raise RuntimeError(
        "No dataset videos matched."
    )


# ==========================================================================================
# 34. AUTOMATIC SMOKE TEST
# ==========================================================================================

print("\n" + "=" * 120)
print("L6 AUTOMATIC SMOKE TEST")
print("=" * 120)

smoke_row = available_df.iloc[0]

smoke_uid = str(
    smoke_row["uid"]
)

smoke_gloss = str(
    smoke_row["gloss"]
)

smoke_path = Path(
    smoke_row["video_path"]
)

print("Video :", smoke_path)
print("UID   :", smoke_uid)
print("Gloss :", smoke_gloss)

smoke_result = process_video(
    smoke_path,
    smoke_uid,
    smoke_gloss
)

print("\nFrames            :", smoke_result["processed_frames"])

print(
    "Original detection:",
    f"{smoke_result['original_detection_rate'] * 100:.2f}%"
)

print(
    "Repaired detection:",
    f"{smoke_result['repaired_detection_rate'] * 100:.2f}%"
)

print(
    "Sampled detection :",
    f"{smoke_result['sampled_detection_rate'] * 100:.2f}%"
)

print(
    "Features          :",
    smoke_result[
        "features_32"
    ].shape
)

print(
    "Normalized        :",
    smoke_result[
        "normalized_32"
    ].shape
)

print(
    "Motion            :",
    smoke_result[
        "motion_32"
    ].shape
)

print(
    "State             :",
    smoke_result[
        "state_32"
    ].shape
)

print(
    "Scale repairs     :",
    smoke_result[
        "scale_repair_count"
    ]
)

print(
    "Max |normalized|  :",
    f"{smoke_result['max_abs_normalized']:.6f}"
)

print(
    "Max |motion|      :",
    f"{smoke_result['max_abs_motion']:.6f}"
)

print(
    "Max |feature|     :",
    f"{smoke_result['max_abs_feature']:.6f}"
)

print(
    "Finite            :",
    smoke_result[
        "finite"
    ]
)

print(
    "Correct shape     :",
    smoke_result[
        "correct_shape"
    ]
)

print(
    "Trainable         :",
    smoke_result[
        "trainable"
    ]
)

print(
    "Mean L1 latency   :",
    f"{smoke_result['mean_latency_ms']:.3f} ms/frame"
)

if not smoke_result["finite"]:

    raise RuntimeError(
        "SMOKE TEST FAILED: non-finite features."
    )

if not smoke_result["correct_shape"]:

    raise RuntimeError(
        "SMOKE TEST FAILED: representation is not 32×254."
    )

print("\n[PASS] L6 smoke test passed.")
print("[INFO] Starting full CISLR extraction...")


# ==========================================================================================
# 35. LOAD OLD MANIFEST FOR RESUME
# ==========================================================================================

previous_lookup = {}

if (
    RESUME
    and
    MANIFEST_PATH.exists()
):

    try:

        previous_manifest = pd.read_csv(
            MANIFEST_PATH
        )

        for _, old_row in previous_manifest.iterrows():

            previous_lookup[
                str(
                    old_row["uid"]
                )
            ] = old_row.to_dict()

        print(
            "\n[RESUME] Existing manifest rows:",
            len(previous_manifest)
        )

    except Exception as error:

        print(
            "[WARNING] Existing manifest could not be loaded:",
            repr(error)
        )


# ==========================================================================================
# 36. FULL DATASET EXTRACTION
# ==========================================================================================

print("\n" + "=" * 120)
print("FULL CISLR EXTRACTION")
print("=" * 120)

total_videos = len(
    available_df
)

manifest_rows = []
failure_rows = []
scale_repair_rows = []

completed = 0
skipped = 0
failed = 0

run_start = time.time()


for dataset_index, row in available_df.iterrows():

    uid = str(
        row["uid"]
    )

    gloss = str(
        row["gloss"]
    )

    video_path = Path(
        row["video_path"]
    )

    feature_path = (
        FEATURE_DIR
        /
        f"{uid}.npz"
    )

    raw_path = (
        RAW_DIR
        /
        f"{uid}.npz"
    )

    # ======================================================================================
    # RESUME CHECK
    # ======================================================================================

    valid_existing = False

    if (
        RESUME
        and
        feature_path.exists()
        and
        (
            not SAVE_RAW
            or
            raw_path.exists()
        )
    ):

        try:

            with np.load(
                feature_path,
                allow_pickle=True
            ) as existing:

                existing_features = np.asarray(
                    existing["features"],
                    dtype=np.float32
                )

                valid_existing = (
                    existing_features.shape
                    ==
                    (
                        SEQUENCE_LENGTH,
                        FEATURE_DIM
                    )
                    and
                    np.isfinite(
                        existing_features
                    ).all()
                )

        except Exception:

            valid_existing = False

    # ======================================================================================
    # SKIP EXISTING
    # ======================================================================================

    if valid_existing:

        skipped += 1

        if uid in previous_lookup:

            manifest_rows.append(
                previous_lookup[
                    uid
                ]
            )

        else:

            manifest_rows.append(
                {
                    "uid":
                        uid,

                    "gloss":
                        gloss,

                    "video_path":
                        str(video_path),

                    "status":
                        "EXISTING",

                    "processed_frames":
                        np.nan,

                    "original_detection_rate":
                        np.nan,

                    "repaired_detection_rate":
                        np.nan,

                    "sampled_detection_rate":
                        np.nan,

                    "scale_repairs":
                        np.nan,

                    "mean_frame_latency_ms":
                        np.nan,

                    "max_abs_normalized":
                        np.nan,

                    "max_abs_motion":
                        np.nan,

                    "max_abs_feature":
                        np.nan,

                    "trainable":
                        np.nan,

                    "raw_path":
                        str(raw_path),

                    "feature_path":
                        str(feature_path)
                }
            )

        if VERBOSE_PROGRESS:

            print(
                f"[{dataset_index + 1:04d}/{total_videos:04d}] "
                f"SKIP | "
                f"{uid:20s} | "
                f"{gloss[:25]}"
            )

        continue

    # ======================================================================================
    # PROCESS NEW VIDEO
    # ======================================================================================

    video_start = time.time()

    try:

        result = process_video(
            video_path,
            uid,
            gloss
        )

        (
            saved_raw,
            saved_feature
        ) = save_video_result(
            result
        )

        video_seconds = (
            time.time()
            -
            video_start
        )

        manifest_rows.append(
            {
                "uid":
                    uid,

                "gloss":
                    gloss,

                "video_path":
                    str(video_path),

                "status":
                    "SUCCESS",

                "processed_frames":
                    result[
                        "processed_frames"
                    ],

                "reported_frames":
                    result[
                        "reported_frames"
                    ],

                "width":
                    result[
                        "width"
                    ],

                "height":
                    result[
                        "height"
                    ],

                "fps":
                    result[
                        "fps"
                    ],

                "original_valid_frames":
                    result[
                        "original_valid_frames"
                    ],

                "original_detection_rate":
                    result[
                        "original_detection_rate"
                    ],

                "repaired_detection_rate":
                    result[
                        "repaired_detection_rate"
                    ],

                "sampled_detection_rate":
                    result[
                        "sampled_detection_rate"
                    ],

                "scale_repairs":
                    result[
                        "scale_repair_count"
                    ],

                "mean_frame_latency_ms":
                    result[
                        "mean_latency_ms"
                    ],

                "median_frame_latency_ms":
                    result[
                        "median_latency_ms"
                    ],

                "processing_seconds":
                    video_seconds,

                "max_abs_normalized":
                    result[
                        "max_abs_normalized"
                    ],

                "max_abs_motion":
                    result[
                        "max_abs_motion"
                    ],

                "max_abs_feature":
                    result[
                        "max_abs_feature"
                    ],

                "finite":
                    result[
                        "finite"
                    ],

                "correct_shape":
                    result[
                        "correct_shape"
                    ],

                "trainable":
                    result[
                        "trainable"
                    ],

                "raw_path":
                    str(saved_raw),

                "feature_path":
                    str(saved_feature)
            }
        )

        # ----------------------------------------------------------------------------------
        # Save scale-repair audit rows
        # ----------------------------------------------------------------------------------

        for repair in result[
            "scale_repairs"
        ]:

            scale_repair_rows.append(
                {
                    "uid":
                        uid,

                    "gloss":
                        gloss,

                    "frame":
                        repair[
                            "frame"
                        ],

                    "track":
                        repair[
                            "track"
                        ],

                    "raw_scale":
                        repair[
                            "raw_scale"
                        ],

                    "repaired_scale":
                        repair[
                            "repaired_scale"
                        ],

                    "outlier_ratio":
                        repair[
                            "outlier_ratio"
                        ]
                }
            )

        completed += 1

        if VERBOSE_PROGRESS:

            print(
                f"[{dataset_index + 1:04d}/{total_videos:04d}] "
                f"OK   | "
                f"{uid:20s} | "
                f"{gloss[:18]:18s} | "
                f"frames={result['processed_frames']:4d} | "
                f"det={result['original_detection_rate'] * 100:6.2f}% | "
                f"sample={result['sampled_detection_rate'] * 100:6.2f}% | "
                f"fix={result['scale_repair_count']:3d} | "
                f"{video_seconds:6.2f}s"
            )

    except Exception as error:

        failed += 1

        error_text = repr(
            error
        )

        failure_rows.append(
            {
                "uid":
                    uid,

                "gloss":
                    gloss,

                "video_path":
                    str(video_path),

                "error":
                    error_text,

                "traceback":
                    traceback.format_exc()
            }
        )

        manifest_rows.append(
            {
                "uid":
                    uid,

                "gloss":
                    gloss,

                "video_path":
                    str(video_path),

                "status":
                    "FAILED",

                "processed_frames":
                    np.nan,

                "original_detection_rate":
                    np.nan,

                "repaired_detection_rate":
                    np.nan,

                "sampled_detection_rate":
                    np.nan,

                "scale_repairs":
                    np.nan,

                "mean_frame_latency_ms":
                    np.nan,

                "max_abs_normalized":
                    np.nan,

                "max_abs_motion":
                    np.nan,

                "max_abs_feature":
                    np.nan,

                "trainable":
                    False,

                "raw_path":
                    "",

                "feature_path":
                    ""
            }
        )

        print(
            f"[{dataset_index + 1:04d}/{total_videos:04d}] "
            f"FAIL | "
            f"{uid:20s} | "
            f"{gloss[:20]:20s} | "
            f"{error_text}"
        )

    # ======================================================================================
    # CHECKPOINT
    # ======================================================================================

    attempted = (
        completed
        +
        failed
    )

    if (
        attempted > 0
        and
        attempted % CHECKPOINT_EVERY == 0
    ):

        checkpoint_df = pd.DataFrame(
            manifest_rows
        )

        checkpoint_df = checkpoint_df.drop_duplicates(
            subset=[
                "uid"
            ],
            keep="last"
        )

        checkpoint_df.to_csv(
            MANIFEST_PATH,
            index=False
        )

        if len(failure_rows) > 0:

            pd.DataFrame(
                failure_rows
            ).to_csv(
                FAILURE_PATH,
                index=False
            )

        if len(scale_repair_rows) > 0:

            pd.DataFrame(
                scale_repair_rows
            ).to_csv(
                SCALE_REPAIR_PATH,
                index=False
            )

        progress = {
            "total":
                total_videos,

            "completed_this_run":
                completed,

            "skipped_this_run":
                skipped,

            "failed_this_run":
                failed,

            "last_uid":
                uid,

            "last_index":
                int(
                    dataset_index
                ),

            "elapsed_minutes":
                float(
                    (
                        time.time()
                        -
                        run_start
                    )
                    /
                    60.0
                )
        }

        with open(
            PROGRESS_PATH,
            "w"
        ) as f:

            json.dump(
                progress,
                f,
                indent=4
            )

        print(
            "\n"
            f"[CHECKPOINT] "
            f"completed={completed} | "
            f"skipped={skipped} | "
            f"failed={failed} | "
            f"elapsed={(time.time() - run_start) / 60:.1f} min"
            "\n"
        )


# ==========================================================================================
# 37. SAVE FINAL MANIFEST
# ==========================================================================================

manifest_df = pd.DataFrame(
    manifest_rows
)

if len(manifest_df) > 0:

    manifest_df = (
        manifest_df
        .drop_duplicates(
            subset=[
                "uid"
            ],
            keep="last"
        )
        .reset_index(
            drop=True
        )
    )

manifest_df.to_csv(
    MANIFEST_PATH,
    index=False
)

if len(failure_rows) > 0:

    pd.DataFrame(
        failure_rows
    ).to_csv(
        FAILURE_PATH,
        index=False
    )

if len(scale_repair_rows) > 0:

    pd.DataFrame(
        scale_repair_rows
    ).to_csv(
        SCALE_REPAIR_PATH,
        index=False
    )


# ==========================================================================================
# 38. FINAL FEATURE AUDIT
# ==========================================================================================

print("\n" + "=" * 120)
print("L6 FINAL DATASET AUDIT")
print("=" * 120)

feature_files = sorted(
    FEATURE_DIR.glob(
        "*.npz"
    )
)

raw_files = sorted(
    RAW_DIR.glob(
        "*.npz"
    )
)

audit_rows = []
invalid_files = []

for feature_path in feature_files:

    try:

        with np.load(
            feature_path,
            allow_pickle=True
        ) as data:

            uid = str(
                data[
                    "uid"
                ].item()
            )

            gloss = str(
                data[
                    "gloss"
                ].item()
            )

            features = np.asarray(
                data[
                    "features"
                ],
                dtype=np.float32
            )

            state = np.asarray(
                data[
                    "hand_state"
                ],
                dtype=np.uint8
            )

            original_detection_rate = float(
                data[
                    "original_detection_rate"
                ].item()
            )

            repaired_detection_rate = float(
                data[
                    "repaired_detection_rate"
                ].item()
            )

            sampled_detection_rate = float(
                data[
                    "sampled_detection_rate"
                ].item()
            )

            scale_repairs = int(
                data[
                    "scale_repairs"
                ].item()
            )

            trainable = bool(
                data[
                    "trainable"
                ].item()
            )

        valid_shape = (
            features.shape
            ==
            (
                SEQUENCE_LENGTH,
                FEATURE_DIM
            )
        )

        finite = bool(
            np.isfinite(
                features
            ).all()
        )

        max_abs_feature = float(
            np.max(
                np.abs(
                    features
                )
            )
        )

        audit_rows.append(
            {
                "uid":
                    uid,

                "gloss":
                    gloss,

                "valid_shape":
                    valid_shape,

                "finite":
                    finite,

                "original_detection_rate":
                    original_detection_rate,

                "repaired_detection_rate":
                    repaired_detection_rate,

                "sampled_detection_rate":
                    sampled_detection_rate,

                "scale_repairs":
                    scale_repairs,

                "max_abs_feature":
                    max_abs_feature,

                "trainable":
                    trainable
            }
        )

        if (
            not valid_shape
            or
            not finite
        ):

            invalid_files.append(
                str(
                    feature_path
                )
            )

    except Exception as error:

        invalid_files.append(
            f"{feature_path}: {repr(error)}"
        )

audit_df = pd.DataFrame(
    audit_rows
)

audit_df.to_csv(
    AUDIT_PATH,
    index=False
)


# ==========================================================================================
# 39. FINAL STATISTICS
# ==========================================================================================

if len(audit_df) > 0:

    valid_shape_count = int(
        audit_df[
            "valid_shape"
        ].sum()
    )

    finite_count = int(
        audit_df[
            "finite"
        ].sum()
    )

    trainable_count = int(
        audit_df[
            "trainable"
        ].sum()
    )

    rejected_count = (
        len(audit_df)
        -
        trainable_count
    )

    mean_detection = float(
        audit_df[
            "original_detection_rate"
        ].mean()
    )

    median_detection = float(
        audit_df[
            "original_detection_rate"
        ].median()
    )

    mean_repaired_detection = float(
        audit_df[
            "repaired_detection_rate"
        ].mean()
    )

    mean_sampled_detection = float(
        audit_df[
            "sampled_detection_rate"
        ].mean()
    )

    total_scale_repairs = int(
        audit_df[
            "scale_repairs"
        ].sum()
    )

    videos_with_repairs = int(
        (
            audit_df[
                "scale_repairs"
            ]
            >
            0
        ).sum()
    )

    largest_feature = float(
        audit_df[
            "max_abs_feature"
        ].max()
    )

    good = int(
        (
            audit_df[
                "original_detection_rate"
            ]
            >=
            0.80
        ).sum()
    )

    review = int(
        (
            (
                audit_df[
                    "original_detection_rate"
                ]
                >=
                0.50
            )
            &
            (
                audit_df[
                    "original_detection_rate"
                ]
                <
                0.80
            )
        ).sum()
    )

    poor = int(
        (
            audit_df[
                "original_detection_rate"
            ]
            <
            0.50
        ).sum()
    )

else:

    valid_shape_count = 0
    finite_count = 0

    trainable_count = 0
    rejected_count = 0

    mean_detection = 0.0
    median_detection = 0.0

    mean_repaired_detection = 0.0
    mean_sampled_detection = 0.0

    total_scale_repairs = 0
    videos_with_repairs = 0

    largest_feature = 0.0

    good = 0
    review = 0
    poor = 0


# ==========================================================================================
# 40. SUMMARY JSON
# ==========================================================================================

runtime_minutes = (
    time.time()
    -
    run_start
) / 60.0

summary = {
    "stage":
        "L6_FULL_CISLR_LANDMARK_EXTRACTION",

    "representation":
        "32x254",

    "dataset_rows":
        int(
            len(dataset)
        ),

    "unique_glosses":
        int(
            dataset[
                "gloss"
            ].nunique()
        ),

    "matched_videos":
        int(
            len(
                available_df
            )
        ),

    "feature_files":
        int(
            len(
                feature_files
            )
        ),

    "raw_files":
        int(
            len(
                raw_files
            )
        ),

    "invalid_files":
        int(
            len(
                invalid_files
            )
        ),

    "valid_32x254":
        valid_shape_count,

    "finite":
        finite_count,

    "trainable":
        trainable_count,

    "rejected":
        rejected_count,

    "mean_original_detection_rate":
        mean_detection,

    "median_original_detection_rate":
        median_detection,

    "mean_repaired_detection_rate":
        mean_repaired_detection,

    "mean_sampled_detection_rate":
        mean_sampled_detection,

    "good_80_plus":
        good,

    "review_50_to_80":
        review,

    "poor_below_50":
        poor,

    "total_scale_repairs":
        total_scale_repairs,

    "videos_with_scale_repairs":
        videos_with_repairs,

    "largest_absolute_feature":
        largest_feature,

    "completed_this_run":
        completed,

    "skipped_this_run":
        skipped,

    "failed_this_run":
        failed,

    "runtime_minutes":
        float(
            runtime_minutes
        )
}

with open(
    SUMMARY_PATH,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# ==========================================================================================
# 41. FINAL REPORT
# ==========================================================================================

print("\n" + "=" * 120)
print("L6 FINAL REPORT")
print("=" * 120)

print("\nDATASET")
print("-" * 120)

print(
    "Dataset rows              :",
    len(dataset)
)

print(
    "Unique glosses            :",
    dataset[
        "gloss"
    ].nunique()
)

print(
    "Matched videos            :",
    len(
        available_df
    )
)

print("\nFILES")
print("-" * 120)

print(
    "Raw files                 :",
    len(
        raw_files
    )
)

print(
    "Feature files             :",
    len(
        feature_files
    )
)

print(
    "Invalid files             :",
    len(
        invalid_files
    )
)

print("\nSTRUCTURAL")
print("-" * 120)

print(
    "Valid 32 x 254            :",
    valid_shape_count
)

print(
    "Finite                    :",
    finite_count
)

print(
    "Largest |feature|         :",
    f"{largest_feature:.6f}"
)

print("\nTRAINING ELIGIBILITY")
print("-" * 120)

print(
    "Provisional trainable     :",
    trainable_count
)

print(
    "Provisional rejected      :",
    rejected_count
)

print("\nHAND COVERAGE")
print("-" * 120)

print(
    "Mean original detection   :",
    f"{mean_detection * 100:.2f}%"
)

print(
    "Median original detection :",
    f"{median_detection * 100:.2f}%"
)

print(
    "Mean repaired detection   :",
    f"{mean_repaired_detection * 100:.2f}%"
)

print(
    "Mean sampled detection    :",
    f"{mean_sampled_detection * 100:.2f}%"
)

print("\nQUALITY GROUPS")
print("-" * 120)

print(
    "GOOD   >= 80%             :",
    good
)

print(
    "REVIEW 50–80%             :",
    review
)

print(
    "POOR   < 50%              :",
    poor
)

print("\nL5.1 SCALE REPAIR")
print("-" * 120)

print(
    "Total scale repairs       :",
    total_scale_repairs
)

print(
    "Videos with repairs       :",
    videos_with_repairs
)

print("\nRUNTIME")
print("-" * 120)

print(
    "Completed this run        :",
    completed
)

print(
    "Skipped this run          :",
    skipped
)

print(
    "Failed this run           :",
    failed
)

print(
    "Runtime                   :",
    f"{runtime_minutes:.2f} minutes"
)


# ==========================================================================================
# 42. WORST COVERAGE VIDEOS
# ==========================================================================================

if len(audit_df) > 0:

    print("\n" + "=" * 120)
    print("20 LOWEST-COVERAGE VIDEOS")
    print("=" * 120)

    worst = (
        audit_df
        .sort_values(
            "original_detection_rate",
            ascending=True
        )
        .head(20)
    )

    print(
        worst[
            [
                "uid",
                "gloss",
                "original_detection_rate",
                "repaired_detection_rate",
                "sampled_detection_rate",
                "scale_repairs",
                "trainable"
            ]
        ].to_string(
            index=False
        )
    )


# ==========================================================================================
# 43. HIGHEST FEATURE MAGNITUDES
# ==========================================================================================

if len(audit_df) > 0:

    print("\n" + "=" * 120)
    print("10 LARGEST FEATURE MAGNITUDES")
    print("=" * 120)

    largest = (
        audit_df
        .sort_values(
            "max_abs_feature",
            ascending=False
        )
        .head(10)
    )

    print(
        largest[
            [
                "uid",
                "gloss",
                "max_abs_feature",
                "scale_repairs",
                "original_detection_rate"
            ]
        ].to_string(
            index=False
        )
    )


# ==========================================================================================
# 44. FINAL CHECKPOINT
# ==========================================================================================

complete = (
    len(feature_files)
    ==
    len(available_df)
)

structural = (
    valid_shape_count
    ==
    len(feature_files)
)

numerical = (
    finite_count
    ==
    len(feature_files)
)

print("\n" + "=" * 120)
print("L6 CHECKPOINT")
print("=" * 120)

if (
    complete
    and
    structural
    and
    numerical
):

    print()
    print(
        "[PASS] L6 FULL CISLR EXTRACTION COMPLETE."
    )

    print()

    print(
        f"{len(feature_files)} / "
        f"{len(available_df)} videos have "
        f"valid finite 32 x 254 representations."
    )

    print()

    print(
        "NEXT STAGE: L7 — FULL LANDMARK DATASET AUDIT"
    )

else:

    print()
    print(
        "[REVIEW] L6 IS NOT YET COMPLETE."
    )

    print()

    print(
        "Expected feature files :",
        len(
            available_df
        )
    )

    print(
        "Actual feature files   :",
        len(
            feature_files
        )
    )

    print(
        "Invalid files          :",
        len(
            invalid_files
        )
    )

    print(
        "Failed this run        :",
        failed
    )

    print()
    print(
        "You can rerun THIS SAME CELL."
    )

    print(
        "RESUME=True will skip valid completed videos "
        "and continue with the remaining videos."
    )


# ==========================================================================================
# 45. OUTPUT LOCATIONS
# ==========================================================================================

print("\n" + "=" * 120)
print("L6 OUTPUT FILES")
print("=" * 120)

print("\nRaw sequences:")
print(RAW_DIR)

print("\n32 x 254 features:")
print(FEATURE_DIR)

print("\nManifest:")
print(MANIFEST_PATH)

print("\nFinal audit:")
print(AUDIT_PATH)

print("\nSummary:")
print(SUMMARY_PATH)

print("\nFailures:")
print(FAILURE_PATH)

print("\nScale repairs:")
print(SCALE_REPAIR_PATH)

print("\n" + "=" * 120)

CISLR LANDMARK MODEL — L6
FULL CISLR LANDMARK EXTRACTION
Dataset       : /home/dipshikha/Music/cao/dataset.csv
Videos        : /home/dipshikha/Music/cao/CISLR_v1.5-a_videos
Palm model    : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/openvino/palm/palm_detection_full.xml
Landmark model: /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/openvino/hand_landmark/hand_landmark_full.xml
Output        : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l6_full_extraction
Device        : CPU
Representation: (32, 254)
Resume        : True

[PASS] Required files found.

Palm input : [1,192,192,3]
Palm outputs:
  0: [1,2016,18]
  1: [1,2016,1]

Hand input : [1,224,224,3]
Hand outputs:
  0: [1,63]
  1: [1,1]
  2: [1,1]
  3: [1,63]

[PASS] Models use expected NHWC layout.

Palm anchors: (2016, 4)
[PASS] Palm anchors generated.

DATASET INDEX

Indexing MP4 files...

Dataset rows   : 7050
Unique glosses : 4764
MP4 files      : 7050
Matched 

In [9]:
# ==========================================================================================
# CISLR LANDMARK MODEL — L7
# FULL LANDMARK DATASET AUDIT
#
# INPUT:
#   L6 features_32x254/*.npz
#   L6 raw_sequences/*.npz
#   dataset.csv
#
# PURPOSE:
#   1. Verify all L6 feature files.
#   2. Audit detection / missingness.
#   3. Audit Track0 / Track1 coverage.
#   4. Audit normalized XYZ and motion distributions.
#   5. Locate extreme feature values exactly:
#        UID -> sampled frame -> track -> landmark -> coordinate.
#   6. Investigate large feature magnitudes WITHOUT clipping.
#   7. Audit class / gloss support.
#   8. Determine provisional usable-video populations.
#   9. Determine how many classes support train/val/test experiments.
#  10. Produce evidence for L8.
#
# IMPORTANT:
#   L7 IS READ-ONLY WITH RESPECT TO L6.
#   NO FEATURE CLIPPING.
#   NO DATASET SPLIT.
#   NO TRAINING.
# ==========================================================================================


# ==========================================================================================
# 0. IMPORTS
# ==========================================================================================

from pathlib import Path
import json
import math
import warnings
import traceback

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

np.set_printoptions(
    precision=6,
    suppress=True
)

pd.set_option(
    "display.max_columns",
    100
)

pd.set_option(
    "display.width",
    220
)


# ==========================================================================================
# 1. PATHS
# ==========================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

DATASET_CSV = Path(
    "/home/dipshikha/Music/cao/dataset.csv"
)

LANDMARK_ROOT = (
    PROJECT_ROOT
    / "CISLR_LANDMARK_MODEL"
)

L6_ROOT = (
    LANDMARK_ROOT
    / "audit"
    / "l6_full_extraction"
)

RAW_DIR = (
    L6_ROOT
    / "raw_sequences"
)

FEATURE_DIR = (
    L6_ROOT
    / "features_32x254"
)

L6_REPORT_DIR = (
    L6_ROOT
    / "reports"
)

L7_ROOT = (
    LANDMARK_ROOT
    / "audit"
    / "l7_full_dataset_audit"
)

REPORT_DIR = (
    L7_ROOT
    / "reports"
)

OUTLIER_DIR = (
    L7_ROOT
    / "outliers"
)

CLASS_DIR = (
    L7_ROOT
    / "class_analysis"
)

for directory in [
    L7_ROOT,
    REPORT_DIR,
    OUTLIER_DIR,
    CLASS_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ==========================================================================================
# 2. OUTPUT FILES
# ==========================================================================================

VIDEO_AUDIT_CSV = (
    REPORT_DIR
    / "l7_video_level_audit.csv"
)

OUTLIER_CSV = (
    OUTLIER_DIR
    / "l7_feature_outliers.csv"
)

EXTREME_SEQUENCE_CSV = (
    OUTLIER_DIR
    / "l7_extreme_sequences.csv"
)

CLASS_SUPPORT_CSV = (
    CLASS_DIR
    / "l7_class_support.csv"
)

SUPPORT_SUMMARY_CSV = (
    CLASS_DIR
    / "l7_support_threshold_summary.csv"
)

QUALITY_CLASS_CSV = (
    CLASS_DIR
    / "l7_quality_class_summary.csv"
)

REJECTED_CSV = (
    REPORT_DIR
    / "l7_rejected_sequences.csv"
)

SUMMARY_JSON = (
    REPORT_DIR
    / "l7_summary.json"
)

CHECKPOINT_JSON = (
    REPORT_DIR
    / "l7_checkpoint.json"
)


# ==========================================================================================
# 3. FROZEN REPRESENTATION
# ==========================================================================================

SEQUENCE_LENGTH = 32

NUM_HANDS = 2

NUM_LANDMARKS = 21

NUM_COORDS = 3

NORMALIZED_DIM = 126

MOTION_DIM = 126

MASK_DIM = 2

FEATURE_DIM = 254


# ==========================================================================================
# 4. L7 AUDIT THRESHOLDS
# ==========================================================================================

# These are AUDIT thresholds.
# They do NOT modify data.

PROVISIONAL_MIN_DETECTION = 0.20

PROVISIONAL_MIN_SAMPLED_FRAMES = 8


# Magnitude levels for diagnostics only.

MAGNITUDE_LEVELS = [
    5.0,
    7.5,
    10.0,
    15.0,
    20.0,
    30.0
]


# We will save the top N extreme individual feature values.

TOP_N_OUTLIERS = 200


# ==========================================================================================
# 5. LANDMARK NAMES
# ==========================================================================================

LANDMARK_NAMES = [
    "WRIST",
    "THUMB_CMC",
    "THUMB_MCP",
    "THUMB_IP",
    "THUMB_TIP",
    "INDEX_MCP",
    "INDEX_PIP",
    "INDEX_DIP",
    "INDEX_TIP",
    "MIDDLE_MCP",
    "MIDDLE_PIP",
    "MIDDLE_DIP",
    "MIDDLE_TIP",
    "RING_MCP",
    "RING_PIP",
    "RING_DIP",
    "RING_TIP",
    "PINKY_MCP",
    "PINKY_PIP",
    "PINKY_DIP",
    "PINKY_TIP"
]

COORD_NAMES = [
    "x",
    "y",
    "z"
]


# ==========================================================================================
# 6. HEADER
# ==========================================================================================

print("=" * 120)
print("CISLR LANDMARK MODEL — L7")
print("FULL LANDMARK DATASET AUDIT")
print("=" * 120)

print("L6 feature directory :", FEATURE_DIR)
print("L6 raw directory     :", RAW_DIR)
print("L7 output            :", L7_ROOT)

print()
print("Expected sequence    :", (32, 254))
print("Expected normalized  :", (32, 2, 21, 3))
print("Expected motion      :", (32, 2, 21, 3))
print("Expected state       :", (32, 2))

print()
print("IMPORTANT:")
print("  L7 DOES NOT MODIFY L6 FEATURES.")
print("  LARGE VALUES ARE AUDITED, NOT CLIPPED.")

print("=" * 120)


# ==========================================================================================
# 7. VERIFY INPUTS
# ==========================================================================================

if not DATASET_CSV.exists():
    raise FileNotFoundError(DATASET_CSV)

if not FEATURE_DIR.exists():
    raise FileNotFoundError(FEATURE_DIR)

if not RAW_DIR.exists():
    raise FileNotFoundError(RAW_DIR)

feature_files = sorted(
    FEATURE_DIR.glob("*.npz")
)

raw_files = sorted(
    RAW_DIR.glob("*.npz")
)

print("\nINPUT COUNTS")
print("-" * 120)

print("Feature files :", len(feature_files))
print("Raw files     :", len(raw_files))

if len(feature_files) == 0:
    raise RuntimeError(
        "No L6 feature files found."
    )


# ==========================================================================================
# 8. LOAD DATASET METADATA
# ==========================================================================================

dataset = pd.read_csv(
    DATASET_CSV
)

if "uid" not in dataset.columns:
    raise RuntimeError(
        "dataset.csv has no uid column."
    )

if "gloss" not in dataset.columns:
    raise RuntimeError(
        "dataset.csv has no gloss column."
    )

dataset["uid"] = (
    dataset["uid"]
    .astype(str)
)

dataset["gloss"] = (
    dataset["gloss"]
    .astype(str)
)

print()
print("Dataset rows   :", len(dataset))
print("Unique glosses :", dataset["gloss"].nunique())


# ==========================================================================================
# 9. BASIC METADATA INTEGRITY
# ==========================================================================================

print("\n" + "=" * 120)
print("METADATA INTEGRITY")
print("=" * 120)

duplicate_uid_count = int(
    dataset["uid"].duplicated().sum()
)

print(
    "Duplicate UID rows:",
    duplicate_uid_count
)

duplicate_uid_table = (
    dataset[
        dataset["uid"].duplicated(
            keep=False
        )
    ]
    .sort_values("uid")
)

if len(duplicate_uid_table) > 0:

    duplicate_uid_table.to_csv(
        REPORT_DIR
        / "l7_duplicate_uids.csv",
        index=False
    )

feature_uid_set = {
    path.stem
    for path in feature_files
}

raw_uid_set = {
    path.stem
    for path in raw_files
}

dataset_uid_set = set(
    dataset["uid"]
)

missing_features = sorted(
    dataset_uid_set
    -
    feature_uid_set
)

extra_features = sorted(
    feature_uid_set
    -
    dataset_uid_set
)

missing_raw = sorted(
    dataset_uid_set
    -
    raw_uid_set
)

extra_raw = sorted(
    raw_uid_set
    -
    dataset_uid_set
)

print(
    "Dataset UIDs missing feature file:",
    len(missing_features)
)

print(
    "Extra feature UIDs:",
    len(extra_features)
)

print(
    "Dataset UIDs missing raw file:",
    len(missing_raw)
)

print(
    "Extra raw UIDs:",
    len(extra_raw)
)


# ==========================================================================================
# 10. FEATURE INDEX DECODER
# ==========================================================================================

def decode_feature_index(feature_index):
    """
    Feature vector:
        0..125   = normalized XYZ
        126..251 = motion XYZ
        252      = Track0 presence
        253      = Track1 presence
    """

    feature_index = int(
        feature_index
    )

    if 0 <= feature_index < 126:

        local = feature_index

        track = (
            local
            //
            (
                21 * 3
            )
        )

        remainder = (
            local
            %
            (
                21 * 3
            )
        )

        landmark = (
            remainder
            //
            3
        )

        coord = (
            remainder
            %
            3
        )

        return {
            "feature_group":
                "normalized",

            "track":
                int(track),

            "landmark_index":
                int(landmark),

            "landmark_name":
                LANDMARK_NAMES[
                    landmark
                ],

            "coordinate":
                COORD_NAMES[
                    coord
                ]
        }

    elif 126 <= feature_index < 252:

        local = (
            feature_index
            -
            126
        )

        track = (
            local
            //
            (
                21 * 3
            )
        )

        remainder = (
            local
            %
            (
                21 * 3
            )
        )

        landmark = (
            remainder
            //
            3
        )

        coord = (
            remainder
            %
            3
        )

        return {
            "feature_group":
                "motion",

            "track":
                int(track),

            "landmark_index":
                int(landmark),

            "landmark_name":
                LANDMARK_NAMES[
                    landmark
                ],

            "coordinate":
                COORD_NAMES[
                    coord
                ]
        }

    elif feature_index == 252:

        return {
            "feature_group":
                "presence",

            "track":
                0,

            "landmark_index":
                -1,

            "landmark_name":
                "N/A",

            "coordinate":
                "mask"
        }

    elif feature_index == 253:

        return {
            "feature_group":
                "presence",

            "track":
                1,

            "landmark_index":
                -1,

            "landmark_name":
                "N/A",

            "coordinate":
                "mask"
        }

    else:

        raise ValueError(
            f"Invalid feature index: "
            f"{feature_index}"
        )


# ==========================================================================================
# 11. SAFE SCALAR
# ==========================================================================================

def safe_scalar(
    data,
    key,
    default=np.nan
):

    if key not in data.files:
        return default

    try:
        return data[key].item()

    except Exception:
        return default


# ==========================================================================================
# 12. MAIN VIDEO-LEVEL AUDIT
# ==========================================================================================

print("\n" + "=" * 120)
print("AUDITING ALL FEATURE SEQUENCES")
print("=" * 120)

video_rows = []

global_extremes = []

global_normalized_values = []
global_motion_values = []

invalid_files = []

for index, feature_path in enumerate(
    feature_files,
    start=1
):

    uid = feature_path.stem

    try:

        with np.load(
            feature_path,
            allow_pickle=True
        ) as data:

            file_uid = str(
                safe_scalar(
                    data,
                    "uid",
                    uid
                )
            )

            gloss = str(
                safe_scalar(
                    data,
                    "gloss",
                    "UNKNOWN"
                )
            )

            features = np.asarray(
                data["features"],
                dtype=np.float32
            )

            normalized = np.asarray(
                data["normalized_landmarks"],
                dtype=np.float32
            )

            motion = np.asarray(
                data["motion"],
                dtype=np.float32
            )

            state = np.asarray(
                data["hand_state"],
                dtype=np.uint8
            )

            sample_indices = np.asarray(
                data["sample_indices"],
                dtype=np.int32
            )

            original_detection_rate = float(
                safe_scalar(
                    data,
                    "original_detection_rate",
                    np.nan
                )
            )

            repaired_detection_rate = float(
                safe_scalar(
                    data,
                    "repaired_detection_rate",
                    np.nan
                )
            )

            sampled_detection_rate = float(
                safe_scalar(
                    data,
                    "sampled_detection_rate",
                    np.nan
                )
            )

            scale_repairs = int(
                safe_scalar(
                    data,
                    "scale_repairs",
                    0
                )
            )

            l6_trainable = bool(
                safe_scalar(
                    data,
                    "trainable",
                    False
                )
            )

            raw_scales = (
                np.asarray(
                    data["raw_scales"],
                    dtype=np.float32
                )
                if
                "raw_scales"
                in
                data.files
                else
                np.zeros(
                    (32, 2),
                    dtype=np.float32
                )
            )

            robust_scales = (
                np.asarray(
                    data["robust_scales"],
                    dtype=np.float32
                )
                if
                "robust_scales"
                in
                data.files
                else
                np.zeros(
                    (32, 2),
                    dtype=np.float32
                )
            )

        # ----------------------------------------------------------------------------------
        # Structural checks
        # ----------------------------------------------------------------------------------

        feature_shape_ok = (
            features.shape
            ==
            (
                32,
                254
            )
        )

        normalized_shape_ok = (
            normalized.shape
            ==
            (
                32,
                2,
                21,
                3
            )
        )

        motion_shape_ok = (
            motion.shape
            ==
            (
                32,
                2,
                21,
                3
            )
        )

        state_shape_ok = (
            state.shape
            ==
            (
                32,
                2
            )
        )

        finite_features = bool(
            np.isfinite(
                features
            ).all()
        )

        finite_normalized = bool(
            np.isfinite(
                normalized
            ).all()
        )

        finite_motion = bool(
            np.isfinite(
                motion
            ).all()
        )

        structural_pass = bool(
            feature_shape_ok
            and
            normalized_shape_ok
            and
            motion_shape_ok
            and
            state_shape_ok
            and
            finite_features
            and
            finite_normalized
            and
            finite_motion
        )

        if not structural_pass:

            invalid_files.append(
                str(feature_path)
            )

        # ----------------------------------------------------------------------------------
        # Missingness / track coverage
        # ----------------------------------------------------------------------------------

        track0_present = (
            state[:, 0]
            >
            0
        )

        track1_present = (
            state[:, 1]
            >
            0
        )

        any_hand = (
            track0_present
            |
            track1_present
        )

        both_hands = (
            track0_present
            &
            track1_present
        )

        zero_hands = (
            ~any_hand
        )

        track0_frames = int(
            track0_present.sum()
        )

        track1_frames = int(
            track1_present.sum()
        )

        any_hand_frames = int(
            any_hand.sum()
        )

        both_hand_frames = int(
            both_hands.sum()
        )

        zero_hand_frames = int(
            zero_hands.sum()
        )

        original_state_count = int(
            np.sum(
                state == 1
            )
        )

        interpolated_state_count = int(
            np.sum(
                state == 2
            )
        )

        missing_state_count = int(
            np.sum(
                state == 0
            )
        )

        # ----------------------------------------------------------------------------------
        # Magnitudes
        # ----------------------------------------------------------------------------------

        abs_features = np.abs(
            features
        )

        abs_normalized = np.abs(
            normalized
        )

        abs_motion = np.abs(
            motion
        )

        max_abs_feature = float(
            np.max(
                abs_features
            )
        )

        max_abs_normalized = float(
            np.max(
                abs_normalized
            )
        )

        max_abs_motion = float(
            np.max(
                abs_motion
            )
        )

        # ----------------------------------------------------------------------------------
        # Exact maximum feature location
        # ----------------------------------------------------------------------------------

        flat_max_index = int(
            np.argmax(
                abs_features
            )
        )

        sequence_index, feature_index = np.unravel_index(
            flat_max_index,
            features.shape
        )

        decoded = decode_feature_index(
            feature_index
        )

        source_frame = (
            int(
                sample_indices[
                    sequence_index
                ]
            )
            if
            len(sample_indices)
            >
            sequence_index
            else
            -1
        )

        max_value = float(
            features[
                sequence_index,
                feature_index
            ]
        )

        # ----------------------------------------------------------------------------------
        # Magnitude threshold counts
        # ----------------------------------------------------------------------------------

        magnitude_counts = {}

        for threshold in MAGNITUDE_LEVELS:

            magnitude_counts[
                f"feature_values_abs_gt_{str(threshold).replace('.', '_')}"
            ] = int(
                np.sum(
                    abs_features
                    >
                    threshold
                )
            )

        # ----------------------------------------------------------------------------------
        # Scale statistics
        # ----------------------------------------------------------------------------------

        valid_raw_scales = raw_scales[
            raw_scales
            >
            0
        ]

        valid_robust_scales = robust_scales[
            robust_scales
            >
            0
        ]

        min_raw_scale = (
            float(
                np.min(
                    valid_raw_scales
                )
            )
            if
            len(valid_raw_scales)
            >
            0
            else
            np.nan
        )

        min_robust_scale = (
            float(
                np.min(
                    valid_robust_scales
                )
            )
            if
            len(valid_robust_scales)
            >
            0
            else
            np.nan
        )

        median_robust_scale = (
            float(
                np.median(
                    valid_robust_scales
                )
            )
            if
            len(valid_robust_scales)
            >
            0
            else
            np.nan
        )

        # ----------------------------------------------------------------------------------
        # L7 provisional usability
        # ----------------------------------------------------------------------------------

        provisional_usable = bool(
            structural_pass
            and
            original_detection_rate
            >=
            PROVISIONAL_MIN_DETECTION
            and
            any_hand_frames
            >=
            PROVISIONAL_MIN_SAMPLED_FRAMES
        )

        # ----------------------------------------------------------------------------------
        # Video row
        # ----------------------------------------------------------------------------------

        row = {
            "uid":
                file_uid,

            "gloss":
                gloss,

            "feature_shape_ok":
                feature_shape_ok,

            "normalized_shape_ok":
                normalized_shape_ok,

            "motion_shape_ok":
                motion_shape_ok,

            "state_shape_ok":
                state_shape_ok,

            "finite_features":
                finite_features,

            "finite_normalized":
                finite_normalized,

            "finite_motion":
                finite_motion,

            "structural_pass":
                structural_pass,

            "original_detection_rate":
                original_detection_rate,

            "repaired_detection_rate":
                repaired_detection_rate,

            "sampled_detection_rate":
                sampled_detection_rate,

            "track0_frames":
                track0_frames,

            "track1_frames":
                track1_frames,

            "any_hand_frames":
                any_hand_frames,

            "both_hand_frames":
                both_hand_frames,

            "zero_hand_frames":
                zero_hand_frames,

            "track0_rate":
                track0_frames
                /
                32.0,

            "track1_rate":
                track1_frames
                /
                32.0,

            "both_hand_rate":
                both_hand_frames
                /
                32.0,

            "zero_hand_rate":
                zero_hand_frames
                /
                32.0,

            "original_state_count":
                original_state_count,

            "interpolated_state_count":
                interpolated_state_count,

            "missing_state_count":
                missing_state_count,

            "scale_repairs":
                scale_repairs,

            "min_raw_scale":
                min_raw_scale,

            "min_robust_scale":
                min_robust_scale,

            "median_robust_scale":
                median_robust_scale,

            "max_abs_normalized":
                max_abs_normalized,

            "max_abs_motion":
                max_abs_motion,

            "max_abs_feature":
                max_abs_feature,

            "max_feature_value":
                max_value,

            "max_sequence_index":
                int(
                    sequence_index
                ),

            "max_source_frame":
                source_frame,

            "max_feature_index":
                int(
                    feature_index
                ),

            "max_feature_group":
                decoded[
                    "feature_group"
                ],

            "max_track":
                decoded[
                    "track"
                ],

            "max_landmark_index":
                decoded[
                    "landmark_index"
                ],

            "max_landmark_name":
                decoded[
                    "landmark_name"
                ],

            "max_coordinate":
                decoded[
                    "coordinate"
                ],

            "l6_trainable":
                l6_trainable,

            "l7_provisional_usable":
                provisional_usable
        }

        row.update(
            magnitude_counts
        )

        video_rows.append(
            row
        )

        # ----------------------------------------------------------------------------------
        # Keep values for global distribution.
        #
        # 7050 × 32 × 2 × 21 × 3 is manageable, but storing all values
        # unnecessarily duplicates memory. Instead sample only active values.
        # ----------------------------------------------------------------------------------

        active_norm_mask = np.repeat(
            (
                state > 0
            )[
                :,
                :,
                None,
                None
            ],
            21,
            axis=2
        )

        active_norm_mask = np.repeat(
            active_norm_mask,
            3,
            axis=3
        )

        active_normalized = abs_normalized[
            active_norm_mask
        ]

        if active_normalized.size > 0:

            # Deterministic thinning only for percentile estimation
            # if sequence contributes many values.

            global_normalized_values.append(
                active_normalized.astype(
                    np.float32
                )
            )

        active_motion_mask = np.repeat(
            (
                state > 0
            )[
                :,
                :,
                None,
                None
            ],
            21,
            axis=2
        )

        active_motion_mask = np.repeat(
            active_motion_mask,
            3,
            axis=3
        )

        active_motion = abs_motion[
            active_motion_mask
        ]

        if active_motion.size > 0:

            global_motion_values.append(
                active_motion.astype(
                    np.float32
                )
            )

        # ----------------------------------------------------------------------------------
        # Candidate extreme individual values
        # ----------------------------------------------------------------------------------

        candidate_count = min(
            20,
            features.size
        )

        flat_abs = abs_features.ravel()

        if candidate_count > 0:

            candidate_indices = np.argpartition(
                flat_abs,
                -candidate_count
            )[
                -candidate_count:
            ]

            for flat_index in candidate_indices:

                seq_idx, feat_idx = np.unravel_index(
                    int(flat_index),
                    features.shape
                )

                info = decode_feature_index(
                    feat_idx
                )

                source = (
                    int(
                        sample_indices[
                            seq_idx
                        ]
                    )
                    if
                    seq_idx
                    <
                    len(sample_indices)
                    else
                    -1
                )

                global_extremes.append(
                    {
                        "uid":
                            file_uid,

                        "gloss":
                            gloss,

                        "abs_value":
                            float(
                                abs_features[
                                    seq_idx,
                                    feat_idx
                                ]
                            ),

                        "value":
                            float(
                                features[
                                    seq_idx,
                                    feat_idx
                                ]
                            ),

                        "sequence_index":
                            int(
                                seq_idx
                            ),

                        "source_frame":
                            source,

                        "feature_index":
                            int(
                                feat_idx
                            ),

                        "feature_group":
                            info[
                                "feature_group"
                            ],

                        "track":
                            info[
                                "track"
                            ],

                        "landmark_index":
                            info[
                                "landmark_index"
                            ],

                        "landmark_name":
                            info[
                                "landmark_name"
                            ],

                        "coordinate":
                            info[
                                "coordinate"
                            ],

                        "state":
                            (
                                int(
                                    state[
                                        seq_idx,
                                        info[
                                            "track"
                                        ]
                                    ]
                                )
                                if
                                info[
                                    "track"
                                ]
                                in
                                [
                                    0,
                                    1
                                ]
                                else
                                -1
                            ),

                        "raw_scale":
                            (
                                float(
                                    raw_scales[
                                        seq_idx,
                                        info[
                                            "track"
                                        ]
                                    ]
                                )
                                if
                                info[
                                    "track"
                                ]
                                in
                                [
                                    0,
                                    1
                                ]
                                else
                                np.nan
                            ),

                        "robust_scale":
                            (
                                float(
                                    robust_scales[
                                        seq_idx,
                                        info[
                                            "track"
                                        ]
                                    ]
                                )
                                if
                                info[
                                    "track"
                                ]
                                in
                                [
                                    0,
                                    1
                                ]
                                else
                                np.nan
                            )
                    }
                )

    except Exception as error:

        invalid_files.append(
            f"{feature_path}: {repr(error)}"
        )

        print(
            "\n[ERROR]",
            feature_path.name,
            repr(error)
        )

    if (
        index % 500 == 0
        or
        index == len(feature_files)
    ):

        print(
            f"Audited "
            f"{index:4d}/{len(feature_files)}"
        )


# ==========================================================================================
# 13. VIDEO AUDIT DATAFRAME
# ==========================================================================================

video_df = pd.DataFrame(
    video_rows
)

video_df.to_csv(
    VIDEO_AUDIT_CSV,
    index=False
)

print()
print("[SAVED]", VIDEO_AUDIT_CSV)


# ==========================================================================================
# 14. GLOBAL OUTLIER TABLE
# ==========================================================================================

extreme_df = pd.DataFrame(
    global_extremes
)

if len(extreme_df) > 0:

    extreme_df = (
        extreme_df
        .sort_values(
            "abs_value",
            ascending=False
        )
        .drop_duplicates(
            subset=[
                "uid",
                "sequence_index",
                "feature_index"
            ]
        )
        .head(
            TOP_N_OUTLIERS
        )
        .reset_index(
            drop=True
        )
    )

extreme_df.to_csv(
    OUTLIER_CSV,
    index=False
)

print("[SAVED]", OUTLIER_CSV)


# ==========================================================================================
# 15. EXTREME SEQUENCE TABLE
# ==========================================================================================

extreme_sequence_df = (
    video_df
    .sort_values(
        "max_abs_feature",
        ascending=False
    )
    .head(100)
    .copy()
)

extreme_sequence_df.to_csv(
    EXTREME_SEQUENCE_CSV,
    index=False
)

print("[SAVED]", EXTREME_SEQUENCE_CSV)


# ==========================================================================================
# 16. GLOBAL VALUE DISTRIBUTIONS
# ==========================================================================================

print("\n" + "=" * 120)
print("GLOBAL FEATURE DISTRIBUTION")
print("=" * 120)

if len(global_normalized_values) > 0:

    all_norm = np.concatenate(
        global_normalized_values
    )

else:

    all_norm = np.asarray(
        [],
        dtype=np.float32
    )

if len(global_motion_values) > 0:

    all_motion = np.concatenate(
        global_motion_values
    )

else:

    all_motion = np.asarray(
        [],
        dtype=np.float32
    )


def percentile_report(
    values,
    name
):

    print()
    print(name)
    print("-" * 120)

    if len(values) == 0:

        print("No values.")

        return {}

    percentiles = [
        50,
        90,
        95,
        99,
        99.5,
        99.9,
        99.99,
        100
    ]

    result = {}

    for p in percentiles:

        value = float(
            np.percentile(
                values,
                p
            )
        )

        result[
            str(p)
        ] = value

        print(
            f"P{p:<6} : "
            f"{value:.6f}"
        )

    return result


normalized_percentiles = percentile_report(
    all_norm,
    "ABSOLUTE NORMALIZED XYZ"
)

motion_percentiles = percentile_report(
    all_motion,
    "ABSOLUTE MOTION XYZ"
)


# Release large temporary arrays after percentile calculation.

del global_normalized_values
del global_motion_values

if "all_norm" in globals():
    del all_norm

if "all_motion" in globals():
    del all_motion


# ==========================================================================================
# 17. STRUCTURAL SUMMARY
# ==========================================================================================

print("\n" + "=" * 120)
print("STRUCTURAL / NUMERICAL AUDIT")
print("=" * 120)

total_audited = len(
    video_df
)

structural_pass_count = int(
    video_df[
        "structural_pass"
    ].sum()
)

structural_fail_count = (
    total_audited
    -
    structural_pass_count
)

print(
    "Sequences audited       :",
    total_audited
)

print(
    "Structural PASS         :",
    structural_pass_count
)

print(
    "Structural FAIL         :",
    structural_fail_count
)

print(
    "Invalid/read failures   :",
    len(
        invalid_files
    )
)

if len(video_df) > 0:

    print(
        "Global max |feature|   :",
        f"{video_df['max_abs_feature'].max():.6f}"
    )

    print(
        "Global max |normalized|:",
        f"{video_df['max_abs_normalized'].max():.6f}"
    )

    print(
        "Global max |motion|    :",
        f"{video_df['max_abs_motion'].max():.6f}"
    )


# ==========================================================================================
# 18. COVERAGE DISTRIBUTION
# ==========================================================================================

print("\n" + "=" * 120)
print("HAND COVERAGE DISTRIBUTION")
print("=" * 120)

coverage = video_df[
    "original_detection_rate"
].dropna()

coverage_percentiles = {}

for p in [
    0,
    1,
    5,
    10,
    25,
    50,
    75,
    90,
    95,
    99,
    100
]:

    value = float(
        np.percentile(
            coverage,
            p
        )
    )

    coverage_percentiles[
        str(p)
    ] = value

    print(
        f"P{p:<3}: "
        f"{value * 100:7.2f}%"
    )


# ==========================================================================================
# 19. COVERAGE BANDS
# ==========================================================================================

coverage_bands = [
    (
        "0%",
        0.0,
        0.0
    ),

    (
        ">0–20%",
        0.0,
        0.20
    ),

    (
        "20–40%",
        0.20,
        0.40
    ),

    (
        "40–50%",
        0.40,
        0.50
    ),

    (
        "50–60%",
        0.50,
        0.60
    ),

    (
        "60–70%",
        0.60,
        0.70
    ),

    (
        "70–80%",
        0.70,
        0.80
    ),

    (
        "80–90%",
        0.80,
        0.90
    ),

    (
        "90–100%",
        0.90,
        1.000001
    )
]

coverage_band_rows = []

for name, low, high in coverage_bands:

    if name == "0%":

        count = int(
            (
                video_df[
                    "original_detection_rate"
                ]
                ==
                0
            ).sum()
        )

    else:

        count = int(
            (
                (
                    video_df[
                        "original_detection_rate"
                    ]
                    >
                    low
                )
                &
                (
                    video_df[
                        "original_detection_rate"
                    ]
                    <=
                    high
                )
            ).sum()
        )

    coverage_band_rows.append(
        {
            "band":
                name,

            "videos":
                count,

            "percentage":
                (
                    100.0
                    *
                    count
                    /
                    max(
                        len(video_df),
                        1
                    )
                )
        }
    )

coverage_band_df = pd.DataFrame(
    coverage_band_rows
)

print()
print(
    coverage_band_df.to_string(
        index=False
    )
)

coverage_band_df.to_csv(
    REPORT_DIR
    / "l7_coverage_bands.csv",
    index=False
)


# ==========================================================================================
# 20. TRACK COVERAGE
# ==========================================================================================

print("\n" + "=" * 120)
print("TRACK COVERAGE")
print("=" * 120)

print(
    "Mean Track0 frames :",
    f"{video_df['track0_frames'].mean():.2f} / 32"
)

print(
    "Mean Track1 frames :",
    f"{video_df['track1_frames'].mean():.2f} / 32"
)

print(
    "Mean any-hand      :",
    f"{video_df['any_hand_frames'].mean():.2f} / 32"
)

print(
    "Mean both-hands    :",
    f"{video_df['both_hand_frames'].mean():.2f} / 32"
)

print(
    "Mean zero-hand     :",
    f"{video_df['zero_hand_frames'].mean():.2f} / 32"
)

print(
    "Videos no Track1   :",
    int(
        (
            video_df[
                "track1_frames"
            ]
            ==
            0
        ).sum()
    )
)

print(
    "Videos no hands in sampled representation:",
    int(
        (
            video_df[
                "any_hand_frames"
            ]
            ==
            0
        ).sum()
    )
)


# ==========================================================================================
# 21. INTERPOLATION SUMMARY
# ==========================================================================================

print("\n" + "=" * 120)
print("INTERPOLATION / MISSINGNESS")
print("=" * 120)

total_original_states = int(
    video_df[
        "original_state_count"
    ].sum()
)

total_interpolated_states = int(
    video_df[
        "interpolated_state_count"
    ].sum()
)

total_missing_states = int(
    video_df[
        "missing_state_count"
    ].sum()
)

print(
    "Original hand states     :",
    total_original_states
)

print(
    "Interpolated hand states :",
    total_interpolated_states
)

print(
    "Missing hand states      :",
    total_missing_states
)

total_states = (
    total_original_states
    +
    total_interpolated_states
    +
    total_missing_states
)

if total_states > 0:

    print(
        "Original %               :",
        f"{100 * total_original_states / total_states:.2f}%"
    )

    print(
        "Interpolated %           :",
        f"{100 * total_interpolated_states / total_states:.2f}%"
    )

    print(
        "Missing %                :",
        f"{100 * total_missing_states / total_states:.2f}%"
    )


# ==========================================================================================
# 22. MAGNITUDE SUMMARY
# ==========================================================================================

print("\n" + "=" * 120)
print("EXTREME FEATURE MAGNITUDES")
print("=" * 120)

for threshold in MAGNITUDE_LEVELS:

    column = (
        "feature_values_abs_gt_"
        +
        str(
            threshold
        ).replace(
            ".",
            "_"
        )
    )

    sequences = int(
        (
            video_df[
                column
            ]
            >
            0
        ).sum()
    )

    values = int(
        video_df[
            column
        ].sum()
    )

    print(
        f"|feature| > {threshold:5.1f}: "
        f"{sequences:4d} sequences, "
        f"{values:7d} values"
    )


# ==========================================================================================
# 23. TOP 20 EXTREME SEQUENCES
# ==========================================================================================

print("\n" + "=" * 120)
print("TOP 20 EXTREME SEQUENCES")
print("=" * 120)

top20 = (
    video_df
    .sort_values(
        "max_abs_feature",
        ascending=False
    )
    .head(20)
)

columns = [
    "uid",
    "gloss",
    "max_abs_feature",
    "max_feature_group",
    "max_track",
    "max_landmark_name",
    "max_coordinate",
    "max_sequence_index",
    "max_source_frame",
    "min_raw_scale",
    "min_robust_scale",
    "scale_repairs",
    "original_detection_rate"
]

print(
    top20[
        columns
    ].to_string(
        index=False
    )
)


# ==========================================================================================
# 24. EXACT GLOBAL MAXIMUM
# ==========================================================================================

print("\n" + "=" * 120)
print("GLOBAL MAXIMUM FEATURE — EXACT LOCATION")
print("=" * 120)

global_max_row = (
    video_df
    .sort_values(
        "max_abs_feature",
        ascending=False
    )
    .iloc[0]
)

for key in [
    "uid",
    "gloss",
    "max_abs_feature",
    "max_feature_value",
    "max_feature_group",
    "max_track",
    "max_landmark_index",
    "max_landmark_name",
    "max_coordinate",
    "max_sequence_index",
    "max_source_frame",
    "min_raw_scale",
    "min_robust_scale",
    "scale_repairs",
    "original_detection_rate",
    "sampled_detection_rate"
]:

    print(
        f"{key:28s}:",
        global_max_row[
            key
        ]
    )


# ==========================================================================================
# 25. RAW CONTEXT FOR GLOBAL MAXIMUM
# ==========================================================================================

print("\n" + "=" * 120)
print("GLOBAL MAXIMUM — RAW FRAME CONTEXT")
print("=" * 120)

global_uid = str(
    global_max_row[
        "uid"
    ]
)

global_feature_path = (
    FEATURE_DIR
    /
    f"{global_uid}.npz"
)

global_raw_path = (
    RAW_DIR
    /
    f"{global_uid}.npz"
)

if (
    global_feature_path.exists()
    and
    global_raw_path.exists()
):

    with np.load(
        global_feature_path,
        allow_pickle=True
    ) as data:

        g_features = np.asarray(
            data["features"],
            dtype=np.float32
        )

        g_norm = np.asarray(
            data["normalized_landmarks"],
            dtype=np.float32
        )

        g_motion = np.asarray(
            data["motion"],
            dtype=np.float32
        )

        g_state = np.asarray(
            data["hand_state"],
            dtype=np.uint8
        )

        g_indices = np.asarray(
            data["sample_indices"],
            dtype=np.int32
        )

        g_raw_scales = np.asarray(
            data["raw_scales"],
            dtype=np.float32
        )

        g_robust_scales = np.asarray(
            data["robust_scales"],
            dtype=np.float32
        )

    with np.load(
        global_raw_path,
        allow_pickle=True
    ) as raw:

        raw_landmarks = np.asarray(
            raw["landmarks_px"],
            dtype=np.float32
        )

        raw_mask = np.asarray(
            raw["hand_mask"],
            dtype=np.uint8
        )

        raw_scores = np.asarray(
            raw["palm_scores"],
            dtype=np.float32
        )

        raw_frame_numbers = np.asarray(
            raw["frame_numbers"],
            dtype=np.int32
        )

    seq_idx = int(
        global_max_row[
            "max_sequence_index"
        ]
    )

    source_frame = int(
        global_max_row[
            "max_source_frame"
        ]
    )

    track = int(
        global_max_row[
            "max_track"
        ]
    )

    landmark_index = int(
        global_max_row[
            "max_landmark_index"
        ]
    )

    print(
        "UID                 :",
        global_uid
    )

    print(
        "Sequence index      :",
        seq_idx
    )

    print(
        "Source frame        :",
        source_frame
    )

    print(
        "Track               :",
        track
    )

    print(
        "Landmark            :",
        global_max_row[
            "max_landmark_name"
        ]
    )

    print(
        "Coordinate          :",
        global_max_row[
            "max_coordinate"
        ]
    )

    print()
    print(
        "State at sampled frame:",
        g_state[
            seq_idx
        ]
    )

    print(
        "Raw scale sampled     :",
        g_raw_scales[
            seq_idx
        ]
    )

    print(
        "Robust scale sampled  :",
        g_robust_scales[
            seq_idx
        ]
    )

    print()
    print(
        "Normalized landmark:"
    )

    print(
        g_norm[
            seq_idx,
            track,
            landmark_index
        ]
    )

    print()
    print(
        "Motion landmark:"
    )

    print(
        g_motion[
            seq_idx,
            track,
            landmark_index
        ]
    )

    print()
    print(
        "Raw extraction around source frame:"
    )

    start_frame = max(
        0,
        source_frame - 2
    )

    end_frame = min(
        len(raw_landmarks),
        source_frame + 3
    )

    for frame in range(
        start_frame,
        end_frame
    ):

        print()
        print(
            f"Frame {frame}"
        )

        print(
            " mask   :",
            raw_mask[
                frame
            ]
        )

        print(
            " scores :",
            raw_scores[
                frame
            ]
        )

        if track < 2:

            print(
                " wrist  :",
                raw_landmarks[
                    frame,
                    track,
                    0
                ]
            )

            print(
                f" {LANDMARK_NAMES[landmark_index]:7s}:",
                raw_landmarks[
                    frame,
                    track,
                    landmark_index
                ]
            )

else:

    print(
        "Raw context unavailable."
    )


# ==========================================================================================
# 26. PROVISIONAL REJECTION TABLE
# ==========================================================================================

rejected_df = (
    video_df[
        ~video_df[
            "l7_provisional_usable"
        ]
    ]
    .copy()
    .sort_values(
        "original_detection_rate"
    )
)

rejected_df.to_csv(
    REJECTED_CSV,
    index=False
)

print("\n" + "=" * 120)
print("PROVISIONAL VIDEO USABILITY")
print("=" * 120)

usable_count = int(
    video_df[
        "l7_provisional_usable"
    ].sum()
)

rejected_count = (
    len(video_df)
    -
    usable_count
)

print(
    "Provisional usable  :",
    usable_count
)

print(
    "Provisional rejected:",
    rejected_count
)

print(
    "Criterion           : structural PASS + "
    f"detection >= {PROVISIONAL_MIN_DETECTION:.0%} + "
    f">= {PROVISIONAL_MIN_SAMPLED_FRAMES} sampled frames with a hand"
)


# ==========================================================================================
# 27. CLASS SUPPORT ANALYSIS
# ==========================================================================================

print("\n" + "=" * 120)
print("CLASS / GLOSS SUPPORT ANALYSIS")
print("=" * 120)

class_rows = []

for gloss, group in video_df.groupby(
    "gloss"
):

    total = len(group)

    usable = int(
        group[
            "l7_provisional_usable"
        ].sum()
    )

    good = int(
        (
            group[
                "original_detection_rate"
            ]
            >=
            0.80
        ).sum()
    )

    review = int(
        (
            (
                group[
                    "original_detection_rate"
                ]
                >=
                0.50
            )
            &
            (
                group[
                    "original_detection_rate"
                ]
                <
                0.80
            )
        ).sum()
    )

    poor = int(
        (
            group[
                "original_detection_rate"
            ]
            <
            0.50
        ).sum()
    )

    class_rows.append(
        {
            "gloss":
                gloss,

            "total_videos":
                total,

            "usable_videos":
                usable,

            "rejected_videos":
                total
                -
                usable,

            "good_videos":
                good,

            "review_videos":
                review,

            "poor_videos":
                poor,

            "mean_detection_rate":
                float(
                    group[
                        "original_detection_rate"
                    ].mean()
                ),

            "median_detection_rate":
                float(
                    group[
                        "original_detection_rate"
                    ].median()
                ),

            "mean_sampled_detection_rate":
                float(
                    group[
                        "sampled_detection_rate"
                    ].mean()
                ),

            "max_abs_feature":
                float(
                    group[
                        "max_abs_feature"
                    ].max()
                ),

            "total_scale_repairs":
                int(
                    group[
                        "scale_repairs"
                    ].sum()
                )
        }
    )

class_df = pd.DataFrame(
    class_rows
)

class_df = class_df.sort_values(
    [
        "usable_videos",
        "total_videos",
        "gloss"
    ],
    ascending=[
        False,
        False,
        True
    ]
).reset_index(
    drop=True
)

class_df.to_csv(
    CLASS_SUPPORT_CSV,
    index=False
)

print(
    "Classes audited:",
    len(class_df)
)


# ==========================================================================================
# 28. CLASS SUPPORT THRESHOLDS
# ==========================================================================================

support_rows = []

for threshold in [
    1,
    2,
    3,
    4,
    5,
    6,
    8,
    10
]:

    total_classes = int(
        (
            class_df[
                "total_videos"
            ]
            >=
            threshold
        ).sum()
    )

    usable_classes = int(
        (
            class_df[
                "usable_videos"
            ]
            >=
            threshold
        ).sum()
    )

    total_videos_in_usable_classes = int(
        class_df.loc[
            class_df[
                "usable_videos"
            ]
            >=
            threshold,
            "usable_videos"
        ].sum()
    )

    support_rows.append(
        {
            "minimum_videos_per_gloss":
                threshold,

            "classes_before_quality_filter":
                total_classes,

            "classes_after_quality_filter":
                usable_classes,

            "usable_videos_in_retained_classes":
                total_videos_in_usable_classes
        }
    )

support_df = pd.DataFrame(
    support_rows
)

support_df.to_csv(
    SUPPORT_SUMMARY_CSV,
    index=False
)

print()
print(
    support_df.to_string(
        index=False
    )
)


# ==========================================================================================
# 29. SPLIT FEASIBILITY
# ==========================================================================================

print("\n" + "=" * 120)
print("CLASS-LEVEL SPLIT FEASIBILITY")
print("=" * 120)

# Descriptive feasibility only.
#
# >=3 usable:
# theoretically permits 1 train + 1 val + 1 test.
#
# >=4 usable:
# theoretically permits 2 train + 1 val + 1 test.
#
# >=5 usable:
# theoretically permits stronger train support.
#
# No split is actually created in L7.

split_requirements = [
    (
        ">=2 usable",
        2
    ),
    (
        ">=3 usable",
        3
    ),
    (
        ">=4 usable",
        4
    ),
    (
        ">=5 usable",
        5
    ),
    (
        ">=6 usable",
        6
    ),
    (
        ">=8 usable",
        8
    ),
    (
        ">=10 usable",
        10
    )
]

for label, minimum in split_requirements:

    subset = class_df[
        class_df[
            "usable_videos"
        ]
        >=
        minimum
    ]

    print(
        f"{label:12s}: "
        f"{len(subset):4d} classes | "
        f"{int(subset['usable_videos'].sum()):5d} usable videos"
    )


# ==========================================================================================
# 30. QUALITY COMPOSITION OF CLASSES
# ==========================================================================================

quality_class_rows = []

for minimum_support in [
    1,
    2,
    3,
    4,
    5
]:

    subset = class_df[
        class_df[
            "usable_videos"
        ]
        >=
        minimum_support
    ]

    quality_class_rows.append(
        {
            "minimum_usable_support":
                minimum_support,

            "classes":
                len(subset),

            "usable_videos":
                int(
                    subset[
                        "usable_videos"
                    ].sum()
                ),

            "good_videos":
                int(
                    subset[
                        "good_videos"
                    ].sum()
                ),

            "review_videos":
                int(
                    subset[
                        "review_videos"
                    ].sum()
                ),

            "poor_videos":
                int(
                    subset[
                        "poor_videos"
                    ].sum()
                )
        }
    )

quality_class_df = pd.DataFrame(
    quality_class_rows
)

quality_class_df.to_csv(
    QUALITY_CLASS_CSV,
    index=False
)


# ==========================================================================================
# 31. ORIGINAL DATASET SUPPORT COMPARISON
# ==========================================================================================

print("\n" + "=" * 120)
print("ORIGINAL DATASET SUPPORT VS LANDMARK-USABLE SUPPORT")
print("=" * 120)

original_support = (
    dataset
    .groupby(
        "gloss"
    )
    .size()
)

for threshold in [
    1,
    2,
    3,
    4,
    5,
    6,
    8,
    10
]:

    original_classes = int(
        (
            original_support
            >=
            threshold
        ).sum()
    )

    usable_classes = int(
        (
            class_df[
                "usable_videos"
            ]
            >=
            threshold
        ).sum()
    )

    difference = (
        original_classes
        -
        usable_classes
    )

    print(
        f">= {threshold:2d} videos/gloss : "
        f"original={original_classes:4d} | "
        f"usable={usable_classes:4d} | "
        f"lost={difference:4d}"
    )


# ==========================================================================================
# 32. SINGLETON / LOW-SUPPORT ANALYSIS
# ==========================================================================================

print("\n" + "=" * 120)
print("LOW-SUPPORT CLASS ANALYSIS")
print("=" * 120)

singleton_total = int(
    (
        class_df[
            "total_videos"
        ]
        ==
        1
    ).sum()
)

singleton_usable = int(
    (
        (
            class_df[
                "total_videos"
            ]
            ==
            1
        )
        &
        (
            class_df[
                "usable_videos"
            ]
            ==
            1
        )
    ).sum()
)

zero_usable_classes = int(
    (
        class_df[
            "usable_videos"
        ]
        ==
        0
    ).sum()
)

print(
    "Singleton classes total  :",
    singleton_total
)

print(
    "Singleton still usable   :",
    singleton_usable
)

print(
    "Classes with 0 usable vid:",
    zero_usable_classes
)


# ==========================================================================================
# 33. TOP CLASSES
# ==========================================================================================

print("\n" + "=" * 120)
print("TOP 20 CLASSES BY USABLE SUPPORT")
print("=" * 120)

print(
    class_df[
        [
            "gloss",
            "total_videos",
            "usable_videos",
            "good_videos",
            "review_videos",
            "poor_videos",
            "mean_detection_rate"
        ]
    ]
    .head(20)
    .to_string(
        index=False
    )
)


# ==========================================================================================
# 34. WORST CLASSES BY MEAN DETECTION
# ==========================================================================================

print("\n" + "=" * 120)
print("20 LOWEST-DETECTION CLASSES")
print("=" * 120)

worst_classes = (
    class_df
    .sort_values(
        [
            "mean_detection_rate",
            "total_videos"
        ],
        ascending=[
            True,
            False
        ]
    )
    .head(20)
)

print(
    worst_classes[
        [
            "gloss",
            "total_videos",
            "usable_videos",
            "mean_detection_rate",
            "median_detection_rate",
            "max_abs_feature"
        ]
    ].to_string(
        index=False
    )
)


# ==========================================================================================
# 35. OUTLIER RELATIONSHIP TO SCALE
# ==========================================================================================

print("\n" + "=" * 120)
print("OUTLIER / SCALE RELATIONSHIP")
print("=" * 120)

extreme_10 = video_df[
    video_df[
        "max_abs_feature"
    ]
    >
    10.0
]

extreme_20 = video_df[
    video_df[
        "max_abs_feature"
    ]
    >
    20.0
]

print(
    "Sequences max |feature| > 10:",
    len(extreme_10)
)

print(
    "Sequences max |feature| > 20:",
    len(extreme_20)
)

if len(extreme_10) > 0:

    print(
        "Median minimum robust scale "
        "for >10 sequences:",
        f"{extreme_10['min_robust_scale'].median():.6f}"
    )

if len(video_df) > 0:

    print(
        "Median minimum robust scale "
        "overall:",
        f"{video_df['min_robust_scale'].median():.6f}"
    )


# ==========================================================================================
# 36. CHECK FEATURE RECONSTRUCTION
# ==========================================================================================

print("\n" + "=" * 120)
print("FEATURE RECONSTRUCTION CONSISTENCY")
print("=" * 120)

reconstruction_failures = []

# Check every sequence.
#
# Expected:
# features[:,0:126]   == normalized.reshape(32,126)
# features[:,126:252] == motion.reshape(32,126)
# features[:,252:254] == (state>0)

for index, feature_path in enumerate(
    feature_files,
    start=1
):

    try:

        with np.load(
            feature_path,
            allow_pickle=True
        ) as data:

            features = np.asarray(
                data["features"],
                dtype=np.float32
            )

            normalized = np.asarray(
                data["normalized_landmarks"],
                dtype=np.float32
            )

            motion = np.asarray(
                data["motion"],
                dtype=np.float32
            )

            state = np.asarray(
                data["hand_state"],
                dtype=np.uint8
            )

        reconstructed = np.concatenate(
            [
                normalized.reshape(
                    32,
                    126
                ),

                motion.reshape(
                    32,
                    126
                ),

                (
                    state > 0
                ).astype(
                    np.float32
                )
            ],
            axis=1
        )

        max_difference = float(
            np.max(
                np.abs(
                    features
                    -
                    reconstructed
                )
            )
        )

        if max_difference > 1e-6:

            reconstruction_failures.append(
                {
                    "uid":
                        feature_path.stem,

                    "max_difference":
                        max_difference
                }
            )

    except Exception as error:

        reconstruction_failures.append(
            {
                "uid":
                    feature_path.stem,

                "max_difference":
                    np.nan,

                "error":
                    repr(error)
            }
        )

print(
    "Reconstruction failures:",
    len(
        reconstruction_failures
    )
)

if len(reconstruction_failures) > 0:

    pd.DataFrame(
        reconstruction_failures
    ).to_csv(
        REPORT_DIR
        / "l7_reconstruction_failures.csv",
        index=False
    )


# ==========================================================================================
# 37. BUILD SUMMARY
# ==========================================================================================

global_max_feature = (
    float(
        video_df[
            "max_abs_feature"
        ].max()
    )
    if
    len(video_df)
    >
    0
    else
    np.nan
)

global_max_normalized = (
    float(
        video_df[
            "max_abs_normalized"
        ].max()
    )
    if
    len(video_df)
    >
    0
    else
    np.nan
)

global_max_motion = (
    float(
        video_df[
            "max_abs_motion"
        ].max()
    )
    if
    len(video_df)
    >
    0
    else
    np.nan
)

classes_ge_2 = int(
    (
        class_df[
            "usable_videos"
        ]
        >=
        2
    ).sum()
)

classes_ge_3 = int(
    (
        class_df[
            "usable_videos"
        ]
        >=
        3
    ).sum()
)

classes_ge_4 = int(
    (
        class_df[
            "usable_videos"
        ]
        >=
        4
    ).sum()
)

classes_ge_5 = int(
    (
        class_df[
            "usable_videos"
        ]
        >=
        5
    ).sum()
)

classes_ge_6 = int(
    (
        class_df[
            "usable_videos"
        ]
        >=
        6
    ).sum()
)

classes_ge_8 = int(
    (
        class_df[
            "usable_videos"
        ]
        >=
        8
    ).sum()
)

classes_ge_10 = int(
    (
        class_df[
            "usable_videos"
        ]
        >=
        10
    ).sum()
)

summary = {
    "stage":
        "L7_FULL_LANDMARK_DATASET_AUDIT",

    "l6_feature_files":
        len(feature_files),

    "l6_raw_files":
        len(raw_files),

    "dataset_rows":
        len(dataset),

    "dataset_unique_glosses":
        int(
            dataset[
                "gloss"
            ].nunique()
        ),

    "duplicate_uid_rows":
        duplicate_uid_count,

    "missing_feature_files":
        len(missing_features),

    "missing_raw_files":
        len(missing_raw),

    "structural_pass":
        structural_pass_count,

    "structural_fail":
        structural_fail_count,

    "reconstruction_failures":
        len(
            reconstruction_failures
        ),

    "provisional_usable_videos":
        usable_count,

    "provisional_rejected_videos":
        rejected_count,

    "global_max_abs_feature":
        global_max_feature,

    "global_max_abs_normalized":
        global_max_normalized,

    "global_max_abs_motion":
        global_max_motion,

    "global_max_uid":
        str(
            global_max_row[
                "uid"
            ]
        ),

    "global_max_gloss":
        str(
            global_max_row[
                "gloss"
            ]
        ),

    "global_max_feature_group":
        str(
            global_max_row[
                "max_feature_group"
            ]
        ),

    "global_max_track":
        int(
            global_max_row[
                "max_track"
            ]
        ),

    "global_max_landmark":
        str(
            global_max_row[
                "max_landmark_name"
            ]
        ),

    "global_max_coordinate":
        str(
            global_max_row[
                "max_coordinate"
            ]
        ),

    "coverage_percentiles":
        coverage_percentiles,

    "normalized_abs_percentiles":
        normalized_percentiles,

    "motion_abs_percentiles":
        motion_percentiles,

    "classes_total":
        len(class_df),

    "classes_ge_2_usable":
        classes_ge_2,

    "classes_ge_3_usable":
        classes_ge_3,

    "classes_ge_4_usable":
        classes_ge_4,

    "classes_ge_5_usable":
        classes_ge_5,

    "classes_ge_6_usable":
        classes_ge_6,

    "classes_ge_8_usable":
        classes_ge_8,

    "classes_ge_10_usable":
        classes_ge_10,

    "singleton_classes":
        singleton_total,

    "classes_zero_usable":
        zero_usable_classes,

    "videos_with_max_feature_gt_10":
        len(
            extreme_10
        ),

    "videos_with_max_feature_gt_20":
        len(
            extreme_20
        )
}

with open(
    SUMMARY_JSON,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# ==========================================================================================
# 38. L7 CHECKPOINT DECISION
# ==========================================================================================

complete_dataset = (
    len(feature_files)
    ==
    len(dataset)
)

all_structural = (
    structural_pass_count
    ==
    len(feature_files)
)

all_reconstruct = (
    len(
        reconstruction_failures
    )
    ==
    0
)

all_raw_present = (
    len(raw_files)
    ==
    len(dataset)
)

l7_pass = bool(
    complete_dataset
    and
    all_structural
    and
    all_reconstruct
    and
    all_raw_present
)

checkpoint = {
    "complete_dataset":
        complete_dataset,

    "all_structural":
        all_structural,

    "all_reconstruct":
        all_reconstruct,

    "all_raw_present":
        all_raw_present,

    "l7_pass":
        l7_pass
}

with open(
    CHECKPOINT_JSON,
    "w"
) as f:

    json.dump(
        checkpoint,
        f,
        indent=4
    )


# ==========================================================================================
# 39. FINAL REPORT
# ==========================================================================================

print("\n" + "=" * 120)
print("L7 FINAL REPORT")
print("=" * 120)

print("\nDATASET INTEGRITY")
print("-" * 120)

print(
    "Dataset rows              :",
    len(dataset)
)

print(
    "Unique glosses            :",
    dataset[
        "gloss"
    ].nunique()
)

print(
    "Feature files             :",
    len(feature_files)
)

print(
    "Raw files                 :",
    len(raw_files)
)

print(
    "Duplicate UID rows        :",
    duplicate_uid_count
)

print(
    "Missing feature files     :",
    len(missing_features)
)

print(
    "Missing raw files         :",
    len(missing_raw)
)


print("\nSTRUCTURAL / NUMERICAL")
print("-" * 120)

print(
    "Structural PASS           :",
    structural_pass_count
)

print(
    "Structural FAIL           :",
    structural_fail_count
)

print(
    "Feature reconstruction FAIL:",
    len(
        reconstruction_failures
    )
)


print("\nPROVISIONAL USABILITY")
print("-" * 120)

print(
    "Usable videos             :",
    usable_count
)

print(
    "Rejected videos           :",
    rejected_count
)


print("\nGLOBAL MAGNITUDES")
print("-" * 120)

print(
    "Max |feature|             :",
    f"{global_max_feature:.6f}"
)

print(
    "Max |normalized|          :",
    f"{global_max_normalized:.6f}"
)

print(
    "Max |motion|              :",
    f"{global_max_motion:.6f}"
)


print("\nGLOBAL MAXIMUM LOCATION")
print("-" * 120)

print(
    "UID                       :",
    global_max_row[
        "uid"
    ]
)

print(
    "Gloss                     :",
    global_max_row[
        "gloss"
    ]
)

print(
    "Feature group             :",
    global_max_row[
        "max_feature_group"
    ]
)

print(
    "Track                     :",
    global_max_row[
        "max_track"
    ]
)

print(
    "Landmark                  :",
    global_max_row[
        "max_landmark_name"
    ]
)

print(
    "Coordinate                :",
    global_max_row[
        "max_coordinate"
    ]
)

print(
    "Sampled sequence index    :",
    global_max_row[
        "max_sequence_index"
    ]
)

print(
    "Original source frame     :",
    global_max_row[
        "max_source_frame"
    ]
)


print("\nCLASS SUPPORT AFTER PROVISIONAL QUALITY FILTER")
print("-" * 120)

print(
    "All glosses               :",
    len(class_df)
)

print(
    ">= 2 usable videos        :",
    classes_ge_2
)

print(
    ">= 3 usable videos        :",
    classes_ge_3
)

print(
    ">= 4 usable videos        :",
    classes_ge_4
)

print(
    ">= 5 usable videos        :",
    classes_ge_5
)

print(
    ">= 6 usable videos        :",
    classes_ge_6
)

print(
    ">= 8 usable videos        :",
    classes_ge_8
)

print(
    ">=10 usable videos        :",
    classes_ge_10
)


print("\n" + "=" * 120)
print("L7 CHECKPOINT")
print("=" * 120)

if l7_pass:

    print()
    print(
        "[PASS] L7 FULL DATASET AUDIT COMPLETE."
    )

    print()
    print(
        "All L6 files are structurally consistent."
    )

    print(
        "No L6 features were modified."
    )

    print(
        "Outliers were preserved for scientific analysis."
    )

    print()
    print(
        "NEXT: L8 — FREEZE VALID CLASS POPULATIONS."
    )

else:

    print()
    print(
        "[REVIEW] L7 FOUND DATASET INTEGRITY ISSUES."
    )

    print()
    print(
        "Do NOT proceed to model training until "
        "the reported integrity issue is understood."
    )


# ==========================================================================================
# 40. OUTPUT LOCATIONS
# ==========================================================================================

print("\n" + "=" * 120)
print("L7 OUTPUTS")
print("=" * 120)

print("\nVideo-level audit:")
print(VIDEO_AUDIT_CSV)

print("\nFeature outliers:")
print(OUTLIER_CSV)

print("\nExtreme sequences:")
print(EXTREME_SEQUENCE_CSV)

print("\nRejected sequences:")
print(REJECTED_CSV)

print("\nClass support:")
print(CLASS_SUPPORT_CSV)

print("\nSupport threshold summary:")
print(SUPPORT_SUMMARY_CSV)

print("\nSummary:")
print(SUMMARY_JSON)

print("\nCheckpoint:")
print(CHECKPOINT_JSON)

print("\n" + "=" * 120)

CISLR LANDMARK MODEL — L7
FULL LANDMARK DATASET AUDIT
L6 feature directory : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l6_full_extraction/features_32x254
L6 raw directory     : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l6_full_extraction/raw_sequences
L7 output            : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l7_full_dataset_audit

Expected sequence    : (32, 254)
Expected normalized  : (32, 2, 21, 3)
Expected motion      : (32, 2, 21, 3)
Expected state       : (32, 2)

IMPORTANT:
  L7 DOES NOT MODIFY L6 FEATURES.
  LARGE VALUES ARE AUDITED, NOT CLIPPED.

INPUT COUNTS
------------------------------------------------------------------------------------------------------------------------
Feature files : 7050
Raw files     : 7050

Dataset rows   : 7050
Unique glosses : 4764

METADATA INTEGRITY
Duplicate UID rows: 0
Dataset UIDs missing feature file: 0
Extra feature UIDs: 0
Dataset UIDs missing raw file: 0

In [10]:
# ==========================================================================================
# CISLR LANDMARK MODEL — L8
# EXPERIMENTAL POPULATION FREEZE
#
# INPUT:
#   dataset.csv
#   L7 video-level audit
#   L6 feature files
#
# PURPOSE:
#   1. Treat dataset.csv as authoritative label metadata.
#   2. Audit the 4764-vs-4765 gloss discrepancy discovered in L7.
#   3. Explicitly identify missing / blank / invalid labels.
#   4. Reconstruct the L7 provisional usability decision.
#   5. Freeze a CONTROLLED population:
#          >= 4 usable videos per gloss
#      This population will later support:
#          >=2 train + >=1 validation + >=1 test
#   6. Freeze a FULL-VOCABULARY population:
#          every usable video with a valid gloss
#   7. Generate deterministic class mappings.
#   8. Generate inclusion / exclusion manifests.
#   9. Generate fingerprints so later stages can verify
#      they are using exactly the frozen L8 population.
#
# IMPORTANT:
#   - L8 DOES NOT MODIFY L6 FEATURES.
#   - L8 DOES NOT MODIFY L7 REPORTS.
#   - L8 DOES NOT CREATE TRAIN / VAL / TEST SPLITS.
#   - L8 DOES NOT TRAIN ANY MODEL.
#   - L8 DOES NOT CLIP OUTLIERS.
#
# NEXT:
#   L9 = TRAIN / VALIDATION / TEST SPLIT FREEZE
# ==========================================================================================


# ==========================================================================================
# 0. IMPORTS
# ==========================================================================================

from pathlib import Path
import hashlib
import json
import math
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 100)

np.set_printoptions(
    precision=6,
    suppress=True
)


# ==========================================================================================
# 1. PATHS
# ==========================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

DATASET_CSV = Path(
    "/home/dipshikha/Music/cao/dataset.csv"
)

LANDMARK_ROOT = (
    PROJECT_ROOT
    / "CISLR_LANDMARK_MODEL"
)

L6_ROOT = (
    LANDMARK_ROOT
    / "audit"
    / "l6_full_extraction"
)

FEATURE_DIR = (
    L6_ROOT
    / "features_32x254"
)

RAW_DIR = (
    L6_ROOT
    / "raw_sequences"
)

L7_ROOT = (
    LANDMARK_ROOT
    / "audit"
    / "l7_full_dataset_audit"
)

L7_VIDEO_AUDIT = (
    L7_ROOT
    / "reports"
    / "l7_video_level_audit.csv"
)

L7_CLASS_SUPPORT = (
    L7_ROOT
    / "class_analysis"
    / "l7_class_support.csv"
)

L7_CHECKPOINT = (
    L7_ROOT
    / "reports"
    / "l7_checkpoint.json"
)


# ==========================================================================================
# 2. L8 OUTPUT DIRECTORIES
# ==========================================================================================

L8_ROOT = (
    LANDMARK_ROOT
    / "audit"
    / "l8_population_freeze"
)

REPORT_DIR = (
    L8_ROOT
    / "reports"
)

CONTROLLED_DIR = (
    L8_ROOT
    / "controlled_population"
)

FULL_VOCAB_DIR = (
    L8_ROOT
    / "full_vocabulary_population"
)

MAPPING_DIR = (
    L8_ROOT
    / "class_mappings"
)

METADATA_DIR = (
    L8_ROOT
    / "metadata_audit"
)

for directory in [
    L8_ROOT,
    REPORT_DIR,
    CONTROLLED_DIR,
    FULL_VOCAB_DIR,
    MAPPING_DIR,
    METADATA_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ==========================================================================================
# 3. OUTPUT FILE PATHS
# ==========================================================================================

METADATA_AUDIT_CSV = (
    METADATA_DIR
    / "l8_metadata_audit.csv"
)

INVALID_LABELS_CSV = (
    METADATA_DIR
    / "l8_invalid_labels.csv"
)

LABEL_MISMATCH_CSV = (
    METADATA_DIR
    / "l8_dataset_vs_feature_label_mismatch.csv"
)

GLOSS_DISCREPANCY_CSV = (
    METADATA_DIR
    / "l8_gloss_discrepancy.csv"
)

MASTER_MANIFEST_CSV = (
    REPORT_DIR
    / "l8_master_manifest.csv"
)

CONTROLLED_MANIFEST_CSV = (
    CONTROLLED_DIR
    / "l8_controlled_manifest.csv"
)

CONTROLLED_CLASSES_CSV = (
    CONTROLLED_DIR
    / "l8_controlled_classes.csv"
)

CONTROLLED_EXCLUDED_CSV = (
    CONTROLLED_DIR
    / "l8_controlled_excluded.csv"
)

CONTROLLED_CLASS_MAP_CSV = (
    MAPPING_DIR
    / "l8_controlled_class_mapping.csv"
)

CONTROLLED_CLASS_MAP_JSON = (
    MAPPING_DIR
    / "l8_controlled_class_mapping.json"
)

FULL_MANIFEST_CSV = (
    FULL_VOCAB_DIR
    / "l8_full_vocabulary_manifest.csv"
)

FULL_CLASSES_CSV = (
    FULL_VOCAB_DIR
    / "l8_full_vocabulary_classes.csv"
)

FULL_EXCLUDED_CSV = (
    FULL_VOCAB_DIR
    / "l8_full_vocabulary_excluded.csv"
)

FULL_CLASS_MAP_CSV = (
    MAPPING_DIR
    / "l8_full_vocabulary_class_mapping.csv"
)

FULL_CLASS_MAP_JSON = (
    MAPPING_DIR
    / "l8_full_vocabulary_class_mapping.json"
)

SUPPORT_SUMMARY_CSV = (
    REPORT_DIR
    / "l8_support_summary.csv"
)

POPULATION_SUMMARY_JSON = (
    REPORT_DIR
    / "l8_population_summary.json"
)

FREEZE_MANIFEST_JSON = (
    REPORT_DIR
    / "l8_freeze_manifest.json"
)

CHECKPOINT_JSON = (
    REPORT_DIR
    / "l8_checkpoint.json"
)


# ==========================================================================================
# 4. FROZEN L8 CONFIGURATION
# ==========================================================================================

# This is inherited from L7's provisional usability logic.

MIN_ORIGINAL_DETECTION_RATE = 0.20

MIN_SAMPLED_HAND_FRAMES = 8


# Controlled experiment:
#
# Need at least:
#   2 train
#   1 validation
#   1 test
#
# Therefore minimum support = 4.

CONTROLLED_MIN_USABLE_VIDEOS = 4


# Representation frozen in L5/L5.1/L6.

EXPECTED_SEQUENCE_LENGTH = 32
EXPECTED_FEATURE_DIM = 254


# Invalid labels.
#
# Important:
# dataset.csv is authoritative.
#
# A genuinely missing CSV value may become NaN when read by pandas.
# We explicitly reject it rather than converting it to the class "nan".

INVALID_LABEL_STRINGS = {
    "",
    "nan",
    "none",
    "null",
    "na",
    "n/a"
}


# ==========================================================================================
# 5. HEADER
# ==========================================================================================

print("=" * 125)
print("CISLR LANDMARK MODEL — L8")
print("EXPERIMENTAL POPULATION FREEZE")
print("=" * 125)

print()
print("Dataset metadata :", DATASET_CSV)
print("L7 audit         :", L7_VIDEO_AUDIT)
print("L6 features      :", FEATURE_DIR)
print("L8 output        :", L8_ROOT)

print()
print("CONTROLLED POPULATION RULE")
print("-" * 125)
print(
    f"Minimum usable videos per gloss : "
    f"{CONTROLLED_MIN_USABLE_VIDEOS}"
)
print(
    "Target split capability        : "
    ">=2 train + >=1 validation + >=1 test"
)

print()
print("FULL-VOCABULARY POPULATION RULE")
print("-" * 125)
print(
    "Every provisionally usable video with a valid "
    "authoritative dataset.csv gloss."
)

print()
print("IMPORTANT")
print("-" * 125)
print("L8 freezes populations only.")
print("L8 DOES NOT create train/validation/test splits.")
print("L8 DOES NOT modify features.")
print("L8 DOES NOT clip outliers.")

print("=" * 125)


# ==========================================================================================
# 6. VERIFY REQUIRED INPUTS
# ==========================================================================================

required_inputs = [
    DATASET_CSV,
    L7_VIDEO_AUDIT,
    L7_CHECKPOINT,
    FEATURE_DIR,
    RAW_DIR
]

for path in required_inputs:

    if not path.exists():

        raise FileNotFoundError(
            f"Required L8 input missing:\n{path}"
        )

print("\n[PASS] Required L8 inputs exist.")


# ==========================================================================================
# 7. VERIFY L7 CHECKPOINT
# ==========================================================================================

with open(
    L7_CHECKPOINT,
    "r"
) as f:

    l7_checkpoint = json.load(f)

print("\nL7 CHECKPOINT")
print("-" * 125)

for key, value in l7_checkpoint.items():

    print(
        f"{key:30s}: {value}"
    )

if not bool(
    l7_checkpoint.get(
        "l7_pass",
        False
    )
):

    raise RuntimeError(
        "L7 checkpoint is not PASS. "
        "Do not freeze L8 populations."
    )

print("\n[PASS] L7 checkpoint verified.")


# ==========================================================================================
# 8. LOAD DATASET.CSV WITHOUT DESTROYING MISSING LABEL INFORMATION
# ==========================================================================================

dataset = pd.read_csv(
    DATASET_CSV,
    dtype={
        "uid": "string",
        "gloss": "string"
    }
)

required_columns = {
    "uid",
    "gloss"
}

missing_columns = (
    required_columns
    -
    set(dataset.columns)
)

if missing_columns:

    raise RuntimeError(
        "dataset.csv missing required columns: "
        f"{sorted(missing_columns)}"
    )


# Preserve original row number for audit.

dataset = dataset.copy()

dataset["dataset_row"] = np.arange(
    len(dataset),
    dtype=np.int64
)


# ==========================================================================================
# 9. CLEAN UID WITHOUT CHANGING LABEL SEMANTICS
# ==========================================================================================

dataset["uid_clean"] = (
    dataset["uid"]
    .astype("string")
    .str.strip()
)

uid_missing_mask = (
    dataset["uid_clean"].isna()
    |
    (
        dataset["uid_clean"]
        ==
        ""
    )
)

if uid_missing_mask.any():

    print(
        "\n[ERROR] Missing/blank UID rows detected:"
    )

    print(
        dataset.loc[
            uid_missing_mask
        ].to_string(
            index=False
        )
    )

    raise RuntimeError(
        "dataset.csv contains missing/blank UIDs."
    )


# ==========================================================================================
# 10. AUTHORITATIVE GLOSS VALIDITY
# ==========================================================================================

def normalize_gloss_for_validation(value):
    """
    Validation only.

    This does NOT rename valid glosses.
    It only helps detect missing/invalid labels.
    """

    if pd.isna(value):
        return None

    text = str(value).strip()

    if text.lower() in INVALID_LABEL_STRINGS:
        return None

    return text


dataset["authoritative_gloss"] = (
    dataset["gloss"]
    .apply(
        normalize_gloss_for_validation
    )
)

dataset["valid_gloss"] = (
    dataset["authoritative_gloss"]
    .notna()
)


# ==========================================================================================
# 11. UID DUPLICATE AUDIT
# ==========================================================================================

duplicate_uid_mask = (
    dataset["uid_clean"]
    .duplicated(
        keep=False
    )
)

duplicate_uid_rows = dataset[
    duplicate_uid_mask
].copy()

print("\n" + "=" * 125)
print("DATASET METADATA AUDIT")
print("=" * 125)

print(
    "Dataset rows                  :",
    len(dataset)
)

print(
    "Duplicate UID rows            :",
    len(
        duplicate_uid_rows
    )
)

print(
    "Invalid/missing gloss rows    :",
    int(
        (
            ~dataset[
                "valid_gloss"
            ]
        ).sum()
    )
)

if len(duplicate_uid_rows) > 0:

    duplicate_uid_path = (
        METADATA_DIR
        / "l8_duplicate_uids.csv"
    )

    duplicate_uid_rows.to_csv(
        duplicate_uid_path,
        index=False
    )

    print(
        "\n[SAVED]",
        duplicate_uid_path
    )

    raise RuntimeError(
        "Duplicate UIDs found. "
        "L8 cannot safely freeze populations."
    )


# ==========================================================================================
# 12. SAVE INVALID LABEL AUDIT
# ==========================================================================================

invalid_labels_df = dataset[
    ~dataset[
        "valid_gloss"
    ]
].copy()

invalid_labels_df.to_csv(
    INVALID_LABELS_CSV,
    index=False
)

print(
    "\nInvalid/missing authoritative labels:"
)

if len(invalid_labels_df) == 0:

    print("None.")

else:

    display_columns = [
        col
        for col in [
            "dataset_row",
            "uid_clean",
            "gloss",
            "category",
            "duration"
        ]
        if col in invalid_labels_df.columns
    ]

    print(
        invalid_labels_df[
            display_columns
        ].to_string(
            index=False
        )
    )

print(
    "\n[SAVED]",
    INVALID_LABELS_CSV
)


# ==========================================================================================
# 13. AUTHORITATIVE UNIQUE GLOSS COUNT
# ==========================================================================================

valid_dataset = dataset[
    dataset[
        "valid_gloss"
    ]
].copy()

authoritative_unique_glosses = int(
    valid_dataset[
        "authoritative_gloss"
    ].nunique()
)

print()
print(
    "Authoritative valid glosses:",
    authoritative_unique_glosses
)


# ==========================================================================================
# 14. LOAD L7 VIDEO AUDIT
# ==========================================================================================

l7 = pd.read_csv(
    L7_VIDEO_AUDIT,
    dtype={
        "uid": "string",
        "gloss": "string"
    }
)

if "uid" not in l7.columns:

    raise RuntimeError(
        "L7 audit has no uid column."
    )

l7["uid_clean"] = (
    l7["uid"]
    .astype("string")
    .str.strip()
)

print()
print(
    "L7 video audit rows:",
    len(l7)
)


# ==========================================================================================
# 15. VERIFY UID SETS
# ==========================================================================================

dataset_uid_set = set(
    dataset[
        "uid_clean"
    ].astype(str)
)

l7_uid_set = set(
    l7[
        "uid_clean"
    ].astype(str)
)

feature_uid_set = {
    path.stem
    for path in FEATURE_DIR.glob(
        "*.npz"
    )
}

raw_uid_set = {
    path.stem
    for path in RAW_DIR.glob(
        "*.npz"
    )
}

dataset_missing_l7 = sorted(
    dataset_uid_set
    -
    l7_uid_set
)

l7_extra = sorted(
    l7_uid_set
    -
    dataset_uid_set
)

dataset_missing_features = sorted(
    dataset_uid_set
    -
    feature_uid_set
)

dataset_missing_raw = sorted(
    dataset_uid_set
    -
    raw_uid_set
)

print("\nUID SET CONSISTENCY")
print("-" * 125)

print(
    "Dataset UIDs missing from L7 :",
    len(
        dataset_missing_l7
    )
)

print(
    "L7 UIDs not in dataset       :",
    len(
        l7_extra
    )
)

print(
    "Dataset UIDs missing feature :",
    len(
        dataset_missing_features
    )
)

print(
    "Dataset UIDs missing raw     :",
    len(
        dataset_missing_raw
    )
)

if (
    dataset_missing_l7
    or
    l7_extra
    or
    dataset_missing_features
    or
    dataset_missing_raw
):

    raise RuntimeError(
        "UID set inconsistency found. "
        "L8 freeze aborted."
    )


# ==========================================================================================
# 16. MERGE AUTHORITATIVE METADATA WITH L7
# ==========================================================================================

metadata_columns = [
    "dataset_row",
    "uid_clean",
    "authoritative_gloss",
    "valid_gloss"
]

# Keep extra metadata if available.

for optional_column in [
    "duration",
    "category"
]:

    if optional_column in dataset.columns:

        metadata_columns.append(
            optional_column
        )

metadata = dataset[
    metadata_columns
].copy()

master = l7.merge(
    metadata,
    on="uid_clean",
    how="left",
    validate="one_to_one"
)


# ==========================================================================================
# 17. PRESERVE L7 FEATURE LABEL FOR AUDIT ONLY
# ==========================================================================================

if "gloss" in master.columns:

    master = master.rename(
        columns={
            "gloss":
                "feature_gloss"
        }
    )

else:

    master[
        "feature_gloss"
    ] = pd.NA


# ==========================================================================================
# 18. LABEL MISMATCH AUDIT
# ==========================================================================================

def clean_comparison_label(value):

    if pd.isna(value):
        return None

    text = str(value).strip()

    if text.lower() in INVALID_LABEL_STRINGS:
        return None

    return text


master[
    "feature_gloss_clean"
] = master[
    "feature_gloss"
].apply(
    clean_comparison_label
)

master[
    "label_match"
] = (
    master[
        "feature_gloss_clean"
    ]
    ==
    master[
        "authoritative_gloss"
    ]
)


# Missing labels are intentionally not considered valid matches.

master.loc[
    ~master[
        "valid_gloss"
    ],
    "label_match"
] = False


label_mismatch_df = master[
    ~master[
        "label_match"
    ]
].copy()

label_mismatch_df.to_csv(
    LABEL_MISMATCH_CSV,
    index=False
)

print("\n" + "=" * 125)
print("DATASET.CSV VS FEATURE LABEL AUDIT")
print("=" * 125)

print(
    "Label mismatch rows:",
    len(
        label_mismatch_df
    )
)

if len(label_mismatch_df) > 0:

    print()

    print(
        label_mismatch_df[
            [
                "uid_clean",
                "authoritative_gloss",
                "feature_gloss"
            ]
        ]
        .head(50)
        .to_string(
            index=False
        )
    )

print(
    "\n[SAVED]",
    LABEL_MISMATCH_CSV
)


# ==========================================================================================
# 19. EXPLICITLY INVESTIGATE 4764-vs-4765 DISCREPANCY
# ==========================================================================================

dataset_gloss_set = set(
    valid_dataset[
        "authoritative_gloss"
    ].astype(str)
)

feature_gloss_series = (
    master[
        "feature_gloss_clean"
    ]
    .dropna()
    .astype(str)
)

feature_gloss_set = set(
    feature_gloss_series
)

only_dataset = sorted(
    dataset_gloss_set
    -
    feature_gloss_set
)

only_feature = sorted(
    feature_gloss_set
    -
    dataset_gloss_set
)

discrepancy_rows = []

for gloss in only_dataset:

    discrepancy_rows.append(
        {
            "gloss":
                gloss,

            "location":
                "dataset_only"
        }
    )

for gloss in only_feature:

    discrepancy_rows.append(
        {
            "gloss":
                gloss,

            "location":
                "feature_only"
        }
    )

gloss_discrepancy_df = pd.DataFrame(
    discrepancy_rows,
    columns=[
        "gloss",
        "location"
    ]
)

gloss_discrepancy_df.to_csv(
    GLOSS_DISCREPANCY_CSV,
    index=False
)

print("\n" + "=" * 125)
print("GLOSS VOCABULARY DISCREPANCY")
print("=" * 125)

print(
    "Authoritative valid gloss count :",
    len(
        dataset_gloss_set
    )
)

print(
    "Feature-derived valid gloss count:",
    len(
        feature_gloss_set
    )
)

print(
    "Glosses only in dataset.csv     :",
    len(
        only_dataset
    )
)

print(
    "Glosses only in feature metadata:",
    len(
        only_feature
    )
)

if len(only_dataset) > 0:

    print(
        "\nDataset-only glosses:"
    )

    print(
        only_dataset[:50]
    )

if len(only_feature) > 0:

    print(
        "\nFeature-only glosses:"
    )

    print(
        only_feature[:50]
    )

print(
    "\n[SAVED]",
    GLOSS_DISCREPANCY_CSV
)


# ==========================================================================================
# 20. RECONSTRUCT PROVISIONAL USABILITY
# ==========================================================================================

required_l7_columns = [
    "structural_pass",
    "original_detection_rate",
    "any_hand_frames"
]

for column in required_l7_columns:

    if column not in master.columns:

        raise RuntimeError(
            f"L7 audit missing column: "
            f"{column}"
        )


master[
    "l8_quality_usable"
] = (
    master[
        "structural_pass"
    ].astype(bool)
    &
    (
        master[
            "original_detection_rate"
        ]
        >=
        MIN_ORIGINAL_DETECTION_RATE
    )
    &
    (
        master[
            "any_hand_frames"
        ]
        >=
        MIN_SAMPLED_HAND_FRAMES
    )
)


# Final L8 usable additionally requires a valid authoritative label.

master[
    "l8_usable"
] = (
    master[
        "l8_quality_usable"
    ]
    &
    master[
        "valid_gloss"
    ]
)


# ==========================================================================================
# 21. VERIFY AGAINST L7 PROVISIONAL FLAG
# ==========================================================================================

if "l7_provisional_usable" in master.columns:

    master[
        "l7_l8_usability_match"
    ] = (
        master[
            "l7_provisional_usable"
        ].astype(bool)
        ==
        master[
            "l8_quality_usable"
        ]
    )

    usability_mismatch_count = int(
        (
            ~master[
                "l7_l8_usability_match"
            ]
        ).sum()
    )

else:

    master[
        "l7_l8_usability_match"
    ] = True

    usability_mismatch_count = 0


print("\n" + "=" * 125)
print("USABILITY RECONSTRUCTION")
print("=" * 125)

print(
    "L8 quality-usable videos      :",
    int(
        master[
            "l8_quality_usable"
        ].sum()
    )
)

print(
    "L8 final usable + valid label :",
    int(
        master[
            "l8_usable"
        ].sum()
    )
)

print(
    "L7/L8 usability mismatches    :",
    usability_mismatch_count
)

if usability_mismatch_count > 0:

    mismatch_path = (
        REPORT_DIR
        / "l8_l7_usability_mismatch.csv"
    )

    master[
        ~master[
            "l7_l8_usability_match"
        ]
    ].to_csv(
        mismatch_path,
        index=False
    )

    raise RuntimeError(
        "L8 could not reproduce L7 usability logic."
    )


# ==========================================================================================
# 22. ADD FEATURE / RAW PATHS
# ==========================================================================================

master[
    "feature_path"
] = master[
    "uid_clean"
].apply(
    lambda uid:
        str(
            FEATURE_DIR
            /
            f"{uid}.npz"
        )
)

master[
    "raw_path"
] = master[
    "uid_clean"
].apply(
    lambda uid:
        str(
            RAW_DIR
            /
            f"{uid}.npz"
        )
)


# ==========================================================================================
# 23. FINAL AUTHORITATIVE GLOSS COLUMN
# ==========================================================================================

# From this point onward:
#
# authoritative_gloss is the ONLY label used for population construction.
#
# feature_gloss remains audit metadata only.

master[
    "gloss"
] = master[
    "authoritative_gloss"
]


# ==========================================================================================
# 24. SAVE MASTER MANIFEST
# ==========================================================================================

master = master.sort_values(
    [
        "dataset_row"
    ]
).reset_index(
    drop=True
)

master.to_csv(
    MASTER_MANIFEST_CSV,
    index=False
)

print(
    "\n[SAVED]",
    MASTER_MANIFEST_CSV
)


# ==========================================================================================
# 25. FULL-VOCABULARY POPULATION
# ==========================================================================================

full_population = master[
    master[
        "l8_usable"
    ]
].copy()

full_population = full_population.sort_values(
    [
        "gloss",
        "uid_clean"
    ]
).reset_index(
    drop=True
)


# ==========================================================================================
# 26. FULL-VOCABULARY CLASS SUPPORT
# ==========================================================================================

full_class_support = (
    full_population
    .groupby(
        "gloss"
    )
    .agg(
        usable_videos=(
            "uid_clean",
            "count"
        ),

        mean_detection_rate=(
            "original_detection_rate",
            "mean"
        ),

        median_detection_rate=(
            "original_detection_rate",
            "median"
        ),

        mean_sampled_hand_frames=(
            "any_hand_frames",
            "mean"
        ),

        max_abs_feature=(
            "max_abs_feature",
            "max"
        )
    )
    .reset_index()
)

full_class_support = (
    full_class_support
    .sort_values(
        [
            "gloss"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ==========================================================================================
# 27. DETERMINISTIC FULL-VOCAB CLASS MAPPING
# ==========================================================================================

full_glosses = sorted(
    full_class_support[
        "gloss"
    ].tolist()
)

full_gloss_to_id = {
    gloss:
        index

    for index, gloss
    in enumerate(
        full_glosses
    )
}

full_id_to_gloss = {
    index:
        gloss

    for gloss, index
    in full_gloss_to_id.items()
}

full_population[
    "class_id"
] = full_population[
    "gloss"
].map(
    full_gloss_to_id
).astype(
    int
)

full_class_support[
    "class_id"
] = full_class_support[
    "gloss"
].map(
    full_gloss_to_id
).astype(
    int
)

full_class_support = full_class_support[
    [
        "class_id",
        "gloss",
        "usable_videos",
        "mean_detection_rate",
        "median_detection_rate",
        "mean_sampled_hand_frames",
        "max_abs_feature"
    ]
]


# ==========================================================================================
# 28. SAVE FULL-VOCABULARY POPULATION
# ==========================================================================================

full_population.to_csv(
    FULL_MANIFEST_CSV,
    index=False
)

full_class_support.to_csv(
    FULL_CLASSES_CSV,
    index=False
)

full_mapping_df = pd.DataFrame(
    {
        "class_id":
            list(
                range(
                    len(
                        full_glosses
                    )
                )
            ),

        "gloss":
            full_glosses
    }
)

full_mapping_df.to_csv(
    FULL_CLASS_MAP_CSV,
    index=False
)

with open(
    FULL_CLASS_MAP_JSON,
    "w"
) as f:

    json.dump(
        {
            "gloss_to_id":
                full_gloss_to_id,

            "id_to_gloss":
                {
                    str(key):
                        value

                    for key, value
                    in full_id_to_gloss.items()
                }
        },
        f,
        indent=4,
        ensure_ascii=False
    )


# ==========================================================================================
# 29. FULL-VOCAB EXCLUSIONS
# ==========================================================================================

full_excluded = master[
    ~master[
        "l8_usable"
    ]
].copy()


def exclusion_reason(row):

    reasons = []

    if not bool(
        row[
            "valid_gloss"
        ]
    ):
        reasons.append(
            "INVALID_OR_MISSING_GLOSS"
        )

    if not bool(
        row[
            "structural_pass"
        ]
    ):
        reasons.append(
            "STRUCTURAL_FAIL"
        )

    if (
        float(
            row[
                "original_detection_rate"
            ]
        )
        <
        MIN_ORIGINAL_DETECTION_RATE
    ):
        reasons.append(
            "LOW_ORIGINAL_DETECTION"
        )

    if (
        int(
            row[
                "any_hand_frames"
            ]
        )
        <
        MIN_SAMPLED_HAND_FRAMES
    ):
        reasons.append(
            "INSUFFICIENT_SAMPLED_HAND_FRAMES"
        )

    if len(reasons) == 0:

        reasons.append(
            "OTHER"
        )

    return "|".join(
        reasons
    )


if len(full_excluded) > 0:

    full_excluded[
        "exclusion_reason"
    ] = full_excluded.apply(
        exclusion_reason,
        axis=1
    )

full_excluded.to_csv(
    FULL_EXCLUDED_CSV,
    index=False
)


# ==========================================================================================
# 30. CONTROLLED POPULATION — DETERMINE ELIGIBLE CLASSES
# ==========================================================================================

controlled_eligible_classes = (
    full_class_support[
        full_class_support[
            "usable_videos"
        ]
        >=
        CONTROLLED_MIN_USABLE_VIDEOS
    ]
    .copy()
)

controlled_glosses = sorted(
    controlled_eligible_classes[
        "gloss"
    ].tolist()
)

controlled_gloss_set = set(
    controlled_glosses
)


# ==========================================================================================
# 31. BUILD CONTROLLED POPULATION
# ==========================================================================================

controlled_population = full_population[
    full_population[
        "gloss"
    ].isin(
        controlled_gloss_set
    )
].copy()

controlled_population = (
    controlled_population
    .sort_values(
        [
            "gloss",
            "uid_clean"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ==========================================================================================
# 32. DETERMINISTIC CONTROLLED CLASS MAPPING
# ==========================================================================================

controlled_gloss_to_id = {
    gloss:
        index

    for index, gloss
    in enumerate(
        controlled_glosses
    )
}

controlled_id_to_gloss = {
    index:
        gloss

    for gloss, index
    in controlled_gloss_to_id.items()
}

controlled_population[
    "class_id"
] = controlled_population[
    "gloss"
].map(
    controlled_gloss_to_id
).astype(
    int
)


# ==========================================================================================
# 33. CONTROLLED CLASS TABLE
# ==========================================================================================

controlled_classes = (
    controlled_population
    .groupby(
        "gloss"
    )
    .agg(
        usable_videos=(
            "uid_clean",
            "count"
        ),

        mean_detection_rate=(
            "original_detection_rate",
            "mean"
        ),

        median_detection_rate=(
            "original_detection_rate",
            "median"
        ),

        min_detection_rate=(
            "original_detection_rate",
            "min"
        ),

        max_detection_rate=(
            "original_detection_rate",
            "max"
        ),

        mean_sampled_hand_frames=(
            "any_hand_frames",
            "mean"
        ),

        mean_zero_hand_frames=(
            "zero_hand_frames",
            "mean"
        ),

        max_abs_feature=(
            "max_abs_feature",
            "max"
        ),

        total_scale_repairs=(
            "scale_repairs",
            "sum"
        )
    )
    .reset_index()
)

controlled_classes[
    "class_id"
] = controlled_classes[
    "gloss"
].map(
    controlled_gloss_to_id
).astype(
    int
)

controlled_classes = (
    controlled_classes[
        [
            "class_id",
            "gloss",
            "usable_videos",
            "mean_detection_rate",
            "median_detection_rate",
            "min_detection_rate",
            "max_detection_rate",
            "mean_sampled_hand_frames",
            "mean_zero_hand_frames",
            "max_abs_feature",
            "total_scale_repairs"
        ]
    ]
    .sort_values(
        "class_id"
    )
    .reset_index(
        drop=True
    )
)


# ==========================================================================================
# 34. CONTROLLED EXCLUSIONS
# ==========================================================================================

controlled_excluded = master[
    ~master[
        "uid_clean"
    ].isin(
        set(
            controlled_population[
                "uid_clean"
            ]
        )
    )
].copy()


def controlled_exclusion_reason(row):

    reasons = []

    if not bool(
        row[
            "valid_gloss"
        ]
    ):

        reasons.append(
            "INVALID_OR_MISSING_GLOSS"
        )

        return "|".join(
            reasons
        )

    if not bool(
        row[
            "l8_quality_usable"
        ]
    ):

        if not bool(
            row[
                "structural_pass"
            ]
        ):
            reasons.append(
                "STRUCTURAL_FAIL"
            )

        if (
            float(
                row[
                    "original_detection_rate"
                ]
            )
            <
            MIN_ORIGINAL_DETECTION_RATE
        ):
            reasons.append(
                "LOW_ORIGINAL_DETECTION"
            )

        if (
            int(
                row[
                    "any_hand_frames"
                ]
            )
            <
            MIN_SAMPLED_HAND_FRAMES
        ):
            reasons.append(
                "INSUFFICIENT_SAMPLED_HAND_FRAMES"
            )

    else:

        reasons.append(
            "CLASS_SUPPORT_BELOW_4"
        )

    if len(reasons) == 0:

        reasons.append(
            "OTHER"
        )

    return "|".join(
        reasons
    )


if len(controlled_excluded) > 0:

    controlled_excluded[
        "exclusion_reason"
    ] = controlled_excluded.apply(
        controlled_exclusion_reason,
        axis=1
    )

controlled_excluded.to_csv(
    CONTROLLED_EXCLUDED_CSV,
    index=False
)


# ==========================================================================================
# 35. SAVE CONTROLLED POPULATION
# ==========================================================================================

controlled_population.to_csv(
    CONTROLLED_MANIFEST_CSV,
    index=False
)

controlled_classes.to_csv(
    CONTROLLED_CLASSES_CSV,
    index=False
)

controlled_mapping_df = pd.DataFrame(
    {
        "class_id":
            list(
                range(
                    len(
                        controlled_glosses
                    )
                )
            ),

        "gloss":
            controlled_glosses
    }
)

controlled_mapping_df.to_csv(
    CONTROLLED_CLASS_MAP_CSV,
    index=False
)

with open(
    CONTROLLED_CLASS_MAP_JSON,
    "w"
) as f:

    json.dump(
        {
            "gloss_to_id":
                controlled_gloss_to_id,

            "id_to_gloss":
                {
                    str(key):
                        value

                    for key, value
                    in controlled_id_to_gloss.items()
                }
        },
        f,
        indent=4,
        ensure_ascii=False
    )


# ==========================================================================================
# 36. SUPPORT DISTRIBUTION
# ==========================================================================================

support_rows = []

for threshold in [
    1,
    2,
    3,
    4,
    5,
    6,
    8,
    10
]:

    classes = int(
        (
            full_class_support[
                "usable_videos"
            ]
            >=
            threshold
        ).sum()
    )

    videos = int(
        full_class_support.loc[
            full_class_support[
                "usable_videos"
            ]
            >=
            threshold,
            "usable_videos"
        ].sum()
    )

    support_rows.append(
        {
            "minimum_usable_videos":
                threshold,

            "classes":
                classes,

            "videos":
                videos
        }
    )

support_summary = pd.DataFrame(
    support_rows
)

support_summary.to_csv(
    SUPPORT_SUMMARY_CSV,
    index=False
)


# ==========================================================================================
# 37. VERIFY CONTROLLED SUPPORT
# ==========================================================================================

controlled_support_min = int(
    controlled_classes[
        "usable_videos"
    ].min()
)

controlled_support_max = int(
    controlled_classes[
        "usable_videos"
    ].max()
)

if (
    controlled_support_min
    <
    CONTROLLED_MIN_USABLE_VIDEOS
):

    raise RuntimeError(
        "Controlled population contains "
        "a class with support below 4."
    )


# ==========================================================================================
# 38. VERIFY EVERY MANIFEST FEATURE FILE
# ==========================================================================================

print("\n" + "=" * 125)
print("VERIFYING FROZEN FEATURE REFERENCES")
print("=" * 125)


def verify_feature_manifest(
    manifest,
    name
):

    missing = []

    bad_shape = []

    nonfinite = []

    for i, row in enumerate(
        manifest.itertuples(
            index=False
        ),
        start=1
    ):

        uid = str(
            row.uid_clean
        )

        path = (
            FEATURE_DIR
            /
            f"{uid}.npz"
        )

        if not path.exists():

            missing.append(
                uid
            )

            continue

        try:

            with np.load(
                path,
                allow_pickle=True
            ) as data:

                features = np.asarray(
                    data[
                        "features"
                    ],
                    dtype=np.float32
                )

            if features.shape != (
                EXPECTED_SEQUENCE_LENGTH,
                EXPECTED_FEATURE_DIM
            ):

                bad_shape.append(
                    {
                        "uid":
                            uid,

                        "shape":
                            str(
                                features.shape
                            )
                    }
                )

            elif not np.isfinite(
                features
            ).all():

                nonfinite.append(
                    uid
                )

        except Exception:

            bad_shape.append(
                {
                    "uid":
                        uid,

                    "shape":
                        "READ_ERROR"
                }
            )

        if (
            i % 1000 == 0
            or
            i == len(
                manifest
            )
        ):

            print(
                f"{name:20s}: "
                f"{i:4d}/{len(manifest)}"
            )

    return {
        "missing":
            missing,

        "bad_shape":
            bad_shape,

        "nonfinite":
            nonfinite
    }


controlled_verification = verify_feature_manifest(
    controlled_population,
    "CONTROLLED"
)

full_verification = verify_feature_manifest(
    full_population,
    "FULL-VOCAB"
)


# ==========================================================================================
# 39. VERIFY CLASS IDS
# ==========================================================================================

controlled_ids_expected = set(
    range(
        len(
            controlled_glosses
        )
    )
)

controlled_ids_actual = set(
    controlled_population[
        "class_id"
    ].unique()
)

full_ids_expected = set(
    range(
        len(
            full_glosses
        )
    )
)

full_ids_actual = set(
    full_population[
        "class_id"
    ].unique()
)

controlled_class_ids_valid = (
    controlled_ids_expected
    ==
    controlled_ids_actual
)

full_class_ids_valid = (
    full_ids_expected
    ==
    full_ids_actual
)


# ==========================================================================================
# 40. HASH / FINGERPRINT HELPERS
# ==========================================================================================

def sha256_text(
    text
):

    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def dataframe_fingerprint(
    dataframe,
    columns
):

    ordered = (
        dataframe[
            columns
        ]
        .copy()
        .sort_values(
            columns
        )
        .reset_index(
            drop=True
        )
    )

    text = ordered.to_csv(
        index=False,
        lineterminator="\n"
    )

    return sha256_text(
        text
    )


# ==========================================================================================
# 41. POPULATION FINGERPRINTS
# ==========================================================================================

controlled_population_hash = dataframe_fingerprint(
    controlled_population,
    [
        "uid_clean",
        "gloss",
        "class_id"
    ]
)

controlled_mapping_hash = dataframe_fingerprint(
    controlled_mapping_df,
    [
        "class_id",
        "gloss"
    ]
)

full_population_hash = dataframe_fingerprint(
    full_population,
    [
        "uid_clean",
        "gloss",
        "class_id"
    ]
)

full_mapping_hash = dataframe_fingerprint(
    full_mapping_df,
    [
        "class_id",
        "gloss"
    ]
)


# ==========================================================================================
# 42. METADATA AUDIT TABLE
# ==========================================================================================

metadata_audit_rows = [
    {
        "metric":
            "dataset_rows",

        "value":
            len(dataset)
    },

    {
        "metric":
            "valid_authoritative_labels",

        "value":
            int(
                dataset[
                    "valid_gloss"
                ].sum()
            )
    },

    {
        "metric":
            "invalid_authoritative_labels",

        "value":
            int(
                (
                    ~dataset[
                        "valid_gloss"
                    ]
                ).sum()
            )
    },

    {
        "metric":
            "authoritative_unique_glosses",

        "value":
            authoritative_unique_glosses
    },

    {
        "metric":
            "feature_derived_valid_glosses",

        "value":
            len(
                feature_gloss_set
            )
    },

    {
        "metric":
            "label_mismatch_rows",

        "value":
            len(
                label_mismatch_df
            )
    },

    {
        "metric":
            "dataset_only_glosses",

        "value":
            len(
                only_dataset
            )
    },

    {
        "metric":
            "feature_only_glosses",

        "value":
            len(
                only_feature
            )
    }
]

metadata_audit_df = pd.DataFrame(
    metadata_audit_rows
)

metadata_audit_df.to_csv(
    METADATA_AUDIT_CSV,
    index=False
)


# ==========================================================================================
# 43. POPULATION SUMMARY
# ==========================================================================================

full_usable_count = len(
    full_population
)

full_class_count = len(
    full_glosses
)

controlled_video_count = len(
    controlled_population
)

controlled_class_count = len(
    controlled_glosses
)

population_summary = {
    "stage":
        "L8_EXPERIMENTAL_POPULATION_FREEZE",

    "representation":
        {
            "sequence_length":
                EXPECTED_SEQUENCE_LENGTH,

            "feature_dim":
                EXPECTED_FEATURE_DIM
        },

    "quality_rule":
        {
            "minimum_original_detection_rate":
                MIN_ORIGINAL_DETECTION_RATE,

            "minimum_sampled_hand_frames":
                MIN_SAMPLED_HAND_FRAMES
        },

    "metadata":
        {
            "dataset_rows":
                len(dataset),

            "valid_label_rows":
                int(
                    dataset[
                        "valid_gloss"
                    ].sum()
                ),

            "invalid_label_rows":
                int(
                    (
                        ~dataset[
                            "valid_gloss"
                        ]
                    ).sum()
                ),

            "authoritative_unique_glosses":
                authoritative_unique_glosses,

            "feature_derived_valid_glosses":
                len(
                    feature_gloss_set
                ),

            "label_mismatch_rows":
                len(
                    label_mismatch_df
                )
        },

    "full_vocabulary_population":
        {
            "videos":
                full_usable_count,

            "classes":
                full_class_count,

            "population_sha256":
                full_population_hash,

            "mapping_sha256":
                full_mapping_hash
        },

    "controlled_population":
        {
            "minimum_usable_videos_per_class":
                CONTROLLED_MIN_USABLE_VIDEOS,

            "videos":
                controlled_video_count,

            "classes":
                controlled_class_count,

            "minimum_class_support":
                controlled_support_min,

            "maximum_class_support":
                controlled_support_max,

            "population_sha256":
                controlled_population_hash,

            "mapping_sha256":
                controlled_mapping_hash
        }
}

with open(
    POPULATION_SUMMARY_JSON,
    "w"
) as f:

    json.dump(
        population_summary,
        f,
        indent=4,
        ensure_ascii=False
    )


# ==========================================================================================
# 44. FREEZE MANIFEST
# ==========================================================================================

freeze_manifest = {
    "stage":
        "L8",

    "status":
        "FROZEN",

    "authoritative_label_source":
        str(
            DATASET_CSV
        ),

    "input_l7_video_audit":
        str(
            L7_VIDEO_AUDIT
        ),

    "input_feature_directory":
        str(
            FEATURE_DIR
        ),

    "controlled_population":
        {
            "manifest":
                str(
                    CONTROLLED_MANIFEST_CSV
                ),

            "class_mapping":
                str(
                    CONTROLLED_CLASS_MAP_JSON
                ),

            "minimum_usable_videos":
                CONTROLLED_MIN_USABLE_VIDEOS,

            "population_sha256":
                controlled_population_hash,

            "mapping_sha256":
                controlled_mapping_hash
        },

    "full_vocabulary_population":
        {
            "manifest":
                str(
                    FULL_MANIFEST_CSV
                ),

            "class_mapping":
                str(
                    FULL_CLASS_MAP_JSON
                ),

            "population_sha256":
                full_population_hash,

            "mapping_sha256":
                full_mapping_hash
        },

    "split_created":
        False,

    "next_stage":
        "L9_TRAIN_VALIDATION_TEST_SPLIT_FREEZE"
}

with open(
    FREEZE_MANIFEST_JSON,
    "w"
) as f:

    json.dump(
        freeze_manifest,
        f,
        indent=4,
        ensure_ascii=False
    )


# ==========================================================================================
# 45. FINAL INTEGRITY CONDITIONS
# ==========================================================================================

controlled_feature_integrity = (
    len(
        controlled_verification[
            "missing"
        ]
    )
    ==
    0
    and
    len(
        controlled_verification[
            "bad_shape"
        ]
    )
    ==
    0
    and
    len(
        controlled_verification[
            "nonfinite"
        ]
    )
    ==
    0
)

full_feature_integrity = (
    len(
        full_verification[
            "missing"
        ]
    )
    ==
    0
    and
    len(
        full_verification[
            "bad_shape"
        ]
    )
    ==
    0
    and
    len(
        full_verification[
            "nonfinite"
        ]
    )
    ==
    0
)

uid_integrity = (
    len(
        dataset_missing_l7
    )
    ==
    0
    and
    len(
        l7_extra
    )
    ==
    0
    and
    len(
        dataset_missing_features
    )
    ==
    0
    and
    len(
        dataset_missing_raw
    )
    ==
    0
)

controlled_support_valid = (
    controlled_support_min
    >=
    CONTROLLED_MIN_USABLE_VIDEOS
)

population_disjoint_exclusion_check = (
    len(
        set(
            controlled_population[
                "uid_clean"
            ]
        )
        &
        set(
            controlled_excluded[
                "uid_clean"
            ]
        )
    )
    ==
    0
)

full_disjoint_exclusion_check = (
    len(
        set(
            full_population[
                "uid_clean"
            ]
        )
        &
        set(
            full_excluded[
                "uid_clean"
            ]
        )
    )
    ==
    0
)


# ==========================================================================================
# 46. EXPECTED COUNTS FROM L7
#
# These are not used to construct the population.
# They are only sanity checks against the completed L7 audit.
# ==========================================================================================

EXPECTED_L7_QUALITY_USABLE = 7004
EXPECTED_CONTROLLED_CLASSES = 211
EXPECTED_CONTROLLED_VIDEOS = 1007


quality_usable_matches_l7 = (
    int(
        master[
            "l8_quality_usable"
        ].sum()
    )
    ==
    EXPECTED_L7_QUALITY_USABLE
)

controlled_classes_match_l7 = (
    controlled_class_count
    ==
    EXPECTED_CONTROLLED_CLASSES
)

controlled_videos_match_l7 = (
    controlled_video_count
    ==
    EXPECTED_CONTROLLED_VIDEOS
)


# If there is an invalid authoritative label among the 7004 quality-usable
# videos, full-vocab count may be <7004. That is correct.
#
# Therefore we DO NOT force full_usable_count == 7004.


# ==========================================================================================
# 47. CHECKPOINT
# ==========================================================================================

l8_pass = bool(
    bool(
        l7_checkpoint.get(
            "l7_pass",
            False
        )
    )
    and
    uid_integrity
    and
    usability_mismatch_count
    ==
    0
    and
    controlled_feature_integrity
    and
    full_feature_integrity
    and
    controlled_class_ids_valid
    and
    full_class_ids_valid
    and
    controlled_support_valid
    and
    population_disjoint_exclusion_check
    and
    full_disjoint_exclusion_check
    and
    quality_usable_matches_l7
)


checkpoint = {
    "l7_pass_verified":
        bool(
            l7_checkpoint.get(
                "l7_pass",
                False
            )
        ),

    "uid_integrity":
        uid_integrity,

    "l7_usability_reproduced":
        usability_mismatch_count
        ==
        0,

    "controlled_feature_integrity":
        controlled_feature_integrity,

    "full_feature_integrity":
        full_feature_integrity,

    "controlled_class_ids_valid":
        controlled_class_ids_valid,

    "full_class_ids_valid":
        full_class_ids_valid,

    "controlled_support_valid":
        controlled_support_valid,

    "controlled_population_exclusion_disjoint":
        population_disjoint_exclusion_check,

    "full_population_exclusion_disjoint":
        full_disjoint_exclusion_check,

    "quality_usable_matches_l7":
        quality_usable_matches_l7,

    "controlled_classes_match_l7_reference":
        controlled_classes_match_l7,

    "controlled_videos_match_l7_reference":
        controlled_videos_match_l7,

    "l8_pass":
        l8_pass
}

with open(
    CHECKPOINT_JSON,
    "w"
) as f:

    json.dump(
        checkpoint,
        f,
        indent=4
    )


# ==========================================================================================
# 48. FINAL REPORT
# ==========================================================================================

print("\n" + "=" * 125)
print("L8 FINAL REPORT")
print("=" * 125)


# ------------------------------------------------------------------------------------------
# Metadata
# ------------------------------------------------------------------------------------------

print("\nMETADATA")
print("-" * 125)

print(
    "Dataset rows                       :",
    len(dataset)
)

print(
    "Valid authoritative label rows     :",
    int(
        dataset[
            "valid_gloss"
        ].sum()
    )
)

print(
    "Invalid/missing label rows         :",
    int(
        (
            ~dataset[
                "valid_gloss"
            ]
        ).sum()
    )
)

print(
    "Authoritative valid glosses        :",
    authoritative_unique_glosses
)

print(
    "Feature-derived valid glosses      :",
    len(
        feature_gloss_set
    )
)

print(
    "Dataset/feature label mismatches   :",
    len(
        label_mismatch_df
    )
)

print(
    "Dataset-only glosses               :",
    len(
        only_dataset
    )
)

print(
    "Feature-only glosses               :",
    len(
        only_feature
    )
)


# ------------------------------------------------------------------------------------------
# Usability
# ------------------------------------------------------------------------------------------

print("\nQUALITY ELIGIBILITY")
print("-" * 125)

print(
    "L7/L8 quality-usable videos        :",
    int(
        master[
            "l8_quality_usable"
        ].sum()
    )
)

print(
    "Final usable + valid-label videos  :",
    full_usable_count
)

print(
    "Excluded from full vocabulary      :",
    len(
        full_excluded
    )
)


# ------------------------------------------------------------------------------------------
# Full vocabulary
# ------------------------------------------------------------------------------------------

print("\nFULL-VOCABULARY POPULATION")
print("-" * 125)

print(
    "Videos                             :",
    full_usable_count
)

print(
    "Classes                            :",
    full_class_count
)

print(
    "Population SHA256                  :",
    full_population_hash
)

print(
    "Mapping SHA256                     :",
    full_mapping_hash
)


# ------------------------------------------------------------------------------------------
# Controlled
# ------------------------------------------------------------------------------------------

print("\nCONTROLLED POPULATION")
print("-" * 125)

print(
    "Minimum usable videos/class        :",
    CONTROLLED_MIN_USABLE_VIDEOS
)

print(
    "Classes                            :",
    controlled_class_count
)

print(
    "Videos                             :",
    controlled_video_count
)

print(
    "Minimum actual class support       :",
    controlled_support_min
)

print(
    "Maximum actual class support       :",
    controlled_support_max
)

print(
    "Population SHA256                  :",
    controlled_population_hash
)

print(
    "Mapping SHA256                     :",
    controlled_mapping_hash
)


# ------------------------------------------------------------------------------------------
# Support distribution
# ------------------------------------------------------------------------------------------

print("\nUSABLE SUPPORT DISTRIBUTION")
print("-" * 125)

print(
    support_summary.to_string(
        index=False
    )
)


# ------------------------------------------------------------------------------------------
# Verification
# ------------------------------------------------------------------------------------------

print("\nINTEGRITY VERIFICATION")
print("-" * 125)

print(
    "UID integrity                      :",
    uid_integrity
)

print(
    "L7 usability reproduced            :",
    usability_mismatch_count
    ==
    0
)

print(
    "Controlled feature integrity       :",
    controlled_feature_integrity
)

print(
    "Full-vocab feature integrity       :",
    full_feature_integrity
)

print(
    "Controlled class IDs valid         :",
    controlled_class_ids_valid
)

print(
    "Full-vocab class IDs valid         :",
    full_class_ids_valid
)

print(
    "Controlled support valid           :",
    controlled_support_valid
)

print(
    "L7 quality count reference match   :",
    quality_usable_matches_l7
)

print(
    "L7 controlled class reference      :",
    controlled_classes_match_l7
)

print(
    "L7 controlled video reference      :",
    controlled_videos_match_l7
)


# ==========================================================================================
# 49. CONTROLLED SUPPORT DISTRIBUTION
# ==========================================================================================

print("\n" + "=" * 125)
print("CONTROLLED POPULATION — CLASS SUPPORT DISTRIBUTION")
print("=" * 125)

controlled_support_distribution = (
    controlled_classes[
        "usable_videos"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "videos_per_class"
    )
    .reset_index(
        name="classes"
    )
)

print(
    controlled_support_distribution.to_string(
        index=False
    )
)


# ==========================================================================================
# 50. TOP CONTROLLED CLASSES
# ==========================================================================================

print("\n" + "=" * 125)
print("TOP 20 CONTROLLED CLASSES BY SUPPORT")
print("=" * 125)

top_controlled = (
    controlled_classes
    .sort_values(
        [
            "usable_videos",
            "gloss"
        ],
        ascending=[
            False,
            True
        ]
    )
    .head(20)
)

print(
    top_controlled[
        [
            "class_id",
            "gloss",
            "usable_videos",
            "mean_detection_rate",
            "max_abs_feature"
        ]
    ].to_string(
        index=False
    )
)


# ==========================================================================================
# 51. LOWEST COVERAGE CONTROLLED CLASSES
# ==========================================================================================

print("\n" + "=" * 125)
print("20 LOWEST-DETECTION CONTROLLED CLASSES")
print("=" * 125)

lowest_detection_controlled = (
    controlled_classes
    .sort_values(
        [
            "mean_detection_rate",
            "gloss"
        ]
    )
    .head(20)
)

print(
    lowest_detection_controlled[
        [
            "class_id",
            "gloss",
            "usable_videos",
            "mean_detection_rate",
            "median_detection_rate",
            "max_abs_feature"
        ]
    ].to_string(
        index=False
    )
)


# ==========================================================================================
# 52. L8 CHECKPOINT
# ==========================================================================================

print("\n" + "=" * 125)
print("L8 CHECKPOINT")
print("=" * 125)

if l8_pass:

    print()
    print(
        "[PASS] L8 EXPERIMENTAL POPULATIONS FROZEN."
    )

    print()
    print(
        "CONTROLLED population:"
    )

    print(
        f"    {controlled_class_count} classes"
    )

    print(
        f"    {controlled_video_count} videos"
    )

    print(
        f"    >= {CONTROLLED_MIN_USABLE_VIDEOS} "
        f"usable videos per class"
    )

    print()
    print(
        "FULL-VOCABULARY population:"
    )

    print(
        f"    {full_class_count} classes"
    )

    print(
        f"    {full_usable_count} videos"
    )

    print()
    print(
        "No train/validation/test split "
        "has been created yet."
    )

    print()
    print(
        "NEXT STAGE: L9 — "
        "TRAIN / VALIDATION / TEST SPLIT FREEZE"
    )

else:

    print()
    print(
        "[REVIEW] L8 FREEZE DID NOT PASS "
        "ALL INTEGRITY CONDITIONS."
    )

    print()
    print(
        "Do NOT proceed to L9 until the "
        "failed checkpoint condition is understood."
    )


# ==========================================================================================
# 53. OUTPUT FILES
# ==========================================================================================

print("\n" + "=" * 125)
print("L8 OUTPUT FILES")
print("=" * 125)

print("\nMASTER MANIFEST")
print(MASTER_MANIFEST_CSV)

print("\nCONTROLLED POPULATION")
print(CONTROLLED_MANIFEST_CSV)

print("\nCONTROLLED CLASSES")
print(CONTROLLED_CLASSES_CSV)

print("\nCONTROLLED CLASS MAPPING")
print(CONTROLLED_CLASS_MAP_JSON)

print("\nCONTROLLED EXCLUSIONS")
print(CONTROLLED_EXCLUDED_CSV)

print("\nFULL-VOCABULARY POPULATION")
print(FULL_MANIFEST_CSV)

print("\nFULL-VOCABULARY CLASSES")
print(FULL_CLASSES_CSV)

print("\nFULL-VOCABULARY CLASS MAPPING")
print(FULL_CLASS_MAP_JSON)

print("\nFULL-VOCABULARY EXCLUSIONS")
print(FULL_EXCLUDED_CSV)

print("\nMETADATA AUDIT")
print(METADATA_AUDIT_CSV)

print("\nLABEL MISMATCH AUDIT")
print(LABEL_MISMATCH_CSV)

print("\nGLOSS DISCREPANCY AUDIT")
print(GLOSS_DISCREPANCY_CSV)

print("\nPOPULATION SUMMARY")
print(POPULATION_SUMMARY_JSON)

print("\nFREEZE MANIFEST")
print(FREEZE_MANIFEST_JSON)

print("\nCHECKPOINT")
print(CHECKPOINT_JSON)

print("\n" + "=" * 125)

CISLR LANDMARK MODEL — L8
EXPERIMENTAL POPULATION FREEZE

Dataset metadata : /home/dipshikha/Music/cao/dataset.csv
L7 audit         : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l7_full_dataset_audit/reports/l7_video_level_audit.csv
L6 features      : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l6_full_extraction/features_32x254
L8 output        : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l8_population_freeze

CONTROLLED POPULATION RULE
-----------------------------------------------------------------------------------------------------------------------------
Minimum usable videos per gloss : 4
Target split capability        : >=2 train + >=1 validation + >=1 test

FULL-VOCABULARY POPULATION RULE
-----------------------------------------------------------------------------------------------------------------------------
Every provisionally usable video with a valid authoritative dataset.csv gloss.

IMPORTANT
---

In [1]:
# ==========================================================================================
# CISLR LANDMARK MODEL — L9
# CONTROLLED TRAIN / VALIDATION / TEST SPLIT FREEZE
# ==========================================================================================
#
# INPUT (FROZEN L8):
#
#   controlled_population/l8_controlled_manifest.csv
#   class_mappings/l8_controlled_class_mapping.json
#   reports/l8_freeze_manifest.json
#   reports/l8_checkpoint.json
#
# PURPOSE:
#
#   1. Verify that L8 is genuinely frozen and passed.
#   2. Load the exact 211-class / 1007-video controlled population.
#   3. Preserve the L8 class IDs exactly.
#   4. Deterministically split EACH CLASS into:
#
#          TRAIN      = all remaining samples
#          VALIDATION = exactly 1 sample
#          TEST       = exactly 1 sample
#
#      Therefore:
#
#          support 4  -> 2 train / 1 val / 1 test
#          support 5  -> 3 train / 1 val / 1 test
#          support N  -> N-2 train / 1 val / 1 test
#
#   5. Guarantee every controlled class occurs in all three partitions.
#   6. Guarantee zero UID overlap between partitions.
#   7. Verify every referenced feature file is valid 32 x 254 and finite.
#   8. Save exact UID manifests and class-level split statistics.
#   9. Save deterministic SHA256 fingerprints.
#  10. Freeze the split for ALL subsequent controlled experiments.
#
# IMPORTANT:
#
#   - L9 DOES NOT TRAIN A MODEL.
#   - L9 DOES NOT NORMALIZE FEATURES.
#   - L9 DOES NOT FIT SCALERS.
#   - L9 DOES NOT MODIFY L6/L7/L8.
#   - L10 GRU AND L11 BiLSTM MUST USE THIS EXACT SPLIT.
#   - Any future preprocessing statistics must be fitted on TRAIN ONLY.
#
# NEXT:
#
#   L10 = GRU BASELINE
#
# ==========================================================================================


# ==========================================================================================
# 0. IMPORTS
# ==========================================================================================

from pathlib import Path
import hashlib
import json
import random
import warnings

import numpy as np
import pandas as pd


warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)

np.set_printoptions(
    precision=6,
    suppress=True
)


# ==========================================================================================
# 1. CONFIGURATION
# ==========================================================================================

SEED = 42

EXPECTED_CLASSES = 211
EXPECTED_VIDEOS = 1007

EXPECTED_SEQUENCE_LENGTH = 32
EXPECTED_FEATURE_DIM = 254

MINIMUM_CLASS_SUPPORT = 4

VALIDATION_PER_CLASS = 1
TEST_PER_CLASS = 1

random.seed(SEED)
np.random.seed(SEED)


# ==========================================================================================
# 2. PROJECT PATHS
# ==========================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

LANDMARK_ROOT = (
    PROJECT_ROOT
    / "CISLR_LANDMARK_MODEL"
)

L6_ROOT = (
    LANDMARK_ROOT
    / "audit"
    / "l6_full_extraction"
)

FEATURE_DIR = (
    L6_ROOT
    / "features_32x254"
)

L8_ROOT = (
    LANDMARK_ROOT
    / "audit"
    / "l8_population_freeze"
)

L8_CONTROLLED_MANIFEST = (
    L8_ROOT
    / "controlled_population"
    / "l8_controlled_manifest.csv"
)

L8_CONTROLLED_CLASSES = (
    L8_ROOT
    / "controlled_population"
    / "l8_controlled_classes.csv"
)

L8_CLASS_MAPPING_JSON = (
    L8_ROOT
    / "class_mappings"
    / "l8_controlled_class_mapping.json"
)

L8_FREEZE_MANIFEST_JSON = (
    L8_ROOT
    / "reports"
    / "l8_freeze_manifest.json"
)

L8_CHECKPOINT_JSON = (
    L8_ROOT
    / "reports"
    / "l8_checkpoint.json"
)


# ==========================================================================================
# 3. L9 OUTPUT DIRECTORIES
# ==========================================================================================

L9_ROOT = (
    LANDMARK_ROOT
    / "audit"
    / "l9_split_freeze"
)

REPORT_DIR = (
    L9_ROOT
    / "reports"
)

SPLIT_DIR = (
    L9_ROOT
    / "splits"
)

AUDIT_DIR = (
    L9_ROOT
    / "audit"
)

MAPPING_DIR = (
    L9_ROOT
    / "class_mapping"
)

for directory in [
    L9_ROOT,
    REPORT_DIR,
    SPLIT_DIR,
    AUDIT_DIR,
    MAPPING_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ==========================================================================================
# 4. OUTPUT FILE PATHS
# ==========================================================================================

TRAIN_MANIFEST_CSV = (
    SPLIT_DIR
    / "l9_train_manifest.csv"
)

VAL_MANIFEST_CSV = (
    SPLIT_DIR
    / "l9_validation_manifest.csv"
)

TEST_MANIFEST_CSV = (
    SPLIT_DIR
    / "l9_test_manifest.csv"
)

MASTER_SPLIT_CSV = (
    SPLIT_DIR
    / "l9_master_split.csv"
)

TRAIN_UIDS_TXT = (
    SPLIT_DIR
    / "l9_train_uids.txt"
)

VAL_UIDS_TXT = (
    SPLIT_DIR
    / "l9_validation_uids.txt"
)

TEST_UIDS_TXT = (
    SPLIT_DIR
    / "l9_test_uids.txt"
)

CLASS_SPLIT_AUDIT_CSV = (
    AUDIT_DIR
    / "l9_class_split_audit.csv"
)

FEATURE_AUDIT_CSV = (
    AUDIT_DIR
    / "l9_feature_integrity_audit.csv"
)

SPLIT_DISTRIBUTION_CSV = (
    AUDIT_DIR
    / "l9_split_distribution.csv"
)

CLASS_MAPPING_CSV = (
    MAPPING_DIR
    / "l9_controlled_class_mapping.csv"
)

CLASS_MAPPING_JSON = (
    MAPPING_DIR
    / "l9_controlled_class_mapping.json"
)

SUMMARY_JSON = (
    REPORT_DIR
    / "l9_summary.json"
)

FREEZE_MANIFEST_JSON = (
    REPORT_DIR
    / "l9_freeze_manifest.json"
)

CHECKPOINT_JSON = (
    REPORT_DIR
    / "l9_checkpoint.json"
)


# ==========================================================================================
# 5. HEADER
# ==========================================================================================

print("=" * 125)
print("CISLR LANDMARK MODEL — L9")
print("CONTROLLED TRAIN / VALIDATION / TEST SPLIT FREEZE")
print("=" * 125)

print()
print("Random seed                  :", SEED)
print("Expected controlled classes  :", EXPECTED_CLASSES)
print("Expected controlled videos   :", EXPECTED_VIDEOS)
print("Minimum class support        :", MINIMUM_CLASS_SUPPORT)
print("Validation samples/class     :", VALIDATION_PER_CLASS)
print("Test samples/class           :", TEST_PER_CLASS)

print()
print("SPLIT RULE")
print("-" * 125)
print("For EACH class:")
print("    1 randomly selected deterministic sample -> VALIDATION")
print("    1 different deterministic sample         -> TEST")
print("    all remaining samples                    -> TRAIN")

print()
print("Examples:")
print("    4 samples  -> 2 train / 1 validation / 1 test")
print("    5 samples  -> 3 train / 1 validation / 1 test")
print("    8 samples  -> 6 train / 1 validation / 1 test")
print("   13 samples  ->11 train / 1 validation / 1 test")

print()
print("IMPORTANT")
print("-" * 125)
print("No feature normalization is fitted in L9.")
print("No model is trained in L9.")
print("L10 and L11 must use this exact frozen split.")

print("=" * 125)


# ==========================================================================================
# 6. VERIFY REQUIRED INPUTS
# ==========================================================================================

required_inputs = [
    FEATURE_DIR,
    L8_CONTROLLED_MANIFEST,
    L8_CONTROLLED_CLASSES,
    L8_CLASS_MAPPING_JSON,
    L8_FREEZE_MANIFEST_JSON,
    L8_CHECKPOINT_JSON
]

for path in required_inputs:

    if not path.exists():

        raise FileNotFoundError(
            f"Required L9 input missing:\n{path}"
        )

print("\n[PASS] All required L8/L6 inputs exist.")


# ==========================================================================================
# 7. LOAD AND VERIFY L8 CHECKPOINT
# ==========================================================================================

with open(
    L8_CHECKPOINT_JSON,
    "r"
) as f:

    l8_checkpoint = json.load(f)

print("\n" + "=" * 125)
print("L8 CHECKPOINT VERIFICATION")
print("=" * 125)

for key, value in l8_checkpoint.items():

    print(
        f"{key:45s}: {value}"
    )

if not bool(
    l8_checkpoint.get(
        "l8_pass",
        False
    )
):

    raise RuntimeError(
        "L8 checkpoint is not PASS. "
        "L9 split freeze aborted."
    )

print("\n[PASS] L8 checkpoint verified.")


# ==========================================================================================
# 8. LOAD L8 FREEZE MANIFEST
# ==========================================================================================

with open(
    L8_FREEZE_MANIFEST_JSON,
    "r"
) as f:

    l8_freeze = json.load(f)

if str(
    l8_freeze.get(
        "status",
        ""
    )
).upper() != "FROZEN":

    raise RuntimeError(
        "L8 freeze manifest is not marked FROZEN."
    )

print(
    "[PASS] L8 freeze status verified."
)


# ==========================================================================================
# 9. LOAD CONTROLLED POPULATION
# ==========================================================================================

controlled = pd.read_csv(
    L8_CONTROLLED_MANIFEST,
    dtype={
        "uid_clean": "string",
        "gloss": "string"
    }
)

print("\n" + "=" * 125)
print("L8 CONTROLLED POPULATION")
print("=" * 125)

print(
    "Rows    :",
    len(controlled)
)

print(
    "Classes :",
    controlled["gloss"].nunique()
)


# ==========================================================================================
# 10. REQUIRED COLUMN CHECK
# ==========================================================================================

required_columns = {
    "uid_clean",
    "gloss",
    "class_id"
}

missing_columns = (
    required_columns
    -
    set(
        controlled.columns
    )
)

if missing_columns:

    raise RuntimeError(
        "Controlled manifest missing required columns: "
        f"{sorted(missing_columns)}"
    )


# ==========================================================================================
# 11. CLEAN BASIC TYPES
# ==========================================================================================

controlled[
    "uid_clean"
] = (
    controlled[
        "uid_clean"
    ]
    .astype("string")
    .str.strip()
)

controlled[
    "gloss"
] = (
    controlled[
        "gloss"
    ]
    .astype("string")
    .str.strip()
)

controlled[
    "class_id"
] = (
    controlled[
        "class_id"
    ]
    .astype(int)
)


# ==========================================================================================
# 12. BASIC POPULATION INTEGRITY
# ==========================================================================================

duplicate_uid_count = int(
    controlled[
        "uid_clean"
    ].duplicated().sum()
)

missing_uid_count = int(
    controlled[
        "uid_clean"
    ].isna().sum()
)

missing_gloss_count = int(
    controlled[
        "gloss"
    ].isna().sum()
)

num_videos = len(
    controlled
)

num_classes = int(
    controlled[
        "gloss"
    ].nunique()
)

print()
print(
    "Duplicate UIDs :",
    duplicate_uid_count
)

print(
    "Missing UIDs   :",
    missing_uid_count
)

print(
    "Missing gloss  :",
    missing_gloss_count
)

if duplicate_uid_count != 0:

    raise RuntimeError(
        "Duplicate UIDs exist in controlled population."
    )

if missing_uid_count != 0:

    raise RuntimeError(
        "Missing UIDs exist in controlled population."
    )

if missing_gloss_count != 0:

    raise RuntimeError(
        "Missing glosses exist in controlled population."
    )

if num_videos != EXPECTED_VIDEOS:

    raise RuntimeError(
        f"Expected {EXPECTED_VIDEOS} controlled videos, "
        f"found {num_videos}."
    )

if num_classes != EXPECTED_CLASSES:

    raise RuntimeError(
        f"Expected {EXPECTED_CLASSES} controlled classes, "
        f"found {num_classes}."
    )

print(
    "\n[PASS] Controlled population count verified."
)


# ==========================================================================================
# 13. VERIFY CLASS_ID ↔ GLOSS IS ONE-TO-ONE
# ==========================================================================================

gloss_to_ids = (
    controlled
    .groupby(
        "gloss"
    )[
        "class_id"
    ]
    .nunique()
)

id_to_glosses = (
    controlled
    .groupby(
        "class_id"
    )[
        "gloss"
    ]
    .nunique()
)

bad_gloss_mapping = gloss_to_ids[
    gloss_to_ids != 1
]

bad_id_mapping = id_to_glosses[
    id_to_glosses != 1
]

if len(
    bad_gloss_mapping
) > 0:

    raise RuntimeError(
        "At least one gloss maps to multiple class IDs."
    )

if len(
    bad_id_mapping
) > 0:

    raise RuntimeError(
        "At least one class ID maps to multiple glosses."
    )

expected_class_ids = set(
    range(
        EXPECTED_CLASSES
    )
)

actual_class_ids = set(
    controlled[
        "class_id"
    ].unique()
)

if actual_class_ids != expected_class_ids:

    raise RuntimeError(
        "Class IDs are not contiguous 0..210."
    )

print(
    "[PASS] class_id ↔ gloss mapping is one-to-one."
)


# ==========================================================================================
# 14. LOAD L8 CLASS MAPPING
# ==========================================================================================

with open(
    L8_CLASS_MAPPING_JSON,
    "r"
) as f:

    l8_mapping = json.load(f)

l8_gloss_to_id = {
    str(gloss):
        int(class_id)

    for gloss, class_id
    in l8_mapping[
        "gloss_to_id"
    ].items()
}

l8_id_to_gloss = {
    int(class_id):
        str(gloss)

    for class_id, gloss
    in l8_mapping[
        "id_to_gloss"
    ].items()
}


# ==========================================================================================
# 15. VERIFY MANIFEST AGAINST L8 MAPPING
# ==========================================================================================

mapping_mismatches = []

for row in controlled.itertuples(
    index=False
):

    gloss = str(
        row.gloss
    )

    class_id = int(
        row.class_id
    )

    expected_id = l8_gloss_to_id.get(
        gloss,
        None
    )

    if expected_id != class_id:

        mapping_mismatches.append(
            {
                "uid":
                    str(
                        row.uid_clean
                    ),

                "gloss":
                    gloss,

                "manifest_class_id":
                    class_id,

                "mapping_class_id":
                    expected_id
            }
        )

if len(
    mapping_mismatches
) > 0:

    mismatch_df = pd.DataFrame(
        mapping_mismatches
    )

    mismatch_path = (
        AUDIT_DIR
        / "l9_class_mapping_mismatch.csv"
    )

    mismatch_df.to_csv(
        mismatch_path,
        index=False
    )

    raise RuntimeError(
        "Controlled manifest does not match "
        "the frozen L8 class mapping."
    )

print(
    "[PASS] Controlled manifest matches frozen L8 mapping."
)


# ==========================================================================================
# 16. COPY / FREEZE CLASS MAPPING INTO L9
# ==========================================================================================

class_mapping_df = (
    controlled[
        [
            "class_id",
            "gloss"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "class_id"
    )
    .reset_index(
        drop=True
    )
)

class_mapping_df.to_csv(
    CLASS_MAPPING_CSV,
    index=False
)

with open(
    CLASS_MAPPING_JSON,
    "w"
) as f:

    json.dump(
        {
            "gloss_to_id":
                {
                    row.gloss:
                        int(
                            row.class_id
                        )

                    for row
                    in class_mapping_df.itertuples(
                        index=False
                    )
                },

            "id_to_gloss":
                {
                    str(
                        int(
                            row.class_id
                        )
                    ):
                        row.gloss

                    for row
                    in class_mapping_df.itertuples(
                        index=False
                    )
                }
        },
        f,
        indent=4,
        ensure_ascii=False
    )

print(
    "[SAVED]",
    CLASS_MAPPING_JSON
)


# ==========================================================================================
# 17. VERIFY CLASS SUPPORT BEFORE SPLITTING
# ==========================================================================================

support = (
    controlled
    .groupby(
        [
            "class_id",
            "gloss"
        ]
    )
    .size()
    .reset_index(
        name="total"
    )
    .sort_values(
        "class_id"
    )
    .reset_index(
        drop=True
    )
)

minimum_support = int(
    support[
        "total"
    ].min()
)

maximum_support = int(
    support[
        "total"
    ].max()
)

print("\n" + "=" * 125)
print("CLASS SUPPORT BEFORE SPLIT")
print("=" * 125)

print(
    "Minimum support:",
    minimum_support
)

print(
    "Maximum support:",
    maximum_support
)

print()
print(
    support[
        "total"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "videos_per_class"
    )
    .reset_index(
        name="classes"
    )
    .to_string(
        index=False
    )
)

if minimum_support < MINIMUM_CLASS_SUPPORT:

    raise RuntimeError(
        "Controlled population contains a class "
        "with fewer than four videos."
    )


# ==========================================================================================
# 18. DETERMINISTIC PER-CLASS SPLIT
# ==========================================================================================
#
# Important design:
#
# We do NOT call one global random split.
#
# Each class receives its own deterministic RNG derived from:
#
#       global seed + class_id
#
# This has two advantages:
#
#   1. Every class is guaranteed to occur in every partition.
#   2. Changing row ordering in the CSV does not alter the split.
#
# Before permutation, UIDs are sorted.
#
# ==========================================================================================

split_rows = []


for class_row in support.itertuples(
    index=False
):

    class_id = int(
        class_row.class_id
    )

    gloss = str(
        class_row.gloss
    )

    class_df = (
        controlled[
            controlled[
                "class_id"
            ]
            ==
            class_id
        ]
        .copy()
        .sort_values(
            "uid_clean"
        )
        .reset_index(
            drop=True
        )
    )

    n = len(
        class_df
    )

    if n < 4:

        raise RuntimeError(
            f"Class {class_id} ({gloss}) "
            f"has only {n} videos."
        )


    # --------------------------------------------------------------------------------------
    # Independent deterministic generator for this class.
    # --------------------------------------------------------------------------------------

    class_rng = np.random.default_rng(
        SEED
        +
        class_id
    )

    permutation = class_rng.permutation(
        n
    )


    # --------------------------------------------------------------------------------------
    # Exactly one validation and one test example.
    #
    # We deliberately choose validation and test first,
    # then leave all remaining examples for training.
    # --------------------------------------------------------------------------------------

    val_position = int(
        permutation[
            0
        ]
    )

    test_position = int(
        permutation[
            1
        ]
    )

    train_positions = [
        int(x)
        for x in permutation[
            2:
        ]
    ]


    # --------------------------------------------------------------------------------------
    # Validation
    # --------------------------------------------------------------------------------------

    val_row = class_df.iloc[
        val_position
    ].copy()

    val_dict = val_row.to_dict()

    val_dict[
        "split"
    ] = "validation"

    val_dict[
        "split_seed"
    ] = SEED

    split_rows.append(
        val_dict
    )


    # --------------------------------------------------------------------------------------
    # Test
    # --------------------------------------------------------------------------------------

    test_row = class_df.iloc[
        test_position
    ].copy()

    test_dict = test_row.to_dict()

    test_dict[
        "split"
    ] = "test"

    test_dict[
        "split_seed"
    ] = SEED

    split_rows.append(
        test_dict
    )


    # --------------------------------------------------------------------------------------
    # Train
    # --------------------------------------------------------------------------------------

    for position in train_positions:

        train_row = class_df.iloc[
            position
        ].copy()

        train_dict = train_row.to_dict()

        train_dict[
            "split"
        ] = "train"

        train_dict[
            "split_seed"
        ] = SEED

        split_rows.append(
            train_dict
        )


# ==========================================================================================
# 19. BUILD MASTER SPLIT DATAFRAME
# ==========================================================================================

master_split = pd.DataFrame(
    split_rows
)

split_order = pd.CategoricalDtype(
    categories=[
        "train",
        "validation",
        "test"
    ],
    ordered=True
)

master_split[
    "split"
] = master_split[
    "split"
].astype(
    split_order
)

master_split = (
    master_split
    .sort_values(
        [
            "split",
            "class_id",
            "uid_clean"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ==========================================================================================
# 20. CREATE INDIVIDUAL SPLITS
# ==========================================================================================

train_df = (
    master_split[
        master_split[
            "split"
        ]
        ==
        "train"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

val_df = (
    master_split[
        master_split[
            "split"
        ]
        ==
        "validation"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

test_df = (
    master_split[
        master_split[
            "split"
        ]
        ==
        "test"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==========================================================================================
# 21. EXPECTED SPLIT COUNTS
# ==========================================================================================
#
# Since exactly 1 validation + 1 test are taken per class:
#
# validation = 211
# test       = 211
# train      = 1007 - 422 = 585
#
# ==========================================================================================

EXPECTED_VAL = EXPECTED_CLASSES
EXPECTED_TEST = EXPECTED_CLASSES

EXPECTED_TRAIN = (
    EXPECTED_VIDEOS
    -
    EXPECTED_VAL
    -
    EXPECTED_TEST
)

print("\n" + "=" * 125)
print("SPLIT COUNTS")
print("=" * 125)

print(
    "Train      :",
    len(train_df),
    f"(expected {EXPECTED_TRAIN})"
)

print(
    "Validation :",
    len(val_df),
    f"(expected {EXPECTED_VAL})"
)

print(
    "Test       :",
    len(test_df),
    f"(expected {EXPECTED_TEST})"
)

print(
    "Total      :",
    len(master_split),
    f"(expected {EXPECTED_VIDEOS})"
)


# ==========================================================================================
# 22. VERIFY EXACT COUNTS
# ==========================================================================================

if len(
    train_df
) != EXPECTED_TRAIN:

    raise RuntimeError(
        "Unexpected train count."
    )

if len(
    val_df
) != EXPECTED_VAL:

    raise RuntimeError(
        "Unexpected validation count."
    )

if len(
    test_df
) != EXPECTED_TEST:

    raise RuntimeError(
        "Unexpected test count."
    )

if len(
    master_split
) != EXPECTED_VIDEOS:

    raise RuntimeError(
        "Unexpected total split count."
    )

print(
    "\n[PASS] Split counts are exactly as expected."
)


# ==========================================================================================
# 23. UID SETS
# ==========================================================================================

train_uids = set(
    train_df[
        "uid_clean"
    ].astype(str)
)

val_uids = set(
    val_df[
        "uid_clean"
    ].astype(str)
)

test_uids = set(
    test_df[
        "uid_clean"
    ].astype(str)
)

source_uids = set(
    controlled[
        "uid_clean"
    ].astype(str)
)

split_uids = (
    train_uids
    |
    val_uids
    |
    test_uids
)


# ==========================================================================================
# 24. ZERO-OVERLAP VERIFICATION
# ==========================================================================================

train_val_overlap = (
    train_uids
    &
    val_uids
)

train_test_overlap = (
    train_uids
    &
    test_uids
)

val_test_overlap = (
    val_uids
    &
    test_uids
)

print("\n" + "=" * 125)
print("UID LEAKAGE CHECK")
print("=" * 125)

print(
    "Train ∩ Validation :",
    len(
        train_val_overlap
    )
)

print(
    "Train ∩ Test       :",
    len(
        train_test_overlap
    )
)

print(
    "Validation ∩ Test  :",
    len(
        val_test_overlap
    )
)

if (
    train_val_overlap
    or
    train_test_overlap
    or
    val_test_overlap
):

    raise RuntimeError(
        "UID leakage detected between splits."
    )

print(
    "\n[PASS] No UID overlap between partitions."
)


# ==========================================================================================
# 25. COMPLETENESS VERIFICATION
# ==========================================================================================

missing_from_split = (
    source_uids
    -
    split_uids
)

unexpected_in_split = (
    split_uids
    -
    source_uids
)

print("\nPOPULATION COMPLETENESS")
print("-" * 125)

print(
    "Source UIDs not assigned :",
    len(
        missing_from_split
    )
)

print(
    "Unexpected split UIDs    :",
    len(
        unexpected_in_split
    )
)

if (
    missing_from_split
    or
    unexpected_in_split
):

    raise RuntimeError(
        "Split UID set does not exactly reproduce "
        "the L8 controlled population."
    )

print(
    "[PASS] Split exactly covers the frozen L8 population."
)


# ==========================================================================================
# 26. CLASS PRESENCE VERIFICATION
# ==========================================================================================

train_classes = set(
    train_df[
        "class_id"
    ].astype(int)
)

val_classes = set(
    val_df[
        "class_id"
    ].astype(int)
)

test_classes = set(
    test_df[
        "class_id"
    ].astype(int)
)

print("\n" + "=" * 125)
print("CLASS COVERAGE")
print("=" * 125)

print(
    "Train classes      :",
    len(
        train_classes
    )
)

print(
    "Validation classes :",
    len(
        val_classes
    )
)

print(
    "Test classes       :",
    len(
        test_classes
    )
)

all_classes_in_train = (
    train_classes
    ==
    expected_class_ids
)

all_classes_in_val = (
    val_classes
    ==
    expected_class_ids
)

all_classes_in_test = (
    test_classes
    ==
    expected_class_ids
)

if not (
    all_classes_in_train
    and
    all_classes_in_val
    and
    all_classes_in_test
):

    raise RuntimeError(
        "Not all 211 classes are represented "
        "in every split."
    )

print(
    "\n[PASS] All 211 classes occur in train, validation and test."
)


# ==========================================================================================
# 27. CLASS-LEVEL SPLIT AUDIT
# ==========================================================================================

class_audit_rows = []

for row in support.itertuples(
    index=False
):

    class_id = int(
        row.class_id
    )

    gloss = str(
        row.gloss
    )

    total = int(
        row.total
    )

    train_count = int(
        (
            train_df[
                "class_id"
            ]
            ==
            class_id
        ).sum()
    )

    val_count = int(
        (
            val_df[
                "class_id"
            ]
            ==
            class_id
        ).sum()
    )

    test_count = int(
        (
            test_df[
                "class_id"
            ]
            ==
            class_id
        ).sum()
    )

    expected_train = (
        total
        -
        VALIDATION_PER_CLASS
        -
        TEST_PER_CLASS
    )

    class_pass = (
        train_count
        ==
        expected_train
        and
        val_count
        ==
        VALIDATION_PER_CLASS
        and
        test_count
        ==
        TEST_PER_CLASS
        and
        (
            train_count
            +
            val_count
            +
            test_count
        )
        ==
        total
    )

    class_audit_rows.append(
        {
            "class_id":
                class_id,

            "gloss":
                gloss,

            "total":
                total,

            "train":
                train_count,

            "validation":
                val_count,

            "test":
                test_count,

            "expected_train":
                expected_train,

            "class_pass":
                class_pass
        }
    )


class_split_audit = pd.DataFrame(
    class_audit_rows
)

class_split_audit.to_csv(
    CLASS_SPLIT_AUDIT_CSV,
    index=False
)

class_split_failures = class_split_audit[
    ~class_split_audit[
        "class_pass"
    ]
]

if len(
    class_split_failures
) > 0:

    raise RuntimeError(
        "One or more classes failed split auditing."
    )

print(
    "\n[PASS] All class-level split rules verified."
)


# ==========================================================================================
# 28. VERIFY FEATURE FILES
# ==========================================================================================

print("\n" + "=" * 125)
print("FEATURE INTEGRITY VERIFICATION")
print("=" * 125)

feature_audit_rows = []

global_max_abs = 0.0


for i, row in enumerate(
    master_split.itertuples(
        index=False
    ),
    start=1
):

    uid = str(
        row.uid_clean
    )

    split = str(
        row.split
    )

    class_id = int(
        row.class_id
    )

    gloss = str(
        row.gloss
    )

    feature_path = (
        FEATURE_DIR
        /
        f"{uid}.npz"
    )

    exists = feature_path.exists()

    shape_ok = False
    finite = False
    max_abs = np.nan
    read_ok = False
    error = ""

    if exists:

        try:

            with np.load(
                feature_path,
                allow_pickle=True
            ) as data:

                if "features" not in data.files:

                    raise KeyError(
                        "features key missing"
                    )

                features = np.asarray(
                    data[
                        "features"
                    ],
                    dtype=np.float32
                )

            read_ok = True

            shape_ok = (
                features.shape
                ==
                (
                    EXPECTED_SEQUENCE_LENGTH,
                    EXPECTED_FEATURE_DIM
                )
            )

            finite = bool(
                np.isfinite(
                    features
                ).all()
            )

            if features.size > 0:

                max_abs = float(
                    np.max(
                        np.abs(
                            features
                        )
                    )
                )

                global_max_abs = max(
                    global_max_abs,
                    max_abs
                )

        except Exception as exc:

            error = str(
                exc
            )

    feature_audit_rows.append(
        {
            "uid":
                uid,

            "class_id":
                class_id,

            "gloss":
                gloss,

            "split":
                split,

            "exists":
                exists,

            "read_ok":
                read_ok,

            "shape_ok":
                shape_ok,

            "finite":
                finite,

            "max_abs_feature":
                max_abs,

            "error":
                error
        }
    )

    if (
        i % 250 == 0
        or
        i == len(
            master_split
        )
    ):

        print(
            f"Verified {i:4d}/{len(master_split)}"
        )


feature_audit = pd.DataFrame(
    feature_audit_rows
)

feature_audit.to_csv(
    FEATURE_AUDIT_CSV,
    index=False
)

invalid_features = feature_audit[
    ~(
        feature_audit[
            "exists"
        ]
        &
        feature_audit[
            "read_ok"
        ]
        &
        feature_audit[
            "shape_ok"
        ]
        &
        feature_audit[
            "finite"
        ]
    )
]

print()
print(
    "Invalid feature files :",
    len(
        invalid_features
    )
)

print(
    "Largest |feature|     :",
    f"{global_max_abs:.6f}"
)

if len(
    invalid_features
) > 0:

    print()
    print(
        invalid_features.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Invalid feature files found."
    )

print(
    "\n[PASS] All 1007 feature files are valid finite 32 x 254 arrays."
)


# ==========================================================================================
# 29. SPLIT DISTRIBUTION SUMMARY
# ==========================================================================================

split_distribution = pd.DataFrame(
    [
        {
            "split":
                "train",

            "videos":
                len(
                    train_df
                ),

            "classes":
                train_df[
                    "class_id"
                ].nunique(),

            "percentage":
                100.0
                *
                len(
                    train_df
                )
                /
                len(
                    master_split
                ),

            "mean_detection_rate":
                (
                    float(
                        train_df[
                            "original_detection_rate"
                        ].mean()
                    )
                    if "original_detection_rate" in train_df.columns
                    else np.nan
                ),

            "mean_max_abs_feature":
                float(
                    feature_audit.loc[
                        feature_audit[
                            "split"
                        ]
                        ==
                        "train",
                        "max_abs_feature"
                    ].mean()
                )
        },

        {
            "split":
                "validation",

            "videos":
                len(
                    val_df
                ),

            "classes":
                val_df[
                    "class_id"
                ].nunique(),

            "percentage":
                100.0
                *
                len(
                    val_df
                )
                /
                len(
                    master_split
                ),

            "mean_detection_rate":
                (
                    float(
                        val_df[
                            "original_detection_rate"
                        ].mean()
                    )
                    if "original_detection_rate" in val_df.columns
                    else np.nan
                ),

            "mean_max_abs_feature":
                float(
                    feature_audit.loc[
                        feature_audit[
                            "split"
                        ]
                        ==
                        "validation",
                        "max_abs_feature"
                    ].mean()
                )
        },

        {
            "split":
                "test",

            "videos":
                len(
                    test_df
                ),

            "classes":
                test_df[
                    "class_id"
                ].nunique(),

            "percentage":
                100.0
                *
                len(
                    test_df
                )
                /
                len(
                    master_split
                ),

            "mean_detection_rate":
                (
                    float(
                        test_df[
                            "original_detection_rate"
                        ].mean()
                    )
                    if "original_detection_rate" in test_df.columns
                    else np.nan
                ),

            "mean_max_abs_feature":
                float(
                    feature_audit.loc[
                        feature_audit[
                            "split"
                        ]
                        ==
                        "test",
                        "max_abs_feature"
                    ].mean()
                )
        }
    ]
)

split_distribution.to_csv(
    SPLIT_DISTRIBUTION_CSV,
    index=False
)


# ==========================================================================================
# 30. SAVE SPLIT MANIFESTS
# ==========================================================================================

# Convert categorical split back to normal strings before CSV export.

master_split[
    "split"
] = master_split[
    "split"
].astype(str)

train_df[
    "split"
] = train_df[
    "split"
].astype(str)

val_df[
    "split"
] = val_df[
    "split"
].astype(str)

test_df[
    "split"
] = test_df[
    "split"
].astype(str)


master_split.to_csv(
    MASTER_SPLIT_CSV,
    index=False
)

train_df.to_csv(
    TRAIN_MANIFEST_CSV,
    index=False
)

val_df.to_csv(
    VAL_MANIFEST_CSV,
    index=False
)

test_df.to_csv(
    TEST_MANIFEST_CSV,
    index=False
)


# ==========================================================================================
# 31. SAVE PLAIN UID LISTS
# ==========================================================================================

def save_uid_list(
    dataframe,
    path
):

    uids = (
        dataframe[
            "uid_clean"
        ]
        .astype(str)
        .tolist()
    )

    with open(
        path,
        "w"
    ) as f:

        for uid in uids:

            f.write(
                uid
                +
                "\n"
            )


save_uid_list(
    train_df,
    TRAIN_UIDS_TXT
)

save_uid_list(
    val_df,
    VAL_UIDS_TXT
)

save_uid_list(
    test_df,
    TEST_UIDS_TXT
)


# ==========================================================================================
# 32. HASH HELPERS
# ==========================================================================================

def sha256_text(
    text
):

    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def dataframe_fingerprint(
    dataframe,
    columns
):

    temp = (
        dataframe[
            columns
        ]
        .copy()
        .sort_values(
            columns
        )
        .reset_index(
            drop=True
        )
    )

    text = temp.to_csv(
        index=False,
        lineterminator="\n"
    )

    return sha256_text(
        text
    )


# ==========================================================================================
# 33. SPLIT FINGERPRINTS
# ==========================================================================================

train_hash = dataframe_fingerprint(
    train_df,
    [
        "uid_clean",
        "class_id",
        "gloss"
    ]
)

val_hash = dataframe_fingerprint(
    val_df,
    [
        "uid_clean",
        "class_id",
        "gloss"
    ]
)

test_hash = dataframe_fingerprint(
    test_df,
    [
        "uid_clean",
        "class_id",
        "gloss"
    ]
)

master_hash = dataframe_fingerprint(
    master_split,
    [
        "uid_clean",
        "class_id",
        "gloss",
        "split"
    ]
)

mapping_hash = dataframe_fingerprint(
    class_mapping_df,
    [
        "class_id",
        "gloss"
    ]
)


# ==========================================================================================
# 34. DETERMINISM SELF-TEST
# ==========================================================================================
#
# Reconstruct the split independently and verify that the UID assignments
# are identical.
#
# ==========================================================================================

def recreate_split_assignments(
    dataframe,
    seed
):

    assignments = {}

    for class_id in sorted(
        dataframe[
            "class_id"
        ].unique()
    ):

        class_id = int(
            class_id
        )

        class_df = (
            dataframe[
                dataframe[
                    "class_id"
                ]
                ==
                class_id
            ]
            .copy()
            .sort_values(
                "uid_clean"
            )
            .reset_index(
                drop=True
            )
        )

        n = len(
            class_df
        )

        rng = np.random.default_rng(
            seed
            +
            class_id
        )

        permutation = rng.permutation(
            n
        )

        val_position = int(
            permutation[
                0
            ]
        )

        test_position = int(
            permutation[
                1
            ]
        )

        train_positions = [
            int(x)
            for x in permutation[
                2:
            ]
        ]

        assignments[
            str(
                class_df.iloc[
                    val_position
                ][
                    "uid_clean"
                ]
            )
        ] = "validation"

        assignments[
            str(
                class_df.iloc[
                    test_position
                ][
                    "uid_clean"
                ]
            )
        ] = "test"

        for position in train_positions:

            assignments[
                str(
                    class_df.iloc[
                        position
                    ][
                        "uid_clean"
                    ]
                )
            ] = "train"

    return assignments


recreated_assignments = recreate_split_assignments(
    controlled,
    SEED
)

actual_assignments = {
    str(
        row.uid_clean
    ):
        str(
            row.split
        )

    for row in master_split.itertuples(
        index=False
    )
}

determinism_pass = (
    recreated_assignments
    ==
    actual_assignments
)

if not determinism_pass:

    raise RuntimeError(
        "Determinism self-test failed."
    )

print(
    "\n[PASS] Deterministic split reproduction verified."
)


# ==========================================================================================
# 35. SPLIT QUALITY INFORMATION
#
# This is descriptive only.
# We do NOT alter the split based on these values.
# ==========================================================================================

def safe_mean(
    dataframe,
    column
):

    if column not in dataframe.columns:

        return None

    value = dataframe[
        column
    ].mean()

    if pd.isna(
        value
    ):

        return None

    return float(
        value
    )


def safe_median(
    dataframe,
    column
):

    if column not in dataframe.columns:

        return None

    value = dataframe[
        column
    ].median()

    if pd.isna(
        value
    ):

        return None

    return float(
        value
    )


# ==========================================================================================
# 36. SUMMARY JSON
# ==========================================================================================

summary = {
    "stage":
        "L9_CONTROLLED_SPLIT_FREEZE",

    "seed":
        SEED,

    "source":
        {
            "l8_controlled_manifest":
                str(
                    L8_CONTROLLED_MANIFEST
                ),

            "l8_class_mapping":
                str(
                    L8_CLASS_MAPPING_JSON
                )
        },

    "representation":
        {
            "sequence_length":
                EXPECTED_SEQUENCE_LENGTH,

            "feature_dim":
                EXPECTED_FEATURE_DIM
        },

    "population":
        {
            "videos":
                len(
                    master_split
                ),

            "classes":
                EXPECTED_CLASSES
        },

    "split_rule":
        {
            "validation_per_class":
                VALIDATION_PER_CLASS,

            "test_per_class":
                TEST_PER_CLASS,

            "train_per_class":
                "all_remaining",

            "minimum_train_per_class":
                MINIMUM_CLASS_SUPPORT
                -
                VALIDATION_PER_CLASS
                -
                TEST_PER_CLASS
        },

    "splits":
        {
            "train":
                {
                    "videos":
                        len(
                            train_df
                        ),

                    "classes":
                        int(
                            train_df[
                                "class_id"
                            ].nunique()
                        ),

                    "mean_original_detection_rate":
                        safe_mean(
                            train_df,
                            "original_detection_rate"
                        ),

                    "median_original_detection_rate":
                        safe_median(
                            train_df,
                            "original_detection_rate"
                        ),

                    "sha256":
                        train_hash
                },

            "validation":
                {
                    "videos":
                        len(
                            val_df
                        ),

                    "classes":
                        int(
                            val_df[
                                "class_id"
                            ].nunique()
                        ),

                    "mean_original_detection_rate":
                        safe_mean(
                            val_df,
                            "original_detection_rate"
                        ),

                    "median_original_detection_rate":
                        safe_median(
                            val_df,
                            "original_detection_rate"
                        ),

                    "sha256":
                        val_hash
                },

            "test":
                {
                    "videos":
                        len(
                            test_df
                        ),

                    "classes":
                        int(
                            test_df[
                                "class_id"
                            ].nunique()
                        ),

                    "mean_original_detection_rate":
                        safe_mean(
                            test_df,
                            "original_detection_rate"
                        ),

                    "median_original_detection_rate":
                        safe_median(
                            test_df,
                            "original_detection_rate"
                        ),

                    "sha256":
                        test_hash
                }
        },

    "master_split_sha256":
        master_hash,

    "class_mapping_sha256":
        mapping_hash,

    "global_max_abs_feature":
        float(
            global_max_abs
        )
}

with open(
    SUMMARY_JSON,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=4,
        ensure_ascii=False
    )


# ==========================================================================================
# 37. FREEZE MANIFEST
# ==========================================================================================

freeze_manifest = {
    "stage":
        "L9",

    "status":
        "FROZEN",

    "seed":
        SEED,

    "population":
        {
            "classes":
                EXPECTED_CLASSES,

            "videos":
                EXPECTED_VIDEOS
        },

    "split_strategy":
        (
            "Per-class deterministic split: "
            "1 validation, 1 test, all remaining train"
        ),

    "train":
        {
            "manifest":
                str(
                    TRAIN_MANIFEST_CSV
                ),

            "uids":
                str(
                    TRAIN_UIDS_TXT
                ),

            "videos":
                len(
                    train_df
                ),

            "sha256":
                train_hash
        },

    "validation":
        {
            "manifest":
                str(
                    VAL_MANIFEST_CSV
                ),

            "uids":
                str(
                    VAL_UIDS_TXT
                ),

            "videos":
                len(
                    val_df
                ),

            "sha256":
                val_hash
        },

    "test":
        {
            "manifest":
                str(
                    TEST_MANIFEST_CSV
                ),

            "uids":
                str(
                    TEST_UIDS_TXT
                ),

            "videos":
                len(
                    test_df
                ),

            "sha256":
                test_hash
        },

    "master_split":
        {
            "manifest":
                str(
                    MASTER_SPLIT_CSV
                ),

            "sha256":
                master_hash
        },

    "class_mapping":
        {
            "path":
                str(
                    CLASS_MAPPING_JSON
                ),

            "sha256":
                mapping_hash
        },

    "preprocessing_statistics_fitted":
        False,

    "model_trained":
        False,

    "next_stage":
        "L10_GRU_BASELINE"
}

with open(
    FREEZE_MANIFEST_JSON,
    "w"
) as f:

    json.dump(
        freeze_manifest,
        f,
        indent=4,
        ensure_ascii=False
    )


# ==========================================================================================
# 38. FINAL CHECKPOINT CONDITIONS
# ==========================================================================================

count_pass = (
    len(
        train_df
    )
    ==
    EXPECTED_TRAIN
    and
    len(
        val_df
    )
    ==
    EXPECTED_VAL
    and
    len(
        test_df
    )
    ==
    EXPECTED_TEST
    and
    len(
        master_split
    )
    ==
    EXPECTED_VIDEOS
)

zero_overlap_pass = (
    len(
        train_val_overlap
    )
    ==
    0
    and
    len(
        train_test_overlap
    )
    ==
    0
    and
    len(
        val_test_overlap
    )
    ==
    0
)

population_complete_pass = (
    source_uids
    ==
    split_uids
)

class_coverage_pass = (
    all_classes_in_train
    and
    all_classes_in_val
    and
    all_classes_in_test
)

class_rule_pass = (
    bool(
        class_split_audit[
            "class_pass"
        ].all()
    )
)

feature_integrity_pass = (
    len(
        invalid_features
    )
    ==
    0
)

mapping_pass = (
    len(
        mapping_mismatches
    )
    ==
    0
)


l9_pass = bool(
    bool(
        l8_checkpoint.get(
            "l8_pass",
            False
        )
    )
    and
    count_pass
    and
    zero_overlap_pass
    and
    population_complete_pass
    and
    class_coverage_pass
    and
    class_rule_pass
    and
    feature_integrity_pass
    and
    mapping_pass
    and
    determinism_pass
)


checkpoint = {
    "l8_pass_verified":
        bool(
            l8_checkpoint.get(
                "l8_pass",
                False
            )
        ),

    "population_count_pass":
        count_pass,

    "zero_uid_overlap":
        zero_overlap_pass,

    "population_complete":
        population_complete_pass,

    "all_classes_in_train":
        all_classes_in_train,

    "all_classes_in_validation":
        all_classes_in_val,

    "all_classes_in_test":
        all_classes_in_test,

    "class_split_rules_pass":
        class_rule_pass,

    "feature_integrity_pass":
        feature_integrity_pass,

    "class_mapping_pass":
        mapping_pass,

    "determinism_pass":
        determinism_pass,

    "l9_pass":
        l9_pass
}

with open(
    CHECKPOINT_JSON,
    "w"
) as f:

    json.dump(
        checkpoint,
        f,
        indent=4
    )


# ==========================================================================================
# 39. FINAL REPORT
# ==========================================================================================

print("\n" + "=" * 125)
print("L9 FINAL REPORT")
print("=" * 125)


# ------------------------------------------------------------------------------------------
# Population
# ------------------------------------------------------------------------------------------

print("\nFROZEN CONTROLLED POPULATION")
print("-" * 125)

print(
    "Classes                     :",
    EXPECTED_CLASSES
)

print(
    "Videos                      :",
    EXPECTED_VIDEOS
)

print(
    "Sequence shape              :",
    f"{EXPECTED_SEQUENCE_LENGTH} x {EXPECTED_FEATURE_DIM}"
)

print(
    "Random seed                 :",
    SEED
)


# ------------------------------------------------------------------------------------------
# Split
# ------------------------------------------------------------------------------------------

print("\nTRAIN / VALIDATION / TEST")
print("-" * 125)

print(
    f"Train      : {len(train_df):4d} videos | "
    f"{train_df['class_id'].nunique():3d} classes | "
    f"{100.0 * len(train_df) / EXPECTED_VIDEOS:6.2f}%"
)

print(
    f"Validation : {len(val_df):4d} videos | "
    f"{val_df['class_id'].nunique():3d} classes | "
    f"{100.0 * len(val_df) / EXPECTED_VIDEOS:6.2f}%"
)

print(
    f"Test       : {len(test_df):4d} videos | "
    f"{test_df['class_id'].nunique():3d} classes | "
    f"{100.0 * len(test_df) / EXPECTED_VIDEOS:6.2f}%"
)

print(
    f"Total      : {len(master_split):4d} videos"
)


# ------------------------------------------------------------------------------------------
# Quality
# ------------------------------------------------------------------------------------------

print("\nDETECTION COVERAGE BY SPLIT")
print("-" * 125)

if "original_detection_rate" in master_split.columns:

    for split_name, df in [
        (
            "Train",
            train_df
        ),
        (
            "Validation",
            val_df
        ),
        (
            "Test",
            test_df
        )
    ]:

        print(
            f"{split_name:10s}: "
            f"mean={100.0 * df['original_detection_rate'].mean():6.2f}% | "
            f"median={100.0 * df['original_detection_rate'].median():6.2f}%"
        )

else:

    print(
        "original_detection_rate unavailable "
        "in the L8 manifest."
    )


# ------------------------------------------------------------------------------------------
# Integrity
# ------------------------------------------------------------------------------------------

print("\nINTEGRITY")
print("-" * 125)

print(
    "L8 checkpoint verified      :",
    bool(
        l8_checkpoint.get(
            "l8_pass",
            False
        )
    )
)

print(
    "Expected counts             :",
    count_pass
)

print(
    "Zero UID overlap            :",
    zero_overlap_pass
)

print(
    "Population complete         :",
    population_complete_pass
)

print(
    "All classes in train        :",
    all_classes_in_train
)

print(
    "All classes in validation   :",
    all_classes_in_val
)

print(
    "All classes in test         :",
    all_classes_in_test
)

print(
    "Class split rules           :",
    class_rule_pass
)

print(
    "Feature integrity           :",
    feature_integrity_pass
)

print(
    "Class mapping               :",
    mapping_pass
)

print(
    "Deterministic reproduction  :",
    determinism_pass
)

print(
    "Largest |feature|           :",
    f"{global_max_abs:.6f}"
)


# ------------------------------------------------------------------------------------------
# Fingerprints
# ------------------------------------------------------------------------------------------

print("\nFROZEN SHA256 FINGERPRINTS")
print("-" * 125)

print(
    "Train       :",
    train_hash
)

print(
    "Validation  :",
    val_hash
)

print(
    "Test        :",
    test_hash
)

print(
    "Master      :",
    master_hash
)

print(
    "Class map   :",
    mapping_hash
)


# ==========================================================================================
# 40. CLASS SPLIT DISTRIBUTION
# ==========================================================================================

print("\n" + "=" * 125)
print("CLASS SPLIT DISTRIBUTION")
print("=" * 125)

split_pattern_distribution = (
    class_split_audit
    .groupby(
        [
            "total",
            "train",
            "validation",
            "test"
        ]
    )
    .size()
    .reset_index(
        name="classes"
    )
    .sort_values(
        "total"
    )
)

print(
    split_pattern_distribution.to_string(
        index=False
    )
)


# ==========================================================================================
# 41. SAMPLE SPLIT ROWS
# ==========================================================================================

print("\n" + "=" * 125)
print("FIRST 15 TRAIN SAMPLES")
print("=" * 125)

print(
    train_df[
        [
            "uid_clean",
            "class_id",
            "gloss",
            "split"
        ]
    ]
    .head(15)
    .to_string(
        index=False
    )
)

print("\n" + "=" * 125)
print("FIRST 15 VALIDATION SAMPLES")
print("=" * 125)

print(
    val_df[
        [
            "uid_clean",
            "class_id",
            "gloss",
            "split"
        ]
    ]
    .head(15)
    .to_string(
        index=False
    )
)

print("\n" + "=" * 125)
print("FIRST 15 TEST SAMPLES")
print("=" * 125)

print(
    test_df[
        [
            "uid_clean",
            "class_id",
            "gloss",
            "split"
        ]
    ]
    .head(15)
    .to_string(
        index=False
    )
)


# ==========================================================================================
# 42. L9 CHECKPOINT
# ==========================================================================================

print("\n" + "=" * 125)
print("L9 CHECKPOINT")
print("=" * 125)

if l9_pass:

    print()
    print(
        "[PASS] L9 TRAIN / VALIDATION / TEST SPLIT FROZEN."
    )

    print()
    print(
        f"TRAIN      : {len(train_df)} videos / "
        f"{train_df['class_id'].nunique()} classes"
    )

    print(
        f"VALIDATION : {len(val_df)} videos / "
        f"{val_df['class_id'].nunique()} classes"
    )

    print(
        f"TEST       : {len(test_df)} videos / "
        f"{test_df['class_id'].nunique()} classes"
    )

    print()
    print(
        "Every one of the 211 classes is represented "
        "in all three partitions."
    )

    print(
        "There is zero UID overlap between partitions."
    )

    print(
        "All 1007 feature files are valid finite "
        "32 x 254 representations."
    )

    print()
    print(
        "IMPORTANT:"
    )

    print(
        "From this point onward, do NOT reshuffle or "
        "regenerate the controlled split."
    )

    print(
        "L10 GRU and L11 BiLSTM must use these exact "
        "train / validation / test manifests."
    )

    print()
    print(
        "NEXT STAGE: L10 — GRU BASELINE"
    )

else:

    print()
    print(
        "[REVIEW] L9 DID NOT PASS ALL CHECKPOINT CONDITIONS."
    )

    print()
    print(
        "Do NOT proceed to L10."
    )


# ==========================================================================================
# 43. OUTPUT FILES
# ==========================================================================================

print("\n" + "=" * 125)
print("L9 OUTPUT FILES")
print("=" * 125)

print("\nMASTER SPLIT")
print(MASTER_SPLIT_CSV)

print("\nTRAIN MANIFEST")
print(TRAIN_MANIFEST_CSV)

print("\nVALIDATION MANIFEST")
print(VAL_MANIFEST_CSV)

print("\nTEST MANIFEST")
print(TEST_MANIFEST_CSV)

print("\nTRAIN UID LIST")
print(TRAIN_UIDS_TXT)

print("\nVALIDATION UID LIST")
print(VAL_UIDS_TXT)

print("\nTEST UID LIST")
print(TEST_UIDS_TXT)

print("\nCLASS SPLIT AUDIT")
print(CLASS_SPLIT_AUDIT_CSV)

print("\nFEATURE AUDIT")
print(FEATURE_AUDIT_CSV)

print("\nCLASS MAPPING")
print(CLASS_MAPPING_JSON)

print("\nSUMMARY")
print(SUMMARY_JSON)

print("\nFREEZE MANIFEST")
print(FREEZE_MANIFEST_JSON)

print("\nCHECKPOINT")
print(CHECKPOINT_JSON)

print("\n" + "=" * 125)

CISLR LANDMARK MODEL — L9
CONTROLLED TRAIN / VALIDATION / TEST SPLIT FREEZE

Random seed                  : 42
Expected controlled classes  : 211
Expected controlled videos   : 1007
Minimum class support        : 4
Validation samples/class     : 1
Test samples/class           : 1

SPLIT RULE
-----------------------------------------------------------------------------------------------------------------------------
For EACH class:
    1 randomly selected deterministic sample -> VALIDATION
    1 different deterministic sample         -> TEST
    all remaining samples                    -> TRAIN

Examples:
    4 samples  -> 2 train / 1 validation / 1 test
    5 samples  -> 3 train / 1 validation / 1 test
    8 samples  -> 6 train / 1 validation / 1 test
   13 samples  ->11 train / 1 validation / 1 test

IMPORTANT
-----------------------------------------------------------------------------------------------------------------------------
No feature normalization is fitted in L9.
No model 